In [ ]:
import sys
import asyncio

# Fix Jupyter's event loop policy on Windows without using deprecated methods
if sys.platform == "win32":
    try:
        from IPython import get_ipython
        _ipython = get_ipython()
        if _ipython and hasattr(_ipython, 'kernel'):
            # Directly swap the selector loop implementation on the active kernel
            _ipython.kernel.io_loop.asyncio_loop = asyncio.SelectorEventLoop()
    except (ImportError, NameError):
        pass

<img src="https://lh3.googleusercontent.com/gps-cs-s/AHRPTWnyfF1euMvSlfuEPYEnwuzoysb96UZ6UGbyWIDAcF6cOUgIvg2hwWIL_e8LRcJh2SuVPqcDW4ZSWegEt5wwv4rSNxA4NMBjdxZ8PFIdaag_eDKx9IYZRHbGlKka5Up-PhclqFd1gg=s680-w680-h510-rw" alt="no image" width="1250" height="300">


# THE BUSINESS PROBLEM

* Analyze historical sales and customer transaction data to identify sales trends, product performance, customer behavior, operational patterns, and opportunities for growth

* Dataset
    - P - Block
    - S - Sliced
    - B - Brown
    - No - Order number
    - Invoice - Invoice Number
    - A/C - Account Number
    - Customers - Clients
    - Mando - Mandazi
    - Maicho - Doughnuts
    - Family Pack - Snacks
    - B-Bitez - Snacks
    - 200g - 200 grams bread both with and brown
    - 600g - 600 gram bread both white and brown
    - 800g - 800 gram bread both white and brown
    - O.P - premium bread
    - 4-square - 400 gram bread

In [97]:
%matplotlib inline
import matplotlib
matplotlib.use('Agg')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import nbconvert
import voila
from ipywidgets import interact, interactive
import plotly.express as px
import ipywidgets as widgets

In [ ]:
from pathlib import Path

BASE_DIR = Path.cwd()
path = BASE_DIR / "data" / "apr-june-2nd-quarter-invoices-datewise.xlsx"
df = pd.read_excel(path)
df.head()

,Date,No,Invoice No.,A/C,Customer,P,S,B,200g,O.P.,...,ST,600g,KATIKATI,MANDAZI,DOUGHNUTS,50-50 SLICED,MANDO FAMILY PACK,4 Square,XTRA LOONG,Qty
0,2026-04-01,55260139,969027,55,LUCY WAIRIMU KAIRU (055),0,270,105,216,60,...,0,21,0,0,0,0,0,0,35,707
1,2026-04-01,55260140,969029,55,LUCY WAIRIMU KAIRU (055),0,0,0,0,0,...,0,0,0,10,0,0,0,0,0,10
2,2026-04-01,57260123,968728,57,JAMES MWANGI (057),105,825,285,1032,45,...,0,82,0,0,0,0,0,165,7,2546
3,2026-04-01,62260123,968703,62,ANTONY MWANGI (062),15,705,255,600,45,...,0,280,0,0,0,0,0,15,14,1929
4,2026-04-01,63260102,968705,63,NASHON KYULE (063),75,1050,225,360,0,...,0,21,0,0,0,0,0,0,0,1737


## Data Quality

In [99]:
# 1.0 Missing values
df.isnull().sum()

Date                 0
No                   0
Invoice No.          0
A/C                  0
Customer             2
P                    0
S                    0
B                    0
200g                 0
O.P.                 0
800g                 0
1500g                0
ST                   0
600g                 0
KATIKATI             0
MANDAZI              0
DOUGHNUTS            0
50-50 SLICED         0
MANDO FAMILY PACK    0
4 Square             0
XTRA LOONG           0
Qty                  0
dtype: int64

In [100]:
df.isna().sum()

Date                 0
No                   0
Invoice No.          0
A/C                  0
Customer             2
P                    0
S                    0
B                    0
200g                 0
O.P.                 0
800g                 0
1500g                0
ST                   0
600g                 0
KATIKATI             0
MANDAZI              0
DOUGHNUTS            0
50-50 SLICED         0
MANDO FAMILY PACK    0
4 Square             0
XTRA LOONG           0
Qty                  0
dtype: int64

In [101]:
# getting the value of the code where the missing values are present
df.loc[df.isnull().any(axis=1)]

,Date,No,Invoice No.,A/C,Customer,P,S,B,200g,O.P.,...,ST,600g,KATIKATI,MANDAZI,DOUGHNUTS,50-50 SLICED,MANDO FAMILY PACK,4 Square,XTRA LOONG,Qty
31886,2026-05-06,1726260235,1011257,1726,NaN,0,20,15,16,6,...,0,41,0,0,0,0,0,0,0,124
49797,2026-05-27,932260250,1038751,932,NaN,0,10,15,0,0,...,0,5,0,0,0,0,0,0,0,35


In [102]:
# 1. Account-to-customer mapping
customer_map = {
    1726: 'QUICK MART LIMITED - JIPANGE (1726)',
    932: 'NAIVAS SUPERMARKET HAZINA SOUTHB (0932)'
}

# 2. ping the 'A/C' column to find the correct names, then fill the missing values
df['Customer'] = df['Customer'].fillna(df['A/C'].map(customer_map))

In [103]:
df.isna().sum()

Date                 0
No                   0
Invoice No.          0
A/C                  0
Customer             0
P                    0
S                    0
B                    0
200g                 0
O.P.                 0
800g                 0
1500g                0
ST                   0
600g                 0
KATIKATI             0
MANDAZI              0
DOUGHNUTS            0
50-50 SLICED         0
MANDO FAMILY PACK    0
4 Square             0
XTRA LOONG           0
Qty                  0
dtype: int64

In [104]:
# 1.1 Duplicate rows
df.duplicated().sum()

np.int64(0)

In [105]:
df.columns

Index(['Date', 'No', 'Invoice No.', 'A/C', 'Customer', 'P', 'S', 'B', '200g',
       'O.P.', '800g', '1500g', 'ST', '600g', 'KATIKATI', 'MANDAZI',
       'DOUGHNUTS', '50-50 SLICED', 'MANDO FAMILY PACK', '4 Square',
       'XTRA LOONG', 'Qty'],
      dtype='str')

In [106]:
# 1.2 Duplicate invoices
df.duplicated(subset=['Invoice No.']).sum()

np.int64(0)

In [107]:
# 1.3 Invalid Date
df['Date'] = pd.to_datetime(df['Date'], errors='coerce', format='%d-%m-%Y')
df.isna().sum()

Date                 0
No                   0
Invoice No.          0
A/C                  0
Customer             0
P                    0
S                    0
B                    0
200g                 0
O.P.                 0
800g                 0
1500g                0
ST                   0
600g                 0
KATIKATI             0
MANDAZI              0
DOUGHNUTS            0
50-50 SLICED         0
MANDO FAMILY PACK    0
4 Square             0
XTRA LOONG           0
Qty                  0
dtype: int64

In [108]:
# 1.4 Negative Quantity
col = df.select_dtypes(include=[np.number]).columns.tolist()

# 2. Check for negatives (keeps your original check output)
col_with_negatives = df[col].lt(0).any()

# 3. Modern replacement: Mask values less than 0 with NaN
df[col] = df[col].mask(df[col] < 0, np.nan)

col_with_negatives


No                   False
Invoice No.          False
A/C                  False
P                    False
S                    False
B                    False
200g                 False
O.P.                 False
800g                 False
1500g                False
ST                   False
600g                 False
KATIKATI             False
MANDAZI              False
DOUGHNUTS            False
50-50 SLICED         False
MANDO FAMILY PACK    False
4 Square             False
XTRA LOONG           False
Qty                  False
dtype: bool

In [109]:
# 1.5 Zero Quantity
df[(df[col] == 0).any(axis=1)].shape[0]

80803

In [110]:
# 1.6 Customer Name Inconsistency
# Adding an underscore to the customer names to make them consistent
df['Customer'] = df['Customer'].str.replace(' ', '_')

df['Customer'].nunique()

1131

In [111]:
df['Customer'].head(10)

0        LUCY_WAIRIMU_KAIRU_(055)
1        LUCY_WAIRIMU_KAIRU_(055)
2              JAMES_MWANGI_(057)
3             ANTONY_MWANGI_(062)
4              NASHON_KYULE_(063)
5    PATRICK_WAWERU_GICHUHI_(082)
6            GERALD_GICHUKI_(100)
7             PETER_KARANJA_(101)
8        PURITY_WANJA_NGATIA(120)
9        PURITY_WANJA_NGATIA(120)
Name: Customer, dtype: str

In [112]:
# 1.7 Columns Name Inconsistency
# Replace spaces with underscores in all column headers
df.columns = df.columns.str.replace(' ', '_')

In [113]:
df.head()

,Date,No,Invoice_No.,A/C,Customer,P,S,B,200g,O.P.,...,ST,600g,KATIKATI,MANDAZI,DOUGHNUTS,50-50_SLICED,MANDO_FAMILY_PACK,4_Square,XTRA_LOONG,Qty
0,2026-04-01,55260139,969027,55,LUCY_WAIRIMU_KAIRU_(055),0,270,105,216,60,...,0,21,0,0,0,0,0,0,35,707
1,2026-04-01,55260140,969029,55,LUCY_WAIRIMU_KAIRU_(055),0,0,0,0,0,...,0,0,0,10,0,0,0,0,0,10
2,2026-04-01,57260123,968728,57,JAMES_MWANGI_(057),105,825,285,1032,45,...,0,82,0,0,0,0,0,165,7,2546
3,2026-04-01,62260123,968703,62,ANTONY_MWANGI_(062),15,705,255,600,45,...,0,280,0,0,0,0,0,15,14,1929
4,2026-04-01,63260102,968705,63,NASHON_KYULE_(063),75,1050,225,360,0,...,0,21,0,0,0,0,0,0,0,1737


In [114]:
# 1.8 Data Type Inconsistency
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 80803 entries, 0 to 80802
Data columns (total 22 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Date               80803 non-null  datetime64[us]
 1   No                 80803 non-null  int64         
 2   Invoice_No.        80803 non-null  int64         
 3   A/C                80803 non-null  int64         
 4   Customer           80803 non-null  str           
 5   P                  80803 non-null  int64         
 6   S                  80803 non-null  int64         
 7   B                  80803 non-null  int64         
 8   200g               80803 non-null  int64         
 9   O.P.               80803 non-null  int64         
 10  800g               80803 non-null  int64         
 11  1500g              80803 non-null  int64         
 12  ST                 80803 non-null  int64         
 13  600g               80803 non-null  int64         
 14  KATIKATI         

In [115]:
# 1.9 Standardize columns while appropriate

# Fixing Trailing or Double Spaces
df.columns = df.columns.str.strip().str.replace('.', ' ')
df.columns = df.columns.str.strip().str.replace('  ', '')

# Fixing Capitalisation (Case-Sensitivity)
df.columns = df.columns.str.title()

# Cleaning Up Customer Suffixes
df.columns = df.columns.str.replace(r'\s*\(.*\)$', '', regex=True)

In [116]:
# 2.0 Creating Useful Date Columns
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Quarter'] = df['Date'].dt.to_period('Q')
df['Week'] = df['Date'].dt.isocalendar().week
df['Day'] = df['Date'].dt.day
df['Day_of_Week'] = df['Date'].dt.day_name()

In [117]:
df.head()

,Date,No,Invoice_No,A/C,Customer,P,S,B,200G,O P,...,Mando_Family_Pack,4_Square,Xtra_Loong,Qty,Year,Month,Quarter,Week,Day,Day_of_Week
0,2026-04-01,55260139,969027,55,LUCY_WAIRIMU_KAIRU_(055),0,270,105,216,60,...,0,0,35,707,2026,4,2026Q2,14,1,Wednesday
1,2026-04-01,55260140,969029,55,LUCY_WAIRIMU_KAIRU_(055),0,0,0,0,0,...,0,0,0,10,2026,4,2026Q2,14,1,Wednesday
2,2026-04-01,57260123,968728,57,JAMES_MWANGI_(057),105,825,285,1032,45,...,0,165,7,2546,2026,4,2026Q2,14,1,Wednesday
3,2026-04-01,62260123,968703,62,ANTONY_MWANGI_(062),15,705,255,600,45,...,0,15,14,1929,2026,4,2026Q2,14,1,Wednesday
4,2026-04-01,63260102,968705,63,NASHON_KYULE_(063),75,1050,225,360,0,...,0,0,0,1737,2026,4,2026Q2,14,1,Wednesday


In [118]:
df.columns

Index(['Date', 'No', 'Invoice_No', 'A/C', 'Customer', 'P', 'S', 'B', '200G',
       'O P', '800G', '1500G', 'St', '600G', 'Katikati', 'Mandazi',
       'Doughnuts', '50-50_Sliced', 'Mando_Family_Pack', '4_Square',
       'Xtra_Loong', 'Qty', 'Year', 'Month', 'Quarter', 'Week', 'Day',
       'Day_of_Week'],
      dtype='str')

In [119]:
# Convert the DataFrame from wide to long format for easier analysis and visualization

# 1. Identifier columns (Descriptive meta-data to keep as columns)
id_vars = [
    'Date', 'No', 'Invoice_No', 'A/C', 'Customer', 
    'Qty', 'Year', 'Month', 'Quarter', 'Week', 'Day', 'Day_of_Week'
]

# 2. Product columns
value_vars = [
    'P', 'S', 'B', '200G', 'O P', '800G', '1500G', 'St', '600G', 
    'Katikati', 'Mandazi', 'Doughnuts', '50-50_Sliced', 
    'Mando_Family_Pack', '4_Square', 'Xtra_Loong'
]

# 3. Melt into long format
df_long = pd.melt(
    df,
    id_vars=id_vars,
    value_vars=value_vars,
    var_name='Product',      # Contains 'P', 'S', 'B', '200G', 'Mandazi', etc.
    value_name='Sales_Value'  # Contains the quantities/amounts for that product
)

# 4. Keeping only rows where an actual sale happened
df_long = df_long[df_long['Sales_Value'] > 0].dropna(subset=['Sales_Value'])


In [120]:
df_long.head()

,Date,No,Invoice_No,A/C,Customer,Qty,Year,Month,Quarter,Week,Day,Day_of_Week,Product,Sales_Value
2,2026-04-01,57260123,968728,57,JAMES_MWANGI_(057),2546,2026,4,2026Q2,14,1,Wednesday,P,105
3,2026-04-01,62260123,968703,62,ANTONY_MWANGI_(062),1929,2026,4,2026Q2,14,1,Wednesday,P,15
4,2026-04-01,63260102,968705,63,NASHON_KYULE_(063),1737,2026,4,2026Q2,14,1,Wednesday,P,75
5,2026-04-01,82260107,969045,82,PATRICK_WAWERU_GICHUHI_(082),801,2026,4,2026Q2,14,1,Wednesday,P,45
6,2026-04-01,100260101,968561,100,GERALD_GICHUKI_(100),1218,2026,4,2026Q2,14,1,Wednesday,P,165


In [121]:
df_long.shape

(344594, 14)

## Overall business performance analysis

In [122]:
# 1.1 Number of Transactions per Customer
import ipywidgets as widgets
from ipywidgets import interact

def transactions_per_customer(df, selected_customer=None):
    # If a specific customer is picked, filter the data first
    if selected_customer and selected_customer != "All Customers":
        df = df[df['Customer'] == selected_customer]
        
    transactions = df.groupby('Customer')['Invoice_No'].nunique().reset_index()
    transactions.columns = ['Customer', 'Number_of_Transactions']
    return transactions

# Get a unique, sorted list of customers for the dropdown menu
customer_list = ["All Customers"] + sorted(df_long['Customer'].dropna().unique().tolist())

# Create the interactive dropdown interface
@interact(customer=widgets.Dropdown(options=customer_list, description='Customer:'))
def show_transactions(customer):
    # This function auto-runs every time the user shifts the dropdown
    result = transactions_per_customer(df_long, selected_customer=customer)
    display(result)


interactive(children=(Dropdown(description='Customer:', options=('All Customers', 'ABDALLAH_MUSA_OKUNGO_(2407)…

In [123]:
# 1.2 Unique Products Sold per Customer
def unique_products_multiple_customers(df, selected_customers, show_all=False):
    """
    Returns unique product types sold to selected customers or all customers.
    """
    if not show_all:
        # Filter only for the explicitly clicked customers
        df = df[df['Customer'].isin(selected_customers)]
    
    # Extract unique products grouped by customer
    unique_products = df.groupby('Customer')['Product'].apply(lambda x: sorted(list(x.unique()))).reset_index()
    unique_products.columns = ['Customer', 'Product_Types_Sold']
    
    return unique_products

# 1. Get unique, sorted list of customers
customer_list = sorted(df_long['Customer'].dropna().unique().tolist())

# 2. Define both widgets explicitly
customer_select = widgets.SelectMultiple(
    options=customer_list,
    value=(),  # Starts empty so nothing displays at boot
    description='Customers:',
    rows=8
)

all_checkbox = widgets.Checkbox(
    value=False,
    description='Show All Customers',
    disabled=False,
    indent=True
)

# 3. Pass both layout controls into the interact function
@interact(customers=customer_select, show_all=all_checkbox)
def show_unique_products_combined(customers, show_all):
    # If checkbox is off AND no specific names are highlighted, keep screen clean
    if not show_all and not customers:
        print("Select names from the list (Hold Ctrl/Cmd to choose multiple) OR check 'Show All Customers'.")
        return
        
    # Execute query based on current widget states
    result = unique_products_multiple_customers(df_long, selected_customers=customers, show_all=show_all)
    display(result.style.set_properties(**{'text-align': 'left'}))


interactive(children=(SelectMultiple(description='Customers:', options=('ABDALLAH_MUSA_OKUNGO_(2407)_-_KONDELE…

In [124]:
# 1.3 Total Sales Value per Customer per Product
def total_sales_per_customer_product(df, selected_customers, show_all=False):
    """
    Calculates total sales value grouped by Customer and Product.
    """
    if not show_all:
        # Filter for rows matching any of the selected customers
        df = df[df['Customer'].isin(selected_customers)]
        
    total_sales = df.groupby(['Customer', 'Product'])['Sales_Value'].sum().reset_index()
    total_sales.columns = ['Customer', 'Product', 'Total_Sales_Value']
    
    # Sort by sales value descending so top items appear first
    return total_sales.sort_values(by='Total_Sales_Value', ascending=False)

# 1. Get unique, sorted list of customers
customer_list = sorted(df_long['Customer'].dropna().unique().tolist())

# 2. Build the interactive layout widgets
customer_select = widgets.SelectMultiple(
    options=customer_list,
    value=(),  # Starts empty so nothing displays on run
    description='Customers:',
    rows=8
)

all_checkbox = widgets.Checkbox(
    value=False,
    description='Show All Customers',
    disabled=False,
    indent=True
)

# 3. Connect the widgets to the execution logic
@interact(customers=customer_select, show_all=all_checkbox)
def show_total_sales(customers, show_all):
    # Hidden state management: keep screen clean if nothing is chosen
    if not show_all and not customers:
        print("Select customer names (Hold Ctrl/Cmd to select multiple) OR check 'Show All Customers'.")
        return
        
    # Execute the breakdown function
    result = total_sales_per_customer_product(df_long, selected_customers=customers, show_all=show_all)
    
    # Format the monetary sales column with thousands separators for readability
    styled_result = result.style.format({'Total_Sales_Value': '{:,.2f}'})
    display(styled_result)


interactive(children=(SelectMultiple(description='Customers:', options=('ABDALLAH_MUSA_OKUNGO_(2407)_-_KONDELE…

In [125]:
# 1.4 Unique Customers per Product
import io
import base64  # Handles secure encoding without relying on pandas internals
import pandas as pd
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display, HTML

def unique_customers_per_product(df, selected_products, selected_customers, show_all_products=False, show_all_customers=False):
    """
    Returns unique customers and their A/C numbers in separate, parallel numbered lists.
    """
    # 1. Apply Product Filtering
    if not show_all_products and selected_products:
        df = df[df['Product'].isin(selected_products)]
        
    # 2. Apply Customer Filtering
    if not show_all_customers and selected_customers:
        df = df[df['Customer'].isin(selected_customers)]
    
    # Drop duplicates early to find true unique Customer + A/C pairings per Product
    df_unique = df[['Product', 'Customer', 'A/C']].dropna().drop_duplicates()
    df_unique = df_unique.sort_values(by=['Product', 'Customer'])

    # Helper function to generate clean parallel vertical text lists
    def build_parallel_lists(group):
        customers = group['Customer'].tolist()
        ac_numbers = group['A/C'].tolist()
        
        numbered_customers = "\n".join([f"{i+1}. {cust}" for i, cust in enumerate(customers)])
        numbered_acs = "\n".join([f"{i+1}. {ac}" for i, ac in enumerate(ac_numbers)])
        
        return pd.Series({
            'Unique_Customers': numbered_customers,
            'Account_Numbers': numbered_acs
        })

    # 3. Group by Product and build our parallel structural lists
    unique_data = df_unique.groupby('Product').apply(build_parallel_lists).reset_index()
    
    return unique_data

# Get sorted unique lists from the dataframe
product_list = sorted(df_long['Product'].dropna().unique().tolist())
customer_list = sorted(df_long['Customer'].dropna().unique().tolist())

# Create the interactive filtering widgets
product_select = widgets.SelectMultiple(options=product_list, value=(), description='Products:', rows=6)
customer_select = widgets.SelectMultiple(options=customer_list, value=(), description='Customers:', rows=6)

all_prod_check = widgets.Checkbox(value=False, description='All Products')
all_cust_check = widgets.Checkbox(value=False, description='All Customers')

download_output = widgets.Output()

@interact(products=product_select, customers=customer_select, all_prods=all_prod_check, all_custs=all_cust_check)
def show_and_export_data(products, customers, all_prods, all_custs):
    download_output.clear_output()
    
    if not all_prods and not products:
        print("Step 1: Select Products (or check 'All Products')")
        return
    if not all_custs and not customers:
        print("Step 2: Select Customers (or check 'All Customers')")
        return
        
    # 1. Execute the query
    result = unique_customers_per_product(
        df_long, 
        selected_products=products, 
        selected_customers=customers, 
        show_all_products=all_prods, 
        show_all_customers=all_custs
    )
    
    if result.empty:
        print("No matching data found for this selection.")
        return

    # 2. Fix Excel Limit: Create a safe copy specifically formatted for Excel layout 
    # Swaps the multi-line breaks (\n) for simple semicolons to prevent character overflow bugs
    excel_result = result.copy()
    excel_result['Unique_Customers'] = excel_result['Unique_Customers'].str.replace('\n', '; ')
    excel_result['Account_Numbers'] = excel_result['Account_Numbers'].str.replace('\n', '; ')

    output_buffer = io.BytesIO()
    with pd.ExcelWriter(output_buffer, engine='xlsxwriter') as writer:
        excel_result.to_excel(writer, index=False, sheet_name='Filtered Report')
        
        workbook  = writer.book
        worksheet = writer.sheets['Filtered Report']
        wrap_format = workbook.add_format({'text_wrap': True, 'valign': 'top'})
        
        # Format individual columns appropriately
        worksheet.set_column('A:A', 15)  # Product
        worksheet.set_column('B:B', 50, wrap_format)  # Unique Customers
        worksheet.set_column('C:C', 25, wrap_format)  # Account Numbers
        
    excel_data = output_buffer.getvalue()
    b64_encoded = base64.b64encode(excel_data).decode()
    
    # 3. Create a functional HTML download button widget
    button_html = f'''
    <a download="filtered_customer_product_report.xlsx" href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64_encoded}" target="_blank">
        <button style="background-color:#1F6E43; color:white; padding:8px 16px; border:none; border-radius:4px; font-weight:bold; cursor:pointer; margin-bottom:12px;">
            Download Filtered Excel Report
        </button>
    </a>
    '''
    
    with download_output:
        display(HTML(button_html))
        
    display(download_output)
    
    # 4. Display the stylized, numbered columns visually inside Jupyter Notebook
    styled_df = result.style.set_properties(**{
        'text-align': 'left',
        'white-space': 'pre-wrap',
        'vertical-align': 'top'
    })
    
    display(styled_df)


interactive(children=(SelectMultiple(description='Products:', options=('200G', '4_Square', '600G', '800G', 'B'…

In [126]:
# 1.5 Total Sales Value per Product
def total_sales_per_product(df, selected_products, show_all=False):
    """
    Returns total sales value grouped by Product.
    """
    if not show_all:
        df = df[df['Product'].isin(selected_products)]
    
    total_sales = df.groupby('Product')['Sales_Value'].sum().reset_index()
    total_sales.columns = ['Product', 'Total_Sales_Value']
    
    return total_sales.sort_values(by='Total_Sales_Value', ascending=False)

unique_products = sorted(df_long['Product'].dropna().unique().tolist())
total_sales_per_product(df_long, selected_products=unique_products, show_all=False)

,Product,Total_Sales_Value
0,200G,8396844
10,S,7255029
4,B,3110738
2,600G,2287818
11,Xtra_Loong,830935
8,O P,653282
1,4_Square,561650
3,800G,515798
9,P,284055
6,Mandazi,132736


In [127]:
# 1.6 Average Sales Value per Transaction per Customer
import io
import base64
import pandas as pd
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display, HTML

def average_sales_per_transaction(df, selected_customers, show_all=False):
    """
    Returns average sales value per transaction grouped by Customer and A/C.
    """
    if not show_all:
        df = df[df['Customer'].isin(selected_customers)]
    
    # Group by both Customer and A/C to keep them in separate columns
    avg_sales = df.groupby(['Customer', 'A/C'])['Sales_Value'].mean().reset_index()
    avg_sales.columns = ['Customer', 'A/C', 'Average_Sales_Value']
    
    return avg_sales.sort_values(by='Average_Sales_Value', ascending=False)

# 1. Get unique, sorted list of customers from the long-form dataframe
customer_list = sorted(df_long['Customer'].dropna().unique().tolist())

# 2. Build the interactive layout widgets
customer_select = widgets.SelectMultiple(
    options=customer_list,
    value=(),  # Starts empty so nothing displays upfront
    description='Customers:',
    rows=8
)

all_checkbox = widgets.Checkbox(
    value=False,
    description='Show All Customers',
    disabled=False,
    indent=True
)

# Dynamic Manual Refresh Button Toggle
refresh_btn = widgets.ToggleButton(
    value=False,
    description='🔄 Refresh Data',
    button_style='info',
    tooltip='Click to refresh calculations'
)

download_output = widgets.Output()

# 3. Connect the components inside the interactive loop
@interact(customers=customer_select, show_all=all_checkbox, refresh=refresh_btn)
def show_and_export_avg_sales(customers, show_all, refresh):
    download_output.clear_output()
    
    # Automatically untoggle the refresh button to prime it for the next click
    if refresh_btn.value:
        refresh_btn.value = False
        
    # Hidden state gate check
    if not show_all and not customers:
        print("Select customer names (Hold Ctrl/Cmd to select multiple) OR check 'Show All Customers'.")
        return
        
    # Execute the query
    result = average_sales_per_transaction(df_long, selected_customers=customers, show_all=show_all)
    
    if result.empty:
        print("No matching data found for this selection.")
        return

    # 4. Excel Generation Engine (in-memory byte buffer)
    output_buffer = io.BytesIO()
    with pd.ExcelWriter(output_buffer, engine='xlsxwriter') as writer:
        result.to_excel(writer, index=False, sheet_name='Avg Sales Report')
        
        workbook  = writer.book
        worksheet = writer.sheets['Avg Sales Report']
        num_format = workbook.add_format({'num_format': '#,##0.00', 'valign': 'top'})
        
        # Set individual column layouts
        worksheet.set_column('A:A', 50)  # Customer
        worksheet.set_column('B:B', 15)  # A/C Number
        worksheet.set_column('C:C', 25, num_format)  # Average Sales Value
        
    excel_data = output_buffer.getvalue()
    b64_encoded = base64.b64encode(excel_data).decode()
    
    # 5. Create the HTML download button widget
    button_html = f'''
    <a download="average_sales_by_account.xlsx" href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64_encoded}" target="_blank">
        <button style="background-color:#1F6E43; color:white; padding:8px 16px; border:none; border-radius:4px; font-weight:bold; cursor:pointer; margin-bottom:12px;">
             Download Filtered Excel Report
        </button>
    </a>
    '''
    
    with download_output:
        display(HTML(button_html))
        
    display(download_output)
    
    # 6. Display the stylized dataframe in your notebook view
    styled_df = result.style.format({'Average_Sales_Value': '{:,.2f}'}).set_properties(**{
        'text-align': 'left',
        'vertical-align': 'top'
    })
    
    display(styled_df)


interactive(children=(SelectMultiple(description='Customers:', options=('ABDALLAH_MUSA_OKUNGO_(2407)_-_KONDELE…

In [128]:
# 1.8 Minimum Sales Value per Transaction per Product per Customer
import io
import base64
import pandas as pd
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display, HTML

def min_sales_per_transaction(df, selected_products, selected_customers, show_all_products=False, show_all_customers=False):
    """
    Returns minimum sales value per transaction grouped by Product, Customer, and A/C.
    """
    # 1. Apply Product Filtering
    if not show_all_products and selected_products:
        df = df[df['Product'].isin(selected_products)]
        
    # 2. Apply Customer Filtering
    if not show_all_customers and selected_customers:
        df = df[df['Customer'].isin(selected_customers)]
    
    # 3. Group by Product, Customer, and A/C to keep columns perfectly isolated
    min_sales = df.groupby(['Product', 'Customer', 'A/C'])['Sales_Value'].min().reset_index()
    min_sales.columns = ['Product', 'Customer', 'A/C', 'Min_Sales_Value']
    
    # Sort ascending so your absolute lowest transaction floors appear first
    return min_sales.sort_values(by='Min_Sales_Value', ascending=True)

# Get sorted unique filter option lists from your long-form dataset
product_list = sorted(df_long['Product'].dropna().unique().tolist())
customer_list = sorted(df_long['Customer'].dropna().unique().tolist())

# Build matching interface selector widgets
product_select = widgets.SelectMultiple(options=product_list, value=(), description='Products:', rows=6)
customer_select = widgets.SelectMultiple(options=customer_list, value=(), description='Customers:', rows=6)

all_prod_check = widgets.Checkbox(value=False, description='All Products')
all_cust_check = widgets.Checkbox(value=False, description='All Customers')

refresh_btn = widgets.ToggleButton(
    value=False,
    description='🔄 Refresh Data',
    button_style='info',
    tooltip='Click to recalculate metrics'
)

download_output = widgets.Output()

# Arrange layout components side-by-side using HBox/VBox structures
ui_layout = widgets.VBox([
    widgets.HBox([product_select, customer_select]),
    widgets.HBox([all_prod_check, all_cust_check, refresh_btn])
])

@interact(products=product_select, customers=customer_select, all_prods=all_prod_check, all_custs=all_cust_check, refresh=refresh_btn)
def show_and_export_min_sales(products, customers, all_prods, all_custs, refresh):
    download_output.clear_output()
    
    # Reset refresh mechanism state for the next user event
    if refresh_btn.value:
        refresh_btn.value = False
        
    # Safe Guard: Hidden state UI logic on notebook execution
    if not all_prods and not products:
        print("Step 1: Select Products (or check 'All Products')")
        return
    if not all_custs and not customers:
        print("Step 2: Select Customers (or check 'All Customers')")
        return
        
    # 1. Execute the query
    result = min_sales_per_transaction(
        df_long, 
        selected_products=products, 
        selected_customers=customers, 
        show_all_products=all_prods, 
        show_all_customers=all_custs
    )
    
    if result.empty:
        print("No matching records found for this custom selection filter.")
        return

    # 2. Excel Generation Engine (In-Memory byte buffer processing)
    output_buffer = io.BytesIO()
    with pd.ExcelWriter(output_buffer, engine='xlsxwriter') as writer:
        result.to_excel(writer, index=False, sheet_name='Min Sales Report')
        
        workbook  = writer.book
        worksheet = writer.sheets['Min Sales Report']
        num_format = workbook.add_format({'num_format': '#,##0.00', 'valign': 'top'})
        
        # Set precise individual column properties
        worksheet.set_column('A:A', 20)  # Product
        worksheet.set_column('B:B', 50)  # Customer Name
        worksheet.set_column('C:C', 15)  # A/C Code Column
        worksheet.set_column('D:D', 25, num_format)  # Min Sales Value
        
    excel_data = output_buffer.getvalue()
    b64_encoded = base64.b64encode(excel_data).decode()
    
    # 3. Create a functional HTML download button widget
    button_html = f'''
    <a download="minimum_sales_by_product_customer.xlsx" href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64_encoded}" target="_blank">
        <button style="background-color:#1F6E43; color:white; padding:8px 16px; border:none; border-radius:4px; font-weight:bold; cursor:pointer; margin-bottom:12px;">
            Download Filtered Excel Report
        </button>
    </a>
    '''
    
    with download_output:
        display(HTML(button_html))
        
    display(download_output)
    
    # 4. Display the stylized, multi-column output view inside the notebook environment
    styled_df = result.style.format({'Min_Sales_Value': '{:,.2f}'}).set_properties(**{
        'text-align': 'left',
        'vertical-align': 'top'
    })
    
    display(styled_df)


interactive(children=(SelectMultiple(description='Products:', options=('200G', '4_Square', '600G', '800G', 'B'…

In [129]:
# 1.9 Maximum Sales Value per Transaction per Customer
import io
import base64
import pandas as pd
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display, HTML

def max_sales_per_transaction(df, selected_products, selected_customers, show_all_products=False, show_all_customers=False):
    """
    Returns maximum sales value per transaction grouped by Product, Customer, and A/C.
    """
    # 1. Apply Product Filtering
    if not show_all_products and selected_products:
        df = df[df['Product'].isin(selected_products)]
        
    # 2. Apply Customer Filtering
    if not show_all_customers and selected_customers:
        df = df[df['Customer'].isin(selected_customers)]
    
    # 3. Group by Product, Customer, and A/C to keep columns perfectly isolated
    max_sales = df.groupby(['Product', 'Customer', 'A/C'])['Sales_Value'].max().reset_index()
    max_sales.columns = ['Product', 'Customer', 'A/C', 'Max_Sales_Value']
    
    # Sort descending so your absolute highest transaction peaks appear first
    return max_sales.sort_values(by='Max_Sales_Value', ascending=False)

# Get sorted unique filter option lists from your long-form dataset
product_list = sorted(df_long['Product'].dropna().unique().tolist())
customer_list = sorted(df_long['Customer'].dropna().unique().tolist())

# Build matching interface selector widgets
product_select = widgets.SelectMultiple(options=product_list, value=(), description='Products:', rows=6)
customer_select = widgets.SelectMultiple(options=customer_list, value=(), description='Customers:', rows=6)

all_prod_check = widgets.Checkbox(value=False, description='All Products')
all_cust_check = widgets.Checkbox(value=False, description='All Customers')

refresh_btn = widgets.ToggleButton(
    value=False,
    description='🔄 Refresh Data',
    button_style='info',
    tooltip='Click to recalculate metrics'
)

download_output = widgets.Output()

# Arrange layout components side-by-side using HBox/VBox structures
ui_layout = widgets.VBox([
    widgets.HBox([product_select, customer_select]),
    widgets.HBox([all_prod_check, all_cust_check, refresh_btn])
])

@interact(products=product_select, customers=customer_select, all_prods=all_prod_check, all_custs=all_cust_check, refresh=refresh_btn)
def show_and_export_max_sales(products, customers, all_prods, all_custs, refresh):
    download_output.clear_output()
    
    # Reset refresh mechanism state for the next user event
    if refresh_btn.value:
        refresh_btn.value = False
        
    # Safe Guard: Hidden state UI logic on notebook execution
    if not all_prods and not products:
        print("Step 1: Select Products (or check 'All Products')")
        return
    if not all_custs and not customers:
        print("Step 2: Select Customers (or check 'All Customers')")
        return
        
    # 1. Execute the query
    result = max_sales_per_transaction(
        df_long, 
        selected_products=products, 
        selected_customers=customers, 
        show_all_products=all_prods, 
        show_all_customers=all_custs
    )
    
    if result.empty:
        print("No matching records found for this custom selection filter.")
        return

    # 2. Excel Generation Engine (In-Memory byte buffer processing)
    output_buffer = io.BytesIO()
    with pd.ExcelWriter(output_buffer, engine='xlsxwriter') as writer:
        result.to_excel(writer, index=False, sheet_name='Max Sales Report')
        
        workbook  = writer.book
        worksheet = writer.sheets['Max Sales Report']
        num_format = workbook.add_format({'num_format': '#,##0.00', 'valign': 'top'})
        
        # Set precise individual column properties
        worksheet.set_column('A:A', 20)  # Product
        worksheet.set_column('B:B', 50)  # Customer Name
        worksheet.set_column('C:C', 15)  # A/C Code Column
        worksheet.set_column('D:D', 25, num_format)  # Max Sales Value
        
    excel_data = output_buffer.getvalue()
    b64_encoded = base64.b64encode(excel_data).decode()
    
    # 3. Create a functional HTML download button widget
    button_html = f'''
    <a download="maximum_sales_by_product_customer.xlsx" href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64_encoded}" target="_blank">
        <button style="background-color:#1F6E43; color:white; padding:8px 16px; border:none; border-radius:4px; font-weight:bold; cursor:pointer; margin-bottom:12px;">
            Download Filtered Excel Report
        </button>
    </a>
    '''
    
    with download_output:
        display(HTML(button_html))
        
    display(download_output)
    
    # 4. Display the stylized, multi-column output view inside the notebook environment
    styled_df = result.style.format({'Max_Sales_Value': '{:,.2f}'}).set_properties(**{
        'text-align': 'left',
        'vertical-align': 'top'
    })
    
    display(styled_df)


interactive(children=(SelectMultiple(description='Products:', options=('200G', '4_Square', '600G', '800G', 'B'…

In [130]:
# 2.0 Total Unique Customers
def total_unique_customers(df, selected_products, selected_customers, show_all_products=False, show_all_customers=False):
    """
    Returns the total number of unique customers for selected products and customers.
    """
    # 1. Apply Product Filtering
    if not show_all_products and selected_products:
        df = df[df['Product'].isin(selected_products)]
        
    # 2. Apply Customer Filtering
    if not show_all_customers and selected_customers:
        df = df[df['Customer'].isin(selected_customers)]
    
    # 3. Count unique customers
    unique_count = df['Customer'].nunique()
    
    return unique_count

total_unique_customers(df_long, selected_products=[], selected_customers=[], show_all_products=True, show_all_customers=True)

1126

In [131]:
# 2.1 Total Sales Value based on Day of the Week
import io
import base64
import pandas as pd
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display, HTML
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# Define calendar map formatting early
month_map = {
    1: 'January', 2: 'February', 3: 'March', 4: 'April', 
    5: 'May', 6: 'June', 7: 'July', 8: 'August', 
    9: 'September', 10: 'October', 11: 'November', 12: 'December'
}

# Clean baseline long dataset month indices to strings
df_long_clean = df_long.copy()
if df_long_clean['Month'].dtype in ['int64', 'float64', 'int32']:
    df_long_clean['Month'] = df_long_clean['Month'].map(month_map)

def total_sales_by_day_consolidated(df, selected_days, selected_quarters, selected_months,
                                   show_all_days=False, show_all_quarters=False, show_all_months=False):
    """
    Applies compounding timeframe filters (Days, Quarters, Months) and aggregates total sales grouped by Day of the Week.
    """
    df_filtered = df.copy()
    
    # 1. Apply Day of Week Filter
    if not show_all_days and selected_days:
        df_filtered = df_filtered[df_filtered['Day_of_Week'].isin(selected_days)]
        
    # 2. Apply Quarter Filter
    if not show_all_quarters and selected_quarters:
        df_filtered = df_filtered[df_filtered['Quarter'].isin(selected_quarters)]
        
    # 3. Apply Month Filter
    if not show_all_months and selected_months:
        df_filtered = df_filtered[df_filtered['Month'].isin(selected_months)]
        
    if df_filtered.empty:
        return pd.DataFrame(columns=['Day_of_Week', 'Total_Sales_Value'])
    
    # 4. Group by Day of the Week and sum sales values
    sales_by_day = df_filtered.groupby('Day_of_Week')['Sales_Value'].sum().reset_index()
    sales_by_day.columns = ['Day_of_Week', 'Total_Sales_Value']
    
    # Enforce strict chronological calendar sorting (Monday to Sunday)
    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    sales_by_day['Day_of_Week'] = pd.Categorical(sales_by_day['Day_of_Week'], categories=day_order, ordered=True)
    
    return sales_by_day.sort_values(by='Day_of_Week')

# Get unique sorted lists for filtering components
day_order_list = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_list = [d for d in day_order_list if d in df_long_clean['Day_of_Week'].unique()]
quarter_list = sorted(df_long_clean['Quarter'].dropna().unique().tolist())

month_order_list = ['January', 'February', 'March', 'April', 'May', 'June', 
                    'July', 'August', 'September', 'October', 'November', 'December']
month_list = [m for m in month_order_list if m in df_long_clean['Month'].unique()]

# Build the Layout Selection Widgets
day_select = widgets.SelectMultiple(options=day_list, value=(), description='Days:', rows=5)
quarter_select = widgets.SelectMultiple(options=quarter_list, value=(), description='Quarters:', rows=4)
month_select = widgets.SelectMultiple(options=month_list, value=(), description='Months:', rows=5)

all_days_check = widgets.Checkbox(value=False, description='All Days')
all_qtrs_check = widgets.Checkbox(value=False, description='All Quarters')
all_mths_check = widgets.Checkbox(value=False, description='All Months')

refresh_btn = widgets.ToggleButton(value=False, description='🔄 Refresh Data', button_style='info')
download_output = widgets.Output()

# Arrange Filter Columns Grid Layout
ui_layout = widgets.VBox([
    widgets.HBox([day_select, quarter_select, month_select]),
    widgets.HBox([all_days_check, all_qtrs_check, all_mths_check, refresh_btn])
])

@interact(days=day_select, quarters=quarter_select, months=month_select,
          all_days=all_days_check, all_qtrs=all_qtrs_check, all_mths=all_mths_check, refresh=refresh_btn)
def show_and_export_time_metrics(days, quarters, months, all_days, all_qtrs, all_mths, refresh):
    download_output.clear_output()
    plt.close('all')
    
    if refresh_btn.value:
        refresh_btn.value = False
        
    # Hidden state gate validations on startup
    if not all_days and not days:
        print("Step 1: Select Days (or check 'All Days')")
        return
    if not all_qtrs and not quarters:
        print("Step 2: Select Quarters (or check 'All Quarters')")
        return
    if not all_mths and not months:
        print("Step 3: Select Months (or check 'All Months')")
        return
        
    # 1. Execute Query Logic
    result = total_sales_by_day_consolidated(
        df_long_clean, days, quarters, months,
        show_all_days=all_days, show_all_quarters=all_qtrs, show_all_months=all_mths
    )
    
    if result.empty:
        print("No matching transaction logs found for this timeframe combination.")
        return

    # 2. Excel Generation Engine
    output_buffer = io.BytesIO()
    with pd.ExcelWriter(output_buffer, engine='xlsxwriter') as writer:
        result.to_excel(writer, index=False, sheet_name='Time Distribution Report')
        workbook  = writer.book
        worksheet = writer.sheets['Time Distribution Report']
        num_format = workbook.add_format({'num_format': '#,##0.00', 'valign': 'top'})
        worksheet.set_column('A:A', 20)
        worksheet.set_column('B:B', 25, num_format)
        
    excel_data = output_buffer.getvalue()
    b64_encoded = base64.b64encode(excel_data).decode()
    
    button_html = f'''
    <a download="sales_by_day_and_timeframe.xlsx" href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64_encoded}" target="_blank">
        <button style="background-color:#1F6E43; color:white; padding:8px 16px; border:none; border-radius:4px; font-weight:bold; cursor:pointer; margin-bottom:12px;">
            Download Filtered Excel Report
        </button>
    </a>
    '''
    
    with download_output:
        display(HTML(button_html))
    display(download_output)
    
    # 3. Display Data Grid
    styled_df = result.style.format({'Total_Sales_Value': '{:,.2f}'}).set_properties(**{
        'text-align': 'left', 'vertical-align': 'top'
    })
    display(styled_df)
    print("\n")
    
    # 4. Generate the Visualization
    plt.figure(figsize=(11, 5))
    sns.set_theme(style="whitegrid")
    
    ax = sns.barplot(
        data=result, 
        x='Day_of_Week', 
        y='Total_Sales_Value', 
        palette='magma', 
        hue='Day_of_Week',
        legend=False
    )
    
    plt.title('Filtered Sales Value Distribution across Days of the Week', fontsize=13, fontweight='bold', pad=15)
    plt.xlabel('Day of the Week', fontsize=11)
    plt.ylabel('Total Sales Value', fontsize=11)
    
    ax.get_yaxis().set_major_formatter(plt.FuncFormatter(lambda x, loc: "{:,}".format(int(x))))
    
    # Add value annotations on top of the bars
    for p in ax.patches:
        height = p.get_height()
        if height > 0:
            ax.annotate(f'{height:,.0f}',
                        (p.get_x() + p.get_width() / 2., height),
                        ha='center', va='bottom', 
                        fontsize=9, fontweight='semibold', color='black',
                        xytext=(0, 4), textcoords='offset points')
            
    plt.tight_layout()
    display(plt.gcf())


interactive(children=(SelectMultiple(description='Days:', options=('Monday', 'Tuesday', 'Wednesday', 'Thursday…

In [132]:
# 2.2 Total Sales Value based on Month, with optional filtering for products and customers
import io
import base64
import pandas as pd
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display, HTML

def total_sales_by_month_product(df, selected_customers, show_all_customers=False):
    """
    Returns total sales value pivoted with Months as rows and Products as columns.
    Converts month indices into string names.
    """
    # 1. Apply Customer Filtering Only (Product filter is removed)
    if not show_all_customers and selected_customers:
        df = df[df['Customer'].isin(selected_customers)]
        
    if df.empty:
        return pd.DataFrame()
    
    # 2. Convert Month numbers into String names safely
    month_map = {
        1: 'January', 2: 'February', 3: 'March', 4: 'April', 
        5: 'May', 6: 'June', 7: 'July', 8: 'August', 
        9: 'September', 10: 'October', 11: 'November', 12: 'December'
    }
    
    # Map the integers to strings if they are numerical, otherwise keep as is
    df = df.copy()
    if df['Month'].dtype in ['int64', 'float64', 'int32']:
        df['Month'] = df['Month'].map(month_map)
        
    # 3. Create a clean matrix structure: Months as Rows, Products as Columns
    pivoted_sales = df.pivot_table(
        index='Month', 
        columns='Product', 
        values='Sales_Value', 
        aggfunc='sum',
        fill_value=0  # Fills missing product combinations with zero instead of NaN
    ).reset_index()
    
    # Enforce logical calendar timeline sorting instead of alphabetical sorting
    month_order = ['January', 'February', 'March', 'April', 'May', 'June', 
                   'July', 'August', 'September', 'October', 'November', 'December']
    pivoted_sales['Month'] = pd.Categorical(pivoted_sales['Month'], categories=month_order, ordered=True)
    
    return pivoted_sales.sort_values(by='Month')

# Get unique sorted option list for customers from your dataset
customer_list = sorted(df_long['Customer'].dropna().unique().tolist())

# Build interface widgets (Product selection widget removed)
customer_select = widgets.SelectMultiple(options=customer_list, value=(), description='Customers:', rows=8)
all_cust_check = widgets.Checkbox(value=False, description='All Customers')

refresh_btn = widgets.ToggleButton(
    value=False,
    description='🔄 Refresh Data',
    button_style='info',
    tooltip='Click to recalculate matrix'
)

download_output = widgets.Output()

# Cleaner arrangement layout
ui_layout = widgets.VBox([
    customer_select,
    widgets.HBox([all_cust_check, refresh_btn])
])

@interact(customers=customer_select, all_custs=all_cust_check, refresh=refresh_btn)
def show_and_export_pivoted_sales(customers, all_custs, refresh):
    download_output.clear_output()
    
    # Reset refresh toggle state
    if refresh_btn.value:
        refresh_btn.value = False
        
    # Safe Guard: Hidden state logic on notebook execution
    if not all_custs and not customers:
        print("Select Customers from the menu (Hold Ctrl/Cmd to select multiple) OR check 'All Customers'.")
        return
        
    # 1. Execute Matrix Query
    result = total_sales_by_month_product(
        df_long, 
        selected_customers=customers, 
        show_all_customers=all_custs
    )
    
    if result.empty:
        print("No matching records found for this selection.")
        return

    # 2. Excel Generation Engine (In-Memory byte buffer processing)
    output_buffer = io.BytesIO()
    with pd.ExcelWriter(output_buffer, engine='xlsxwriter') as writer:
        result.to_excel(writer, index=False, sheet_name='Monthly Product Matrix')
        
        workbook  = writer.book
        worksheet = writer.sheets['Monthly Product Matrix']
        num_format = workbook.add_format({'num_format': '#,##0.00', 'valign': 'top'})
        
        # Format Column A (Month Name)
        worksheet.set_column('A:A', 18)
        
        # Dynamic Formatting: Automatically apply standard currency styles across all dynamic product columns
        # (Loops through columns starting from column index 1 to the end)
        num_cols = len(result.columns)
        for col_idx in range(1, num_cols):
            worksheet.set_column(col_idx, col_idx, 15, num_format)
        
    excel_data = output_buffer.getvalue()
    b64_encoded = base64.b64encode(excel_data).decode()
    
    # 3. Functional HTML download button widget
    button_html = f'''
    <a download="monthly_product_sales_matrix.xlsx" href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64_encoded}" target="_blank">
        <button style="background-color:#1F6E43; color:white; padding:8px 16px; border:none; border-radius:4px; font-weight:bold; cursor:pointer; margin-bottom:12px;">
            Download Filtered Excel Matrix
        </button>
    </a>
    '''
    
    with download_output:
        display(HTML(button_html))
        
    display(download_output)
    
    # 4. Apply dynamic thousands-separator formatting to all numerical columns except the 'Month' label
    numeric_cols = [c for c in result.columns if c != 'Month']
    styled_df = result.style.format({col: '{:,.2f}' for col in numeric_cols}).set_properties(**{
        'text-align': 'right',
        'vertical-align': 'top'
    }).set_properties(subset=['Month'], **{'text-align': 'left'}) # Keep month column text left-aligned
    
    display(styled_df)


interactive(children=(SelectMultiple(description='Customers:', options=('ABDALLAH_MUSA_OKUNGO_(2407)_-_KONDELE…

In [133]:
df_long.head()

,Date,No,Invoice_No,A/C,Customer,Qty,Year,Month,Quarter,Week,Day,Day_of_Week,Product,Sales_Value
2,2026-04-01,57260123,968728,57,JAMES_MWANGI_(057),2546,2026,4,2026Q2,14,1,Wednesday,P,105
3,2026-04-01,62260123,968703,62,ANTONY_MWANGI_(062),1929,2026,4,2026Q2,14,1,Wednesday,P,15
4,2026-04-01,63260102,968705,63,NASHON_KYULE_(063),1737,2026,4,2026Q2,14,1,Wednesday,P,75
5,2026-04-01,82260107,969045,82,PATRICK_WAWERU_GICHUHI_(082),801,2026,4,2026Q2,14,1,Wednesday,P,45
6,2026-04-01,100260101,968561,100,GERALD_GICHUKI_(100),1218,2026,4,2026Q2,14,1,Wednesday,P,165


In [134]:
# 2.3 Total Sales Value based on Quarter, with optional filtering for products, customers, and months

import io
import base64
import pandas as pd
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display, HTML
import plotly.express as px  # For interactive charts with tooltips and data labels

# Standardise the quarter values before filtering or sorting
quarter_map = {1: 'Q1', 2: 'Q2', 3: 'Q3', 4: 'Q4'}
month_map = {
    1: 'January', 2: 'February', 3: 'March', 4: 'April',
    5: 'May', 6: 'June', 7: 'July', 8: 'August',
    9: 'September', 10: 'October', 11: 'November', 12: 'December'
}

df_long_clean = df_long.copy().dropna(subset=['Quarter']).copy()

# Convert month numbers to names when needed
if pd.api.types.is_numeric_dtype(df_long_clean['Month']):
    df_long_clean['Month'] = df_long_clean['Month'].map(month_map)

# Convert period/int quarter values into consistent quarter labels like Q1, Q2, ...
def normalize_quarter_label(value):
    if pd.isna(value):
        return None
    if isinstance(value, str):
        value = value.strip().upper()
        if value.startswith('Q'):
            return value
        if value.isdigit():
            return quarter_map.get(int(value), value)
        return value
    if hasattr(value, 'quarter'):
        return quarter_map.get(int(value.quarter), str(value))
    if isinstance(value, (int, float)):
        return quarter_map.get(int(value), str(value))
    return str(value)

df_long_clean['Quarter'] = df_long_clean['Quarter'].map(normalize_quarter_label)

def total_sales_by_quarter_product(df, selected_customers, selected_products, selected_quarters, selected_months,
                                   show_all_customers=False, show_all_products=False,
                                   show_all_quarters=False, show_all_months=False):
    """
    Returns total sales value pivoted with Quarters as rows and Products as columns.
    Filters dynamically by Customer, Product, Quarter, and Month strings.
    """
    df_filtered = df.copy()

    if not show_all_customers and selected_customers:
        df_filtered = df_filtered[df_filtered['Customer'].isin(selected_customers)]

    if not show_all_products and selected_products:
        df_filtered = df_filtered[df_filtered['Product'].isin(selected_products)]

    if not show_all_quarters and selected_quarters:
        df_filtered = df_filtered[df_filtered['Quarter'].isin(selected_quarters)]

    if not show_all_months and selected_months:
        df_filtered = df_filtered[df_filtered['Month'].isin(selected_months)]

    if df_filtered.empty:
        return pd.DataFrame()

    pivoted_sales = df_filtered.pivot_table(
        index='Quarter',
        columns='Product',
        values='Sales_Value',
        aggfunc='sum',
        fill_value=0
    ).reset_index()

    quarter_order = ['Q1', 'Q2', 'Q3', 'Q4']
    pivoted_sales = pivoted_sales[pivoted_sales['Quarter'].isin(quarter_order)].copy()
    pivoted_sales['Quarter'] = pd.Categorical(
        pivoted_sales['Quarter'],
        categories=quarter_order,
        ordered=True
    )

    return pivoted_sales.sort_values(by='Quarter')

# Extract options lists for filters from the updated dataframe
customer_list = sorted(df_long_clean['Customer'].dropna().unique().tolist())
product_list = sorted(df_long_clean['Product'].dropna().unique().tolist())
quarter_list = [q for q in ['Q1', 'Q2', 'Q3', 'Q4'] if q in df_long_clean['Quarter'].dropna().unique()]
month_order_list = ['January', 'February', 'March', 'April', 'May', 'June',
                    'July', 'August', 'September', 'October', 'November', 'December']
month_list = [m for m in month_order_list if m in df_long_clean['Month'].dropna().unique()]

# Create Multi-Select List Widgets (Months now use string names)
customer_select = widgets.SelectMultiple(options=customer_list, value=(), description='Customers:', rows=5)
product_select = widgets.SelectMultiple(options=product_list, value=(), description='Products:', rows=5)
quarter_select = widgets.SelectMultiple(options=quarter_list, value=(), description='Quarters:', rows=4)
month_select = widgets.SelectMultiple(options=month_list, value=(), description='Months:', rows=4)

# Create Checkbox Toggles
all_cust_check = widgets.Checkbox(value=False, description='All Customers')
all_prod_check = widgets.Checkbox(value=False, description='All Products')
all_qtr_check = widgets.Checkbox(value=False, description='All Quarters')
all_mth_check = widgets.Checkbox(value=False, description='All Months')

refresh_btn = widgets.ToggleButton(value=False, description='🔄 Refresh Data', button_style='info')
download_output = widgets.Output()

# Arrange Dashboard Grid Layout Layout View
ui_layout = widgets.VBox([
    widgets.HBox([customer_select, product_select]),
    widgets.HBox([all_cust_check, all_prod_check]),
    widgets.HTML("<hr style='margin:10px 0;'>"),
    widgets.HBox([quarter_select, month_select]),
    widgets.HBox([all_qtr_check, all_mth_check, refresh_btn])
])

@interact(customers=customer_select, products=product_select, quarters=quarter_select, months=month_select,
          all_custs=all_cust_check, all_prods=all_prod_check, all_qtrs=all_qtr_check, all_mths=all_mth_check, refresh=refresh_btn)
def show_and_export_quarterly_matrix(customers, products, quarters, months, all_custs, all_prods, all_qtrs, all_mths, refresh):
    download_output.clear_output()

    if refresh_btn.value:
        refresh_btn.value = False

    if not all_custs and not customers:
        print("Step 1: Choose Customers (or click 'All Customers')")
        return
    if not all_prods and not products:
        print("Step 2: Choose Products (or click 'All Products')")
        return
    if not all_qtrs and not quarters:
        print("Step 3: Choose Quarters (or click 'All Quarters')")
        return
    if not all_mths and not months:
        print("Step 4: Choose Months (or click 'All Months')")
        return

    result = total_sales_by_quarter_product(
        df_long_clean, customers, products, quarters, months,
        show_all_customers=all_custs, show_all_products=all_prods,
        show_all_quarters=all_qtrs, show_all_months=all_mths
    )

    if result.empty:
        print("No matching transaction data fits this combined filter query.")
        return

    output_buffer = io.BytesIO()
    with pd.ExcelWriter(output_buffer, engine='xlsxwriter') as writer:
        result.to_excel(writer, index=False, sheet_name='Quarterly Product Matrix')

        workbook = writer.book
        worksheet = writer.sheets['Quarterly Product Matrix']
        num_format = workbook.add_format({'num_format': '#,##0.00', 'valign': 'top'})

        worksheet.set_column('A:A', 15)
        for col_idx in range(1, len(result.columns)):
            worksheet.set_column(col_idx, col_idx, 15, num_format)

    excel_data = output_buffer.getvalue()
    b64_encoded = base64.b64encode(excel_data).decode()

    button_html = f'''
    <a download="quarterly_product_sales_matrix.xlsx" href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64_encoded}" target="_blank">
        <button style="background-color:#1F6E43; color:white; padding:8px 16px; border:none; border-radius:4px; font-weight:bold; cursor:pointer; margin-bottom:12px;">
            Download Filtered Excel Matrix
        </button>
    </a>
    '''

    with download_output:
        display(HTML(button_html))
    display(download_output)

    # Create the summary table
    numeric_cols = [c for c in result.columns if c != 'Quarter']
    styled_df = result.style.format({col: '{:,.2f}' for col in numeric_cols}).set_properties(**{
        'text-align': 'right', 'vertical-align': 'top'
    }).set_properties(subset=['Quarter'], **{'text-align': 'left'})

    display(styled_df)

    # --- PREPARE DATA FOR GRAPH SORTING ---
    df_melted = result.melt(id_vars=['Quarter'], var_name='Product', value_name='Sales Value')
    
    # Calculate global product rank across active selections to force largest to smallest order
    product_totals = df_melted.groupby('Product')['Sales Value'].sum().reset_index()
    sorted_products = product_totals.sort_values(by='Sales Value', ascending=False)['Product'].tolist()

    # --- 1. PRODUCT COMPARISON GRAPH (LARGEST TO SMALLEST) ---
    fig_prod = px.bar(
        df_melted, 
        x='Product', 
        y='Sales Value', 
        color='Quarter',
        barmode='group',
        category_orders={'Product': sorted_products},  # Forces largest to smallest sorting
        title='Total Sales Value by Product (Sorted Largest to Smallest)',
        text='Sales Value',
        labels={'Sales Value': 'Sales Value ($)', 'Product': 'Product Type'}
    )
    fig_prod.update_traces(
        texttemplate='%{text:,.2f}',
        textposition='outside',
        hovertemplate='<b>Product:</b> %{x}<br><b>Quarter:</b> %{data.name}<br><b>Sales:</b> %{y:,.2f}<extra></extra>'
    )
    fig_prod.update_layout(yaxis_title="Sales Value", xaxis_title="Products", height=450, uniformtext_mode='hide')
    fig_prod.show()

    # --- 2. QUARTER BREAKDOWN GRAPH ---
    # Aggregate total sales per quarter across all selected items
    quarter_totals = df_melted.groupby('Quarter', observed=False)['Sales Value'].sum().reset_index()
    
    fig_qtr = px.bar(
        quarter_totals,
        x='Quarter',
        y='Sales Value',
        color='Quarter',
        title='Total Overall Sales Value by Quarter',
        text='Sales Value',
        labels={'Sales Value': 'Total Sales ($)', 'Quarter': 'Quarter Period'}
    )
    fig_qtr.update_traces(
        texttemplate='%{text:,.2f}',
        textposition='outside',
        hovertemplate='<b>Quarter:</b> %{x}<br><b>Total Sales:</b> %{y:,.2f}<extra></extra>'
    )
    fig_qtr.update_layout(yaxis_title="Total Sales Value", xaxis_title="Quarters", height=400, showlegend=False)
    fig_qtr.show()


interactive(children=(SelectMultiple(description='Customers:', options=('ABDALLAH_MUSA_OKUNGO_(2407)_-_KONDELE…

## TIME ANALYSIS

In [135]:
# DAILY
import io
import base64
import pandas as pd
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display, HTML
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# Define calendar map formatting early
month_map = {
    1: 'January', 2: 'February', 3: 'March', 4: 'April', 
    5: 'May', 6: 'June', 7: 'July', 8: 'August', 
    9: 'September', 10: 'October', 11: 'November', 12: 'December'
}

# Clean baseline long dataset month indices to strings
df_long_clean = df_long.copy()
if df_long_clean['Month'].dtype in ['int64', 'float64', 'int32']:
    df_long_clean['Month'] = df_long_clean['Month'].map(month_map)

def total_sales_by_day_consolidated(df, selected_days, selected_quarters, selected_months,
                                   show_all_days=False, show_all_quarters=False, show_all_months=False):
    """
    Applies compounding timeframe filters (Days, Quarters, Months) and aggregates total sales grouped by Day of the Week.
    """
    df_filtered = df.copy()
    
    # 1. Apply Day of Week Filter
    if not show_all_days and selected_days:
        df_filtered = df_filtered[df_filtered['Day_of_Week'].isin(selected_days)]
        
    # 2. Apply Quarter Filter
    if not show_all_quarters and selected_quarters:
        df_filtered = df_filtered[df_filtered['Quarter'].isin(selected_quarters)]
        
    # 3. Apply Month Filter
    if not show_all_months and selected_months:
        df_filtered = df_filtered[df_filtered['Month'].isin(selected_months)]
        
    if df_filtered.empty:
        return pd.DataFrame(columns=['Day_of_Week', 'Total_Sales_Value'])
    
    # 4. Group by Day of the Week and sum sales values
    sales_by_day = df_filtered.groupby('Day_of_Week')['Sales_Value'].sum().reset_index()
    sales_by_day.columns = ['Day_of_Week', 'Total_Sales_Value']
    
    # Enforce strict chronological calendar sorting (Monday to Sunday)
    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    sales_by_day['Day_of_Week'] = pd.Categorical(sales_by_day['Day_of_Week'], categories=day_order, ordered=True)
    
    return sales_by_day.sort_values(by='Day_of_Week')

# Get unique sorted lists for filtering components
day_order_list = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_list = [d for d in day_order_list if d in df_long_clean['Day_of_Week'].unique()]
quarter_list = sorted(df_long_clean['Quarter'].dropna().unique().tolist())

month_order_list = ['January', 'February', 'March', 'April', 'May', 'June', 
                    'July', 'August', 'September', 'October', 'November', 'December']
month_list = [m for m in month_order_list if m in df_long_clean['Month'].unique()]

# Build the Layout Selection Widgets
day_select = widgets.SelectMultiple(options=day_list, value=(), description='Days:', rows=5)
quarter_select = widgets.SelectMultiple(options=quarter_list, value=(), description='Quarters:', rows=4)
month_select = widgets.SelectMultiple(options=month_list, value=(), description='Months:', rows=5)

all_days_check = widgets.Checkbox(value=False, description='All Days')
all_qtrs_check = widgets.Checkbox(value=False, description='All Quarters')
all_mths_check = widgets.Checkbox(value=False, description='All Months')

refresh_btn = widgets.ToggleButton(value=False, description='🔄 Refresh Data', button_style='info')
download_output = widgets.Output()

# Arrange Filter Columns Grid Layout
ui_layout = widgets.VBox([
    widgets.HBox([day_select, quarter_select, month_select]),
    widgets.HBox([all_days_check, all_qtrs_check, all_mths_check, refresh_btn])
])

@interact(days=day_select, quarters=quarter_select, months=month_select,
          all_days=all_days_check, all_qtrs=all_qtrs_check, all_mths=all_mths_check, refresh=refresh_btn)
def show_and_export_time_metrics(days, quarters, months, all_days, all_qtrs, all_mths, refresh):
    download_output.clear_output()
    plt.close('all')
    
    if refresh_btn.value:
        refresh_btn.value = False
        
    # Hidden state gate validations on startup
    if not all_days and not days:
        print("Step 1: Select Days (or check 'All Days')")
        return
    if not all_qtrs and not quarters:
        print("Step 2: Select Quarters (or check 'All Quarters')")
        return
    if not all_mths and not months:
        print("Step 3: Select Months (or check 'All Months')")
        return
        
    # 1. Execute Query Logic
    result = total_sales_by_day_consolidated(
        df_long_clean, days, quarters, months,
        show_all_days=all_days, show_all_quarters=all_qtrs, show_all_months=all_mths
    )
    
    if result.empty:
        print("No matching transaction logs found for this timeframe combination.")
        return

    # 2. Excel Generation Engine
    output_buffer = io.BytesIO()
    with pd.ExcelWriter(output_buffer, engine='xlsxwriter') as writer:
        result.to_excel(writer, index=False, sheet_name='Time Distribution Report')
        workbook  = writer.book
        worksheet = writer.sheets['Time Distribution Report']
        num_format = workbook.add_format({'num_format': '#,##0.00', 'valign': 'top'})
        worksheet.set_column('A:A', 20)
        worksheet.set_column('B:B', 25, num_format)
        
    excel_data = output_buffer.getvalue()
    b64_encoded = base64.b64encode(excel_data).decode()
    
    button_html = f'''
    <a download="sales_by_day_and_timeframe.xlsx" href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64_encoded}" target="_blank">
        <button style="background-color:#1F6E43; color:white; padding:8px 16px; border:none; border-radius:4px; font-weight:bold; cursor:pointer; margin-bottom:12px;">
            Download Filtered Excel Report
        </button>
    </a>
    '''
    
    with download_output:
        display(HTML(button_html))
    display(download_output)
    
    # 3. Display Data Grid
    styled_df = result.style.format({'Total_Sales_Value': '{:,.2f}'}).set_properties(**{
        'text-align': 'left', 'vertical-align': 'top'
    })
    display(styled_df)
    print("\n")
    
    # 4. Generate the Visualization
    plt.figure(figsize=(11, 5))
    sns.set_theme(style="whitegrid")
    
    ax = sns.barplot(
        data=result, 
        x='Day_of_Week', 
        y='Total_Sales_Value', 
        palette='magma', 
        hue='Day_of_Week',
        legend=False
    )
    
    plt.title('Filtered Sales Value Distribution across Days of the Week', fontsize=13, fontweight='bold', pad=15)
    plt.xlabel('Day of the Week', fontsize=11)
    plt.ylabel('Total Sales Value', fontsize=11)
    
    ax.get_yaxis().set_major_formatter(plt.FuncFormatter(lambda x, loc: "{:,}".format(int(x))))
    
    # Add value annotations on top of the bars
    for p in ax.patches:
        height = p.get_height()
        if height > 0:
            ax.annotate(f'{height:,.0f}',
                        (p.get_x() + p.get_width() / 2., height),
                        ha='center', va='bottom', 
                        fontsize=9, fontweight='semibold', color='black',
                        xytext=(0, 4), textcoords='offset points')
            
    plt.tight_layout()
    display(plt.gcf())


interactive(children=(SelectMultiple(description='Days:', options=('Monday', 'Tuesday', 'Wednesday', 'Thursday…

In [136]:
# WEEKLY
import io
import base64
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display, HTML
import plotly.express as px

# 1. CLEAN AND NORMALISE DATAFRAME WITHOUT DROPPING DATA
df_long_clean = df_long.copy()

# Ensure Week is safe (handle numbers or strings)
df_long_clean['Week_Number'] = pd.to_numeric(df_long_clean['Week'], errors='coerce').fillna(0).astype(int)

# Map Quarter to uniform string labels ('Q1', 'Q2', etc.)
quarter_map = {1: 'Q1', 2: 'Q2', 3: 'Q3', 4: 'Q4', '1': 'Q1', '2': 'Q2', '3': 'Q3', '4': 'Q4'}
df_long_clean['Quarter_Label'] = df_long_clean['Quarter'].map(quarter_map).fillna(df_long_clean['Quarter'].astype(str))

# Map Month to uniform string names
month_map = {
    1: 'January', 2: 'February', 3: 'March', 4: 'April', 
    5: 'May', 6: 'June', 7: 'July', 8: 'August', 
    9: 'September', 10: 'October', 11: 'November', 12: 'December'
}
if pd.api.types.is_numeric_dtype(df_long_clean['Month']):
    df_long_clean['Month_Label'] = df_long_clean['Month'].map(month_map)
else:
    df_long_clean['Month_Label'] = df_long_clean['Month'].astype(str).str.strip().str.capitalize()

# 2. CORE TIME-FILTER MATRIX CALCULATION ENGINE
def total_sales_by_weekly_basis(df, selected_weeks, selected_quarters, selected_months,
                                show_all_weeks=False, show_all_quarters=False, show_all_months=False):
    """
    Filters data dynamically and pivots it with Weeks as rows and Products as columns, 
    including a Total Sales column.
    """
    df_filtered = df.copy()
    
    # Apply Week Filter
    if not show_all_weeks and selected_weeks:
        df_filtered = df_filtered[df_filtered['Week_Number'].isin(selected_weeks)]
        
    # Apply Quarter Filter
    if not show_all_quarters and selected_quarters:
        df_filtered = df_filtered[df_filtered['Quarter_Label'].isin(selected_quarters)]
        
    # Apply Month Filter
    if not show_all_months and selected_months:
        df_filtered = df_filtered[df_filtered['Month_Label'].isin(selected_months)]
        
    if df_filtered.empty:
        return pd.DataFrame()
    
    # Pivot data: Weeks as Rows, Products as Columns
    pivoted_weeks = df_filtered.pivot_table(
        index='Week_Number',
        columns='Product',
        values='Sales_Value',
        aggfunc='sum',
        fill_value=0
    ).reset_index()
    
    # Calculate Total Sales column (summing across all product columns)
    product_cols = [c for c in pivoted_weeks.columns if c != 'Week_Number']
    pivoted_weeks['Total_Sales_Value'] = pivoted_weeks[product_cols].sum(axis=1)
    
    return pivoted_weeks.sort_values(by='Week_Number').reset_index(drop=True)

# 3. EXTRACT CLEAN OPTION LISTS FOR WIDGETS
total_weeks = sorted([w for w in df_long_clean['Week_Number'].unique() if w > 0])
total_quarters = sorted([q for q in df_long_clean['Quarter_Label'].unique() if q not in ['nan', 'None', '']])

month_order_list = ['January', 'February', 'March', 'April', 'May', 'June', 
                    'July', 'August', 'September', 'October', 'November', 'December']
total_months = [m for m in month_order_list if m in df_long_clean['Month_Label'].unique()]

# 4. CONSTRUCT INTERACTIVE DASHBOARD WIDGETS
week_select = widgets.SelectMultiple(options=total_weeks, value=(), description='Weeks:', rows=5)
quarter_select = widgets.SelectMultiple(options=total_quarters, value=(), description='Quarters:', rows=4)
month_select = widgets.SelectMultiple(options=total_months, value=(), description='Months:', rows=5)

all_wk_check = widgets.Checkbox(value=False, description='All Weeks')
all_qtr_check = widgets.Checkbox(value=False, description='All Quarters')
all_mth_check = widgets.Checkbox(value=False, description='All Months')

refresh_btn = widgets.ToggleButton(value=False, description='🔄 Refresh Data', button_style='info')
download_output = widgets.Output()

ui_layout = widgets.VBox([
    widgets.HBox([week_select, quarter_select, month_select]),
    widgets.HBox([all_wk_check, all_qtr_check, all_mth_check, refresh_btn])
])

# 5. LIVE DASHBOARD SELECTION WRAPPER CONTROLLER
@interact(weeks=week_select, quarters=quarter_select, months=month_select,
          all_wks=all_wk_check, all_qtrs=all_qtr_check, all_mths=all_mth_check, refresh=refresh_btn)
def show_and_export_weekly_dashboard(weeks, quarters, months, all_wks, all_qtrs, all_mths, refresh):
    download_output.clear_output()

    if refresh_btn.value:
        refresh_btn.value = False

    # Validation Checks
    if not all_wks and not weeks:
        print("Step 1: Choose Weeks (or click 'All Weeks')")
        return
    if not all_qtrs and not quarters:
        print("Step 2: Choose Quarters (or click 'All Quarters')")
        return
    if not all_mths and not months:
        print("Step 3: Choose Months (or click 'All Months')")
        return

    # Process metrics collection
    result = total_sales_by_weekly_basis(
        df_long_clean, weeks, quarters, months,
        show_all_weeks=all_wks, show_all_quarters=all_qtrs, show_all_months=all_mths
    )

    if result.empty:
        print("No matching transaction data fits this combined filter query.")
        return

    # Excel Exporter Architecture
    output_buffer = io.BytesIO()
    with pd.ExcelWriter(output_buffer, engine='xlsxwriter') as writer:
        result.to_excel(writer, index=False, sheet_name='Weekly Product Breakdown')
        workbook = writer.book
        worksheet = writer.sheets['Weekly Product Breakdown']
        num_format = workbook.add_format({'num_format': '#,##0.00', 'valign': 'top'})
        
        worksheet.set_column('A:A', 15)
        for col_idx in range(1, len(result.columns)):
            worksheet.set_column(col_idx, col_idx, 18, num_format)

    excel_data = output_buffer.getvalue()
    b64_encoded = base64.b64encode(excel_data).decode()

    button_html = f'''
    <a download="weekly_product_sales_matrix.xlsx" href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64_encoded}" target="_blank">
        <button style="background-color:#1F6E43; color:white; padding:8px 16px; border:none; border-radius:4px; font-weight:bold; cursor:pointer; margin-bottom:12px;">
            Download Filtered Excel Matrix
        </button>
    </a>
    '''

    with download_output:
        display(HTML(button_html))
    display(download_output)

    # Render Clean Pivot Table Frame with custom right alignment formatting
    numeric_cols = [c for c in result.columns if c != 'Week_Number']
    styled_df = result.style.format({col: '{:,.2f}' for col in numeric_cols}).set_properties(**{
        'text-align': 'right', 'vertical-align': 'top'
    }).set_properties(subset=['Week_Number'], **{'text-align': 'left'})
    
    display(styled_df)

    # --- GRAPH 1: INTERACTIVE WEEKLY TREND PROGRESSION (RETAINED OVERSIGHT) ---
    fig_trend = px.line(
        result,
        x='Week_Number',
        y='Total_Sales_Value',
        title='Weekly Total Sales Value Performance Curve',
        markers=True,
        labels={'Total_Sales_Value': 'Total Combined Sales ($)', 'Week_Number': 'Week Number'}
    )
    fig_trend.update_traces(
        text=result['Total_Sales_Value'],
        texttemplate='%{text:,.2f}',
        textposition='top center',
        hovertemplate='<b>Timeframe:</b> Week %{x}<br><b>Combined Revenue:</b> %{y:,.2f}<extra></extra>',
        line=dict(width=3, color='#1F6E43')
    )
    fig_trend.update_layout(
        yaxis_title="Total Combined Sales", xaxis_title="Week Number",
        xaxis=dict(tickmode='linear', dtick=1), height=400, margin=dict(t=50, b=50, l=50, r=50)
    )
    fig_trend.show()

    # --- GRAPH 2: PRODUCT VOLUMETRIC HISTOGRAM (SORTED LARGEST TO SMALLEST) ---
    # Melt the product columns back to a long structure specifically for the chart
    product_only_cols = [c for c in result.columns if c not in ['Week_Number', 'Total_Sales_Value']]
    df_melted_products = result.melt(id_vars=['Week_Number'], value_vars=product_only_cols, 
                                     var_name='Product', value_name='Sales Value')
    
    # Group by product to establish clean sorting rankings across filtered frame
    prod_totals = df_melted_products.groupby('Product')['Sales Value'].sum().reset_index()
    sorted_product_order = prod_totals.sort_values(by='Sales Value', ascending=False)['Product'].tolist()
    
    fig_prod = px.bar(
        df_melted_products,
        x='Product',
        y='Sales Value',
        color='Week_Number',
        color_continuous_scale=px.colors.sequential.Viridis,
        category_orders={'Product': sorted_product_order}, # Largest to smallest order
        title='Total Product Contribution Analysis (Grouped By Week)',
        text='Sales Value',
        labels={'Sales Value': 'Product Sales ($)', 'Product': 'Product Classification', 'Week_Number': 'Week'}
    )
    fig_prod.update_traces(
        texttemplate='%{text:,.2f}',
        textposition='outside',
        hovertemplate='<b>Product Name:</b> %{x}<br><b>Week Context:</b> Week %{customdata[0]}<br><b>Sales Segment:</b> %{y:,.2f}<extra></extra>',
        customdata=np.stack((df_melted_products['Week_Number'],), axis=-1)
    )
    fig_prod.update_layout(
        title='Total Product Contribution Analysis (Grouped By Week)',
        yaxis_title="Aggregated Product Sales",
        xaxis_title="Products",
        height=450,
        margin=dict(t=50, b=50, l=50, r=50)
    )
    fig_prod.show()


interactive(children=(SelectMultiple(description='Weeks:', options=(np.int64(14), np.int64(15), np.int64(16), …

In [137]:
# MONTHLY
import io
import base64
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display, HTML
import plotly.express as px

# 1. CLEAN AND NORMALISE DATAFRAME WITHOUT DROPPING DATA
df_long_clean = df_long.copy()

# Map Quarter to uniform string labels ('Q1', 'Q2', etc.)
quarter_map = {1: 'Q1', 2: 'Q2', 3: 'Q3', 4: 'Q4', '1': 'Q1', '2': 'Q2', '3': 'Q3', '4': 'Q4'}
df_long_clean['Quarter_Label'] = df_long_clean['Quarter'].map(quarter_map).fillna(df_long_clean['Quarter'].astype(str))

# Map Month to uniform string names for structured filtering and sorting
month_map = {
    1: 'January', 2: 'February', 3: 'March', 4: 'April', 
    5: 'May', 6: 'June', 7: 'July', 8: 'August', 
    9: 'September', 10: 'October', 11: 'November', 12: 'December'
}
if pd.api.types.is_numeric_dtype(df_long_clean['Month']):
    df_long_clean['Month_Label'] = df_long_clean['Month'].map(month_map)
else:
    df_long_clean['Month_Label'] = df_long_clean['Month'].astype(str).str.strip().str.capitalize()

# Define standard chronological calendar order for month alignment
month_order_list = ['January', 'February', 'March', 'April', 'May', 'June', 
                    'July', 'August', 'September', 'October', 'November', 'December']

# 2. CORE MONTHLY TIME-FILTER MATRIX CALCULATION ENGINE
def total_sales_by_monthly_basis(df, selected_months, selected_quarters, 
                                 show_all_months=False, show_all_quarters=False):
    """
    Filters data dynamically and pivots it with Months as rows and Products as columns, 
    including a Total Sales column.
    """
    df_filtered = df.copy()
        
    # Apply Quarter Filter
    if not show_all_quarters and selected_quarters:
        df_filtered = df_filtered[df_filtered['Quarter_Label'].isin(selected_quarters)]
        
    # Apply Month Filter
    if not show_all_months and selected_months:
        df_filtered = df_filtered[df_filtered['Month_Label'].isin(selected_months)]
        
    if df_filtered.empty:
        return pd.DataFrame()
    
    # Pivot data: Months as Rows, Products as Columns
    pivoted_months = df_filtered.pivot_table(
        index='Month_Label',
        columns='Product',
        values='Sales_Value',
        aggfunc='sum',
        fill_value=0
    ).reset_index()
    
    # Calculate Total Sales column (summing across all product columns)
    product_cols = [c for c in pivoted_months.columns if c != 'Month_Label']
    pivoted_months['Total_Sales_Value'] = pivoted_months[product_cols].sum(axis=1)
    
    # Enforce strict chronological calendar sorting instead of alphabetical sorting
    pivoted_months['Month_Label'] = pd.Categorical(pivoted_months['Month_Label'], categories=month_order_list, ordered=True)
    
    return pivoted_months.sort_values(by='Month_Label').reset_index(drop=True)

# 3. EXTRACT CLEAN OPTION LISTS FOR WIDGETS
total_quarters = sorted([q for q in df_long_clean['Quarter_Label'].unique() if q not in ['nan', 'None', '']])
total_months = [m for m in month_order_list if m in df_long_clean['Month_Label'].unique()]

# 4. CONSTRUCT INTERACTIVE DASHBOARD WIDGETS (ADJUSTED FOR ROW VISIBILITY)
quarter_select = widgets.SelectMultiple(options=total_quarters, value=(), description='Quarters:', rows=4)
month_select = widgets.SelectMultiple(options=total_months, value=(), description='Months:', rows=7) # Clear visibility up to 7 options

all_qtr_check = widgets.Checkbox(value=False, description='All Quarters')
all_mth_check = widgets.Checkbox(value=False, description='All Months')

refresh_btn = widgets.ToggleButton(value=False, description='🔄 Refresh Data', button_style='info')
download_output = widgets.Output()

ui_layout = widgets.VBox([
    widgets.HBox([quarter_select, month_select]),
    widgets.HBox([all_qtr_check, all_mth_check, refresh_btn])
])

# 5. LIVE DASHBOARD SELECTION WRAPPER CONTROLLER
@interact(quarters=quarter_select, months=month_select,
          all_qtrs=all_qtr_check, all_mths=all_mth_check, refresh=refresh_btn)
def show_and_export_monthly_dashboard(quarters, months, all_qtrs, all_mths, refresh):
    download_output.clear_output()

    if refresh_btn.value:
        refresh_btn.value = False

    # Validation Checks
    if not all_qtrs and not quarters:
        print("Step 1: Choose Quarters (or click 'All Quarters')")
        return
    if not all_mths and not months:
        print("Step 2: Choose Months (or click 'All Months')")
        return

    # Process metrics collection
    result = total_sales_by_monthly_basis(
        df_long_clean, months, quarters,
        show_all_months=all_mths, show_all_quarters=all_qtrs
    )

    if result.empty:
        print("No matching transaction data fits this combined filter query.")
        return

    # Excel Exporter Architecture
    output_buffer = io.BytesIO()
    with pd.ExcelWriter(output_buffer, engine='xlsxwriter') as writer:
        result.to_excel(writer, index=False, sheet_name='Monthly Product Breakdown')
        workbook = writer.book
        worksheet = writer.sheets['Monthly Product Breakdown']
        num_format = workbook.add_format({'num_format': '#,##0.00', 'valign': 'top'})
        
        worksheet.set_column('A:A', 15)
        for col_idx in range(1, len(result.columns)):
            worksheet.set_column(col_idx, col_idx, 18, num_format)

    excel_data = output_buffer.getvalue()
    b64_encoded = base64.b64encode(excel_data).decode()

    button_html = f'''
    <a download="monthly_product_sales_matrix.xlsx" href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64_encoded}" target="_blank">
        <button style="background-color:#1F6E43; color:white; padding:8px 16px; border:none; border-radius:4px; font-weight:bold; cursor:pointer; margin-bottom:12px;">
            Download Filtered Excel Matrix
        </button>
    </a>
    '''

    with download_output:
        display(HTML(button_html))
    display(download_output)

    # Render Clean Pivot Table Frame with custom right alignment formatting
    numeric_cols = [c for c in result.columns if c != 'Month_Label']
    styled_df = result.style.format({col: '{:,.2f}' for col in numeric_cols}).set_properties(**{
        'text-align': 'right', 'vertical-align': 'top'
    }).set_properties(subset=['Month_Label'], **{'text-align': 'left'})
    
    display(styled_df)

    # --- GRAPH 1: INTERACTIVE MONTHLY TREND PROGRESSION ---
    fig_trend = px.line(
        result,
        x='Month_Label',
        y='Total_Sales_Value',
        title='Monthly Total Sales Value Performance Curve',
        markers=True,
        labels={'Total_Sales_Value': 'Total Combined Sales ($)', 'Month_Label': 'Month'}
    )
    fig_trend.update_traces(
        text=result['Total_Sales_Value'],
        texttemplate='%{text:,.2f}',
        textposition='top center',
        hovertemplate='<b>Timeframe:</b> %{x}<br><b>Combined Revenue:</b> %{y:,.2f}<extra></extra>',
        line=dict(width=3, color='#1F6E43')
    )
    fig_trend.update_layout(
        yaxis_title="Total Combined Sales", xaxis_title="Month",
        height=400, margin=dict(t=50, b=50, l=50, r=50)
    )
    fig_trend.show()

    # --- GRAPH 2: PRODUCT VOLUMETRIC HISTOGRAM (SORTED LARGEST TO SMALLEST) ---
    product_only_cols = [c for c in result.columns if c not in ['Month_Label', 'Total_Sales_Value']]
    df_melted_products = result.melt(id_vars=['Month_Label'], value_vars=product_only_cols, 
                                     var_name='Product', value_name='Sales Value')
    
    # Group by product to establish clean sorting rankings from largest to smallest
    prod_totals = df_melted_products.groupby('Product')['Sales Value'].sum().reset_index()
    sorted_product_order = prod_totals.sort_values(by='Sales Value', ascending=False)['Product'].tolist()
    
    fig_prod = px.bar(
        df_melted_products,
        x='Product',
        y='Sales Value',
        color='Month_Label',
        color_discrete_sequence=px.colors.qualitative.Safe,
        category_orders={'Product': sorted_product_order}, # Forces largest to smallest sorting
        title='Total Product Contribution Analysis (Grouped By Month)',
        text='Sales Value',
        labels={'Sales Value': 'Product Sales ($)', 'Product': 'Product Classification', 'Month_Label': 'Month'}
    )
    fig_prod.update_traces(
        texttemplate='%{text:,.2f}',
        textposition='outside',
        hovertemplate='<b>Product Name:</b> %{x}<br><b>Month Context:</b> %{customdata}<br><b>Sales Segment:</b> %{y:,.2f}<extra></extra>',
        customdata=np.stack((df_melted_products['Month_Label'],), axis=-1)
    )
    fig_prod.update_layout(
        yaxis_title="Aggregated Product Sales", xaxis_title="Products",
        height=450, margin=dict(t=50, b=50, l=50, r=50)
    )
    fig_prod.show()


interactive(children=(SelectMultiple(description='Quarters:', options=('2026Q2',), rows=4, value=()), SelectMu…

In [138]:
# QUARTERLY WITH MONTH FILTER

import io
import base64
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display, HTML
import plotly.express as px

# 1. CLEAN AND NORMALISE DATAFRAME WITHOUT DROPPING DATA
df_long_clean = df_long.copy()

# Convert numeric quarter values like 1,2,3,4 into normalized strings
if pd.api.types.is_numeric_dtype(df_long_clean['Quarter']):
    df_long_clean['Quarter'] = pd.to_numeric(df_long_clean['Quarter'], errors='coerce')
    df_long_clean['Quarter'] = df_long_clean['Quarter'].fillna(0).astype(int)

# Normalize Month numbers to standard string names early
month_map = {
    1: 'January', 2: 'February', 3: 'March', 4: 'April', 
    5: 'May', 6: 'June', 7: 'July', 8: 'August', 
    9: 'September', 10: 'October', 11: 'November', 12: 'December'
}
if df_long_clean['Month'].dtype in ['int64', 'float64', 'int32']:
    df_long_clean['Month'] = df_long_clean['Month'].map(month_map)

quarter_order_list = ['Q1', 'Q2', 'Q3', 'Q4']

def normalize_quarter_label(value):
    if pd.isna(value):
        return None
    if isinstance(value, str):
        text = value.strip().upper()
        if not text:
            return None
        if text.startswith('Q') and text[1:].isdigit():
            q = int(text[1:])
            return f'Q{q}' if q in (1, 2, 3, 4) else None
        if 'Q' in text:
            digits = ''.join(ch for ch in text if ch.isdigit())
            if digits:
                q = int(digits[-1])
                return f'Q{q}' if q in (1, 2, 3, 4) else None
        if text.isdigit():
            q = int(text)
            return f'Q{q}' if q in (1, 2, 3, 4) else None
        if text.startswith('20') and 'Q' in text:
            q_part = text.split('Q')[-1].strip()
            if q_part.isdigit():
                q = int(q_part)
                return f'Q{q}' if q in (1, 2, 3, 4) else None
        return None
    if hasattr(value, 'quarter'):
        q = int(value.quarter)
        return f'Q{q}' if q in (1, 2, 3, 4) else None
    if isinstance(value, (int, float)):
        q = int(value)
        return f'Q{q}' if q in (1, 2, 3, 4) else None
    return None

# Normalize all quarter values and discard invalid ones
df_long_clean['Quarter_Label'] = df_long_clean['Quarter'].map(normalize_quarter_label)
df_long_clean = df_long_clean[df_long_clean['Quarter_Label'].notna()].copy()
valid_q_labels = [q for q in quarter_order_list if q in df_long_clean['Quarter_Label'].unique()]

# 2. CORE QUARTERLY TIME-FILTER MATRIX CALCULATION ENGINE
def total_sales_by_quarterly_basis(df, selected_quarters, selected_months, show_all_quarters=False, show_all_months=False):
    df_filtered = df.copy()

    # Apply Quarter Filter
    if not show_all_quarters and selected_quarters:
        df_filtered = df_filtered[df_filtered['Quarter_Label'].isin(selected_quarters)]
        
    # Apply Month Filter
    if not show_all_months and selected_months:
        df_filtered = df_filtered[df_filtered['Month'].isin(selected_months)]

    if df_filtered.empty:
        return pd.DataFrame()

    pivoted_quarters = df_filtered.pivot_table(
        index='Quarter_Label',
        columns='Product',
        values='Sales_Value',
        aggfunc='sum',
        fill_value=0
    ).reset_index()

    product_cols = [c for c in pivoted_quarters.columns if c != 'Quarter_Label']
    pivoted_quarters['Total_Sales_Value'] = pivoted_quarters[product_cols].sum(axis=1)

    pivoted_quarters = pivoted_quarters[pivoted_quarters['Quarter_Label'].isin(quarter_order_list)].copy()
    pivoted_quarters['Quarter_Label'] = pd.Categorical(
        pivoted_quarters['Quarter_Label'],
        categories=quarter_order_list,
        ordered=True
    )
    pivoted_quarters = pivoted_quarters.sort_values(by='Quarter_Label').reset_index(drop=True)
    pivoted_quarters['Quarter_Label'] = pivoted_quarters['Quarter_Label'].astype(str)

    return pivoted_quarters

# 3. EXTRACT CLEAN OPTION LISTS FOR WIDGETS
total_quarters = [q for q in quarter_order_list if q in valid_q_labels]

month_order_list = ['January', 'February', 'March', 'April', 'May', 'June', 
                    'July', 'August', 'September', 'October', 'November', 'December']
total_months = [m for m in month_order_list if m in df_long_clean['Month'].dropna().unique()]

# 4. DASHBOARD WIDGETS
quarter_select = widgets.SelectMultiple(options=total_quarters, value=(), description='Quarters:', rows=4)
all_qtr_check = widgets.Checkbox(value=False, description='All Quarters')

month_select = widgets.SelectMultiple(options=total_months, value=(), description='Months:', rows=5)
all_mth_check = widgets.Checkbox(value=False, description='All Months')

refresh_btn = widgets.ToggleButton(value=False, description='🔄 Refresh Data', button_style='info')
download_output = widgets.Output()

# Arrange Filter Columns Grid Layout Layout View
ui_layout = widgets.VBox([
    widgets.HBox([quarter_select, month_select]),
    widgets.HBox([all_qtr_check, all_mth_check, refresh_btn])
])

# 5. LIVE DASHBOARD WRAPPER
@interact(quarters=quarter_select, months=month_select, all_qtrs=all_qtr_check, all_mths=all_mth_check, refresh=refresh_btn)
def show_and_export_quarterly_dashboard(quarters, months, all_qtrs, all_mths, refresh):
    download_output.clear_output()

    if refresh_btn.value:
        refresh_btn.value = False

    if not all_qtrs and not quarters:
        print("Step 1: Choose Quarters (or click 'All Quarters')")
        return
    if not all_mths and not months:
        print("Step 2: Choose Months (or click 'All Months')")
        return

    result = total_sales_by_quarterly_basis(
        df_long_clean, quarters, months, show_all_quarters=all_qtrs, show_all_months=all_mths
    )

    if result.empty:
        print("No matching transaction data fits this combined filter query.")
        return

    output_buffer = io.BytesIO()
    with pd.ExcelWriter(output_buffer, engine='xlsxwriter') as writer:
        result.to_excel(writer, index=False, sheet_name='Quarterly Product Breakdown')
        workbook = writer.book
        worksheet = writer.sheets['Quarterly Product Breakdown']
        num_format = workbook.add_format({'num_format': '#,##0.00', 'valign': 'top'})
        worksheet.set_column('A:A', 15)
        for col_idx in range(1, len(result.columns)):
            worksheet.set_column(col_idx, col_idx, 18, num_format)

    excel_data = output_buffer.getvalue()
    b64_encoded = base64.b64encode(excel_data).decode()

    button_html = f'''
    <a download="quarterly_product_sales_matrix.xlsx" href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64_encoded}" target="_blank">
        <button style="background-color:#1F6E43; color:white; padding:8px 16px; border:none; border-radius:4px; font-weight:bold; cursor:pointer; margin-bottom:12px;">
            Download Filtered Excel Matrix
        </button>
    </a>
    '''

    with download_output:
        display(HTML(button_html))
    display(download_output)

    numeric_cols = [c for c in result.columns if c != 'Quarter_Label']
    styled_df = result.style.format({col: '{:,.2f}' for col in numeric_cols}).set_properties(**{
        'text-align': 'right', 'vertical-align': 'top'
    }).set_properties(subset=['Quarter_Label'], **{'text-align': 'left'})

    display(styled_df)
    print("\n")

    fig_trend = px.line(
        result,
        x='Quarter_Label',
        y='Total_Sales_Value',
        title='Quarterly Total Sales Value Performance Curve',
        markers=True,
        labels={'Total_Sales_Value': 'Total Combined Sales', 'Quarter_Label': 'Quarter'}
    )
    fig_trend.update_traces(
        text=result['Total_Sales_Value'],
        texttemplate='%{text:,.2f}',
        textposition='top center',
        hovertemplate='<b>Timeframe:</b> %{x}<br><b>Combined Revenue:</b> %{y:,.2f}<extra></extra>',
        line=dict(width=3, color='#1F6E43')
    )
    fig_trend.update_layout(
        yaxis_title="Total Combined Sales",
        xaxis_title="Quarter",
        height=400,
        margin=dict(t=50, b=50, l=50, r=50)
    )
    fig_trend.show()

    product_only_cols = [c for c in result.columns if c not in ['Quarter_Label', 'Total_Sales_Value']]
    df_melted_products = result.melt(
        id_vars=['Quarter_Label'],
        value_vars=product_only_cols,
        var_name='Product',
        value_name='Sales Value'
    )

    prod_totals = df_melted_products.groupby('Product')['Sales Value'].sum().reset_index()
    sorted_product_order = prod_totals.sort_values(by='Sales Value', ascending=False)['Product'].tolist()

    fig_prod = px.bar(
        df_melted_products,
        x='Product',
        y='Sales Value',
        color='Quarter_Label',
        color_discrete_sequence=px.colors.qualitative.Dark24,
        category_orders={'Product': sorted_product_order},
        title='Total Product Contribution Analysis (Grouped By Quarter)',
        text='Sales Value',
        labels={'Sales Value': 'Product Sales', 'Product': 'Product Classification', 'Quarter_Label': 'Quarter'}
    )
    fig_prod.update_traces(
        texttemplate='%{text:,.2f}',
        textposition='outside',
        hovertemplate='<b>Product Name:</b> %{x}<br><b>Quarter Context:</b> %{customdata[0]}<br><b>Sales Segment:</b> %{y:,.2f}<extra></extra>',
        customdata=np.stack((df_melted_products['Quarter_Label'],), axis=-1)
    )
    fig_prod.update_layout(
        yaxis_title="Aggregated Product Sales",
        xaxis_title="Products",
        barmode='group',
        height=450,
        margin=dict(t=50, b=50, l=50, r=50)
    )
    fig_prod.show()


interactive(children=(SelectMultiple(description='Quarters:', options=('Q2',), rows=4, value=()), SelectMultip…

## PRODUCT ANALYSIS

In [139]:
# Best-selling product and Worst-selling product
import io
import base64
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display, HTML
import plotly.express as px

def get_best_and_worst_selling_products(df, selected_customers, selected_quarters, selected_months,
                                         show_all_customers=False, show_all_quarters=False, show_all_months=False):
    """
    Filters data dynamically, calculates total sales per product, 
    and identifies the best and worst performing items alongside the complete rankings.
    """
    df_filtered = df.copy()
    
    # 1. Apply Dynamic Filters
    if not show_all_customers and selected_customers:
        df_filtered = df_filtered[df_filtered['Customer'].isin(selected_customers)]
    if not show_all_quarters and selected_quarters:
        df_filtered = df_filtered[df_filtered['Quarter_Label'].isin(selected_quarters)]
    if not show_all_months and selected_months:
        df_filtered = df_filtered[df_filtered['Month'].isin(selected_months)]
        
    if df_filtered.empty:
        return pd.DataFrame(), None, None
    
    # 2. Aggregate Sales Performance Metrics
    product_sales = df_filtered.groupby('Product')['Sales_Value'].sum().reset_index()
    product_sales = product_sales.sort_values(by='Sales_Value', ascending=False).reset_index(drop=True)
    
    # Extract structural boundaries safely
    best_selling = product_sales.iloc[0]
    worst_selling = product_sales.iloc[-1]
    
    # 3. Label performance statuses for clean chart color mapping
    def assign_performance_tier(row):
        if row['Product'] == best_selling['Product']:
            return '⭐ Best Seller'
        elif row['Product'] == worst_selling['Product']:
            return '⚠️ Worst Seller'
        return 'Standard Catalog Item'
        
    product_sales['Performance_Status'] = product_sales.apply(assign_performance_tier, axis=1)
    
    return product_sales, best_selling, worst_selling

# Extract option profiles from clean baseline dataset
customer_list = sorted(df_long_clean['Customer'].dropna().unique().tolist())
quarter_list = ['Q1', 'Q2', 'Q3', 'Q4']
month_order_list = ['January', 'February', 'March', 'April', 'May', 'June', 
                    'July', 'August', 'September', 'October', 'November', 'December']
month_list = [m for m in month_order_list if m in df_long_clean['Month'].dropna().unique()]

# Build the layout selector widgets
customer_select = widgets.SelectMultiple(options=customer_list, value=(), description='Customers:', rows=5)
quarter_select = widgets.SelectMultiple(options=quarter_list, value=(), description='Quarters:', rows=4)
month_select = widgets.SelectMultiple(options=month_list, value=(), description='Months:', rows=5)

all_cust_check = widgets.Checkbox(value=False, description='All Customers')
all_qtr_check = widgets.Checkbox(value=False, description='All Quarters')
all_mth_check = widgets.Checkbox(value=False, description='All Months')

refresh_btn = widgets.ToggleButton(value=False, description='🔄 Refresh Data', button_style='info')
download_output = widgets.Output()

# Arrange Filter Columns Grid Layout
ui_layout = widgets.VBox([
    widgets.HBox([customer_select, quarter_select, month_select]),
    widgets.HBox([all_cust_check, all_qtr_check, all_mth_check, refresh_btn])
])

@interact(customers=customer_select, quarters=quarter_select, months=month_select,
          all_custs=all_cust_check, all_qtrs=all_qtr_check, all_mths=all_mth_check, refresh=refresh_btn)
def show_and_export_product_performance(customers, quarters, months, all_custs, all_qtrs, all_mths, refresh):
    download_output.clear_output()
    
    if refresh_btn.value:
        refresh_btn.value = False
        
    # Hidden state launch validations
    if not all_custs and not customers:
        print("Step 1: Choose Customers (or click 'All Customers')")
        return
    if not all_qtrs and not quarters:
        print("Step 2: Choose Quarters (or click 'All Quarters')")
        return
    if not all_mths and not months:
        print("Step 3: Choose Months (or click 'All Months')")
        return
        
    # 1. Execute performance calculation pipeline
    rankings, best, worst = get_best_and_worst_selling_products(
        df_long_clean, customers, quarters, months,
        show_all_customers=all_custs, show_all_quarters=all_qtrs, show_all_months=all_mths
    )
    
    if rankings.empty:
        print("No matching transaction data fits this combined filter query.")
        return

    # 2. Display High-Impact Summary Cards Text
    summary_html = f"""
    <div style="display: flex; gap: 20px; margin-bottom: 15px;">
        <div style="background-color: #E6F4EA; border-left: 5px solid #137333; padding: 12px; border-radius: 4px; flex: 1;">
            <span style="color: #137333; font-weight: bold; font-size: 11px; text-transform: uppercase;">🏆 Best Selling Product</span>
            <h3 style="margin: 5px 0 0 0; color: #202124;">{best['Product']}</h3>
            <p style="margin: 2px 0 0 0; color: #137333; font-weight: bold;">Value: KSh {best['Sales_Value']:,.2f}</p>
        </div>
        <div style="background-color: #FCE8E6; border-left: 5px solid #C5221F; padding: 12px; border-radius: 4px; flex: 1;">
            <span style="color: #C5221F; font-weight: bold; font-size: 11px; text-transform: uppercase;">⚠️ Worst Selling Product</span>
            <h3 style="margin: 5px 0 0 0; color: #202124;">{worst['Product']}</h3>
            <p style="margin: 2px 0 0 0; color: #C5221F; font-weight: bold;">Value: KSh {worst['Sales_Value']:,.2f}</p>
        </div>
    </div>
    """
    display(HTML(summary_html))

    # 3. In-Memory Excel Exporter Architecture
    output_buffer = io.BytesIO()
    with pd.ExcelWriter(output_buffer, engine='xlsxwriter') as writer:
        rankings[['Product', 'Sales_Value']].to_excel(writer, index=False, sheet_name='Product Performance Rankings')
        workbook  = writer.book
        worksheet = writer.sheets['Product Performance Rankings']
        num_format = workbook.add_format({'num_format': '#,##0.00', 'valign': 'top'})
        worksheet.set_column('A:A', 25)  # Product
        worksheet.set_column('B:B', 25, num_format)  # Sales Value
        
    excel_data = output_buffer.getvalue()
    b64_encoded = base64.b64encode(excel_data).decode()
    
    button_html = f'''
    <a download="product_sales_performance_rankings.xlsx" href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64_encoded}" target="_blank">
        <button style="background-color:#1F6E43; color:white; padding:8px 16px; border:none; border-radius:4px; font-weight:bold; cursor:pointer; margin-bottom:15px;">
            Download Performance Report (.xlsx)
        </button>
    </a>
    '''
    with download_output:
        display(HTML(button_html))
    display(download_output)
    
    # 4. Display stylized ranking datagrid framework
    styled_df = rankings[['Product', 'Sales_Value']].style.format({'Sales_Value': '{:,.2f}'}).set_properties(**{
        'text-align': 'right', 'vertical-align': 'top'
    }).set_properties(subset=['Product'], **{'text-align': 'left'})
    display(styled_df)
    print("\n")

    # 5. --- GRAPH FEATURE: INTERACTIVE VOLUME CONTRIBUTION ANALYSIS ---
    # Assign signature contrasting colors explicitly using a map dictionary
    color_map = {
        '⭐ Best Seller': '#137333',     # Crisp operations green
        '⚠️ Worst Seller': '#C5221F',    # Alert dashboard red
        'Standard Catalog Item': '#1A73E8' # Corporate slate blue
    }
    
    fig = px.bar(
        rankings,
        x='Product',
        y='Sales_Value',
        color='Performance_Status',
        color_discrete_map=color_map,
        title='Complete Product Catalog Revenue Contributions & Performance Standing',
        text='Sales_Value',
        labels={'Sales_Value': 'Aggregated Revenue (KSh)', 'Product': 'Product Classification', 'Performance_Status': 'Status'}
    )
    
    fig.update_traces(
        texttemplate='%{text:,.2f}',
        textposition='outside',
        hovertemplate='<b>Product Code:</b> %{x}<br><b>Combined Revenue:</b> KSh %{y:,.2f}<br><b>Standing:</b> %{text}<extra></extra>'
    )
    
    fig.update_layout(
        yaxis_title="Total Corporate Revenue (KSh)", 
        xaxis_title="Product Portfolio Classification",
        xaxis={'categoryorder':'total descending'}, # Keeps the descending chart layout locked
        height=500, 
        margin=dict(t=60, b=40, l=50, r=50),
        legend_title_text='Catalog Tiers'
    )
    fig.show()


interactive(children=(SelectMultiple(description='Customers:', options=('ABDALLAH_MUSA_OKUNGO_(2407)_-_KONDELE…

In [140]:
# Product volume
import io
import base64
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display, HTML
import plotly.express as px

def get_product_volume_analysis(df, selected_customers, selected_quarters, selected_months, selected_days_of_week,
                                  show_all_customers=False, show_all_quarters=False, show_all_months=False, show_all_days_of_week=False):
    """
    Filters data dynamically and calculates total sales volume per product.
    Tracks Customers, Quarters, Months, and Days of the Week.
    """
    df_filtered = df.copy()
    
    # 1. Apply Dynamic Time and Identity Filters
    if not show_all_customers and selected_customers:
        df_filtered = df_filtered[df_filtered['Customer'].isin(selected_customers)]
    if not show_all_quarters and selected_quarters:
        df_filtered = df_filtered[df_filtered['Quarter_Label'].isin(selected_quarters)]
    if not show_all_months and selected_months:
        df_filtered = df_filtered[df_filtered['Month'].isin(selected_months)]
    if not show_all_days_of_week and selected_days_of_week:
        df_filtered = df_filtered[df_filtered['Day_of_Week'].isin(selected_days_of_week)]
        
    if df_filtered.empty:
        return pd.DataFrame()
    
    # 2. Aggregate sales volume per product (Sorting largest to smallest volume)
    product_volume = df_filtered.groupby('Product')['Sales_Value'].sum().reset_index()
    product_volume.columns = ['Product', 'Total_Volume_Sold']
    product_volume = product_volume.sort_values(by='Total_Volume_Sold', ascending=False).reset_index(drop=True)
    
    return product_volume

# Extract clean option arrays from the baseline dataset framework
customer_list = sorted(df_long_clean['Customer'].dropna().unique().tolist())
quarter_list = ['Q1', 'Q2', 'Q3', 'Q4']
month_order_list = ['January', 'February', 'March', 'April', 'May', 'June', 
                    'July', 'August', 'September', 'October', 'November', 'December']
month_list = [m for m in month_order_list if m in df_long_clean['Month'].dropna().unique()]

# Define strict calendar day-of-week sorting options
day_order_list = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_list = [d for d in day_order_list if d in df_long_clean['Day_of_Week'].dropna().unique()]

# 3. CONSTRUCT INTERACTIVE INTERFACE WIDGETS
customer_select = widgets.SelectMultiple(options=customer_list, value=(), description='Customers:', rows=5)
quarter_select = widgets.SelectMultiple(options=quarter_list, value=(), description='Quarters:', rows=4)
month_select = widgets.SelectMultiple(options=month_list, value=(), description='Months:', rows=5)
day_of_week_select = widgets.SelectMultiple(options=weekday_list, value=(), description='Weekdays:', rows=5)

all_cust_check = widgets.Checkbox(value=False, description='All Customers')
all_qtr_check = widgets.Checkbox(value=False, description='All Quarters')
all_mth_check = widgets.Checkbox(value=False, description='All Months')
all_dow_check = widgets.Checkbox(value=False, description='All Weekdays')

refresh_btn = widgets.ToggleButton(value=False, description='🔄 Refresh Data', button_style='info')
download_output = widgets.Output()

# Organize dashboard columns spatially
ui_layout = widgets.VBox([
    widgets.HBox([customer_select, quarter_select]),
    widgets.HBox([all_cust_check, all_qtr_check]),
    widgets.HTML("<hr style='margin:10px 0;'>"),
    widgets.HBox([month_select, day_of_week_select]),
    widgets.HBox([all_mth_check, all_dow_check, refresh_btn])
])

# 4. LIVE DASHBOARD WRAPPER CONTROLLER
@interact(customers=customer_select, quarters=quarter_select, months=month_select, weekdays=day_of_week_select,
          all_custs=all_cust_check, all_qtrs=all_qtr_check, all_mths=all_mth_check, all_weekdays=all_dow_check, refresh=refresh_btn)
def show_and_export_volume_dashboard(customers, quarters, months, weekdays, all_custs, all_qtrs, all_mths, all_weekdays, refresh):
    download_output.clear_output()

    if refresh_btn.value:
        refresh_btn.value = False

    # Selection State Gate Validations
    if not all_custs and not customers:
        print("Step 1: Choose Customers (or click 'All Customers')")
        return
    if not all_qtrs and not quarters:
        print("Step 2: Choose Quarters (or click 'All Quarters')")
        return
    if not all_mths and not months:
        print("Step 3: Choose Months (or click 'All Months')")
        return
    if not all_weekdays and not weekdays:
        print("Step 4: Choose Weekdays (or click 'All Weekdays')")
        return

    # Execute computation pipeline
    result = get_product_volume_analysis(
        df_long_clean, customers, quarters, months, weekdays,
        show_all_customers=all_custs, show_all_quarters=all_qtrs, 
        show_all_months=all_mths, show_all_days_of_week=all_weekdays
    )

    if result.empty:
        print("No matching transaction data fits this combined filter query.")
        return

    # In-Memory Excel Exporter Architecture
    output_buffer = io.BytesIO()
    with pd.ExcelWriter(output_buffer, engine='xlsxwriter') as writer:
        result.to_excel(writer, index=False, sheet_name='Product Volumetric Data')
        workbook = writer.book
        worksheet = writer.sheets['Product Volumetric Data']
        num_format = workbook.add_format({'num_format': '#,##0', 'valign': 'top'})
        
        worksheet.set_column('A:A', 25)  # Product Column
        worksheet.set_column('B:B', 20, num_format)  # Volume Column

    excel_data = output_buffer.getvalue()
    b64_encoded = base64.b64encode(excel_data).decode()

    button_html = f'''
    <a download="product_sales_volume_report.xlsx" href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64_encoded}" target="_blank">
        <button style="background-color:#1F6E43; color:white; padding:8px 16px; border:none; border-radius:4px; font-weight:bold; cursor:pointer; margin-bottom:12px;">
            Download Volume Matrix Report (.xlsx)
        </button>
    </a>
    '''
    with download_output:
        display(HTML(button_html))
    display(download_output)

    # Render Clean Data Grid View Frame
    styled_df = result.style.format({'Total_Volume_Sold': '{:,.0f}'}).set_properties(**{
        'text-align': 'right', 'vertical-align': 'top'
    }).set_properties(subset=['Product'], **{'text-align': 'left'})
    
    display(styled_df)
    print("\n")

    # 5. --- GRAPH FEATURE: INTERACTIVE VOLUME DISTRIBUTION MATRIX ---
    fig = px.bar(
        result,
        x='Product',
        y='Total_Volume_Sold',
        title='Aggregated Volumetric Product Performance Rankings',
        text='Total_Volume_Sold',
        color='Total_Volume_Sold',
        color_continuous_scale=px.colors.sequential.Viridis,
        labels={'Total_Volume_Sold': 'Units/Volume Sold', 'Product': 'Product Variant Name'}
    )
    
    fig.update_traces(
        texttemplate='%{text:,.0f}',
        textposition='outside',
        hovertemplate='<b>Product Code:</b> %{x}<br><b>Units Dispatched:</b> %{y:,.0f}<extra></extra>'
    )
    
    fig.update_layout(
        yaxis_title="Total Dispatched Volume Count", 
        xaxis_title="Product Variations Catalog",
        height=450, 
        margin=dict(t=50, b=40, l=50, r=50),
        coloraxis_showscale=False  # Hides extra palette sidebar clutter
    )
    fig.show()


interactive(children=(SelectMultiple(description='Customers:', options=('ABDALLAH_MUSA_OKUNGO_(2407)_-_KONDELE…

In [141]:
# Product percentage contribution
import io
import base64
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display, HTML
import plotly.express as px

def get_product_percentage_contribution(df, selected_customers, selected_quarters, selected_months, selected_days_of_week,
                                        show_all_customers=False, show_all_quarters=False, show_all_months=False, show_all_days_of_week=False):
    """
    Filters data dynamically and calculates the percentage contribution of each product
    to the total sales value within the selected filters.
    """
    df_filtered = df.copy()
    
    # 1. Apply Dynamic Structural Filters
    if not show_all_customers and selected_customers:
        df_filtered = df_filtered[df_filtered['Customer'].isin(selected_customers)]
    if not show_all_quarters and selected_quarters:
        df_filtered = df_filtered[df_filtered['Quarter_Label'].isin(selected_quarters)]
    if not show_all_months and selected_months:
        df_filtered = df_filtered[df_filtered['Month'].isin(selected_months)]
    if not show_all_days_of_week and selected_days_of_week:
        df_filtered = df_filtered[df_filtered['Day_of_Week'].isin(selected_days_of_week)]
        
    if df_filtered.empty:
        return pd.DataFrame()
    
    # 2. Calculate dynamic grand total revenue baseline
    total_sales_value = df_filtered['Sales_Value'].sum()
    if total_sales_value == 0:
        return pd.DataFrame()
    
    # 3. Calculate absolute contribution metrics per catalog line
    product_contribution = df_filtered.groupby('Product')['Sales_Value'].sum().reset_index()
    product_contribution.columns = ['Product', 'Total_Sales_Value']
    product_contribution['Percentage_Contribution'] = (product_contribution['Total_Sales_Value'] / total_sales_value) * 100
    
    return product_contribution.sort_values(by='Percentage_Contribution', ascending=False).reset_index(drop=True)

# Extract clean option data series
customer_list = sorted(df_long_clean['Customer'].dropna().unique().tolist())
quarter_list = ['Q1', 'Q2', 'Q3', 'Q4']
month_order_list = ['January', 'February', 'March', 'April', 'May', 'June', 
                    'July', 'August', 'September', 'October', 'November', 'December']
month_list = [m for m in month_order_list if m in df_long_clean['Month'].dropna().unique()]

day_order_list = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_list = [d for d in day_order_list if d in df_long_clean['Day_of_Week'].dropna().unique()]

# Build the layout selector components
customer_select = widgets.SelectMultiple(options=customer_list, value=(), description='Customers:', rows=5)
quarter_select = widgets.SelectMultiple(options=quarter_list, value=(), description='Quarters:', rows=4)
month_select = widgets.SelectMultiple(options=month_list, value=(), description='Months:', rows=5)
day_of_week_select = widgets.SelectMultiple(options=weekday_list, value=(), description='Weekdays:', rows=5)

all_cust_check = widgets.Checkbox(value=False, description='All Customers')
all_qtr_check = widgets.Checkbox(value=False, description='All Quarters')
all_mth_check = widgets.Checkbox(value=False, description='All Months')
all_dow_check = widgets.Checkbox(value=False, description='All Weekdays')

refresh_btn = widgets.ToggleButton(value=False, description='🔄 Refresh Data', button_style='info')
download_output = widgets.Output()

# Arrange UI components cleanly via vertical and horizontal box frameworks
ui_layout = widgets.VBox([
    widgets.HBox([customer_select, quarter_select]),
    widgets.HBox([all_cust_check, all_qtr_check]),
    widgets.HTML("<hr style='margin:10px 0;'>"),
    widgets.HBox([month_select, day_of_week_select]),
    widgets.HBox([all_mth_check, all_dow_check, refresh_btn])
])

@interact(customers=customer_select, quarters=quarter_select, months=month_select, weekdays=day_of_week_select,
          all_custs=all_cust_check, all_qtrs=all_qtr_check, all_mths=all_mth_check, all_weekdays=all_dow_check, refresh=refresh_btn)
def show_and_export_contribution_dashboard(customers, quarters, months, weekdays, all_custs, all_qtrs, all_mths, all_weekdays, refresh):
    download_output.clear_output()

    if refresh_btn.value:
        refresh_btn.value = False

    # Selection State Gate Validations
    if not all_custs and not customers:
        print("Step 1: Choose Customers (or click 'All Customers')")
        return
    if not all_qtrs and not quarters:
        print("Step 2: Choose Quarters (or click 'All Quarters')")
        return
    if not all_mths and not months:
        print("Step 3: Choose Months (or click 'All Months')")
        return
    if not all_weekdays and not weekdays:
        print("Step 4: Choose Weekdays (or click 'All Weekdays')")
        return

    # Execute computation pipeline
    result = get_product_percentage_contribution(
        df_long_clean, customers, quarters, months, weekdays,
        show_all_customers=all_custs, show_all_quarters=all_qtrs, 
        show_all_months=all_mths, show_all_days_of_week=all_weekdays
    )

    if result.empty:
        print("No matching transaction data fits this combined filter query.")
        return

    # In-Memory Excel Exporter Architecture
    output_buffer = io.BytesIO()
    with pd.ExcelWriter(output_buffer, engine='xlsxwriter') as writer:
        result.to_excel(writer, index=False, sheet_name='Product Revenue Breakdown')
        workbook = writer.book
        worksheet = writer.sheets['Product Revenue Breakdown']
        num_format = workbook.add_format({'num_format': '#,##0.00', 'valign': 'top'})
        pct_format = workbook.add_format({'num_format': '0.00"%"', 'valign': 'top'})
        
        worksheet.set_column('A:A', 25)  # Product
        worksheet.set_column('B:B', 20, num_format)  # Total Sales Value
        worksheet.set_column('C:C', 22, pct_format)  # Percentage Share

    excel_data = output_buffer.getvalue()
    b64_encoded = base64.b64encode(excel_data).decode()

    button_html = f'''
    <a download="product_revenue_contribution_report.xlsx" href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64_encoded}" target="_blank">
        <button style="background-color:#1F6E43; color:white; padding:8px 16px; border:none; border-radius:4px; font-weight:bold; cursor:pointer; margin-bottom:12px;">
            Download Share Report (.xlsx)
        </button>
    </a>
    '''
    with download_output:
        display(HTML(button_html))
    display(download_output)

    # Render Clean Data Grid View Frame
    styled_df = result.style.format({
        'Total_Sales_Value': 'KSh {:,.2f}',
        'Percentage_Contribution': '{:.2f}%'
    }).set_properties(**{
        'text-align': 'right', 'vertical-align': 'top'
    }).set_properties(subset=['Product'], **{'text-align': 'left'})
    
    display(styled_df)
    print("\n")

    # 4. --- GRAPH FEATURE: SOLID INTERACTIVE PIE CHART ---
    fig = px.pie(
        result,
        names='Product',
        values='Total_Sales_Value',
        title='Proportional Product Portfolio Revenue Share Split',
        color_discrete_sequence=px.colors.qualitative.Prism  # High-contrast corporate palette
    )
    
    fig.update_traces(
        textinfo='percent+label',
        textposition='outside',  # Places text outside wedges to avoid overlapping strings
        hovertemplate='<b>Product Description:</b> %{label}<br><b>Revenue Value:</b> KSh %{value:,.2f}<br><b>Share Fraction:</b> %{percent}<extra></extra>'
    )
    
    fig.update_layout(
        height=500,
        margin=dict(t=60, b=40, l=40, r=40),
        legend_title_text='Catalog Variations'
    )
    fig.show()


interactive(children=(SelectMultiple(description='Customers:', options=('ABDALLAH_MUSA_OKUNGO_(2407)_-_KONDELE…

In [142]:
# Growing and declining products
import io
import base64
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display, HTML
import plotly.express as px

def get_growing_and_declining_products(df, selected_customers, selected_quarters, selected_months, selected_days_of_week,
                                       show_all_customers=False, show_all_quarters=False, show_all_months=False, show_all_days_of_week=False):
    """
    Filters data dynamically and calculates the growth or decline in sales value for each product
    compared to the previous chronological period within the active filters.
    """
    df_filtered = df.copy()
    
    # 1. Apply Dynamic Structural Filters
    if not show_all_customers and selected_customers:
        df_filtered = df_filtered[df_filtered['Customer'].isin(selected_customers)]
    if not show_all_quarters and selected_quarters:
        df_filtered = df_filtered[df_filtered['Quarter_Label'].isin(selected_quarters)]
    if not show_all_months and selected_months:
        df_filtered = df_filtered[df_filtered['Month'].isin(selected_months)]
    if not show_all_days_of_week and selected_days_of_week:
        df_filtered = df_filtered[df_filtered['Day_of_Week'].isin(selected_days_of_week)]
        
    if df_filtered.empty:
        return pd.DataFrame()
    
    # 2. Calculate sales value per product per quarter
    product_sales = df_filtered.groupby(['Product', 'Quarter_Label'])['Sales_Value'].sum().reset_index()
    
    if product_sales.empty or product_sales['Quarter_Label'].nunique() < 2:
        # If there's only one period available, we calculate absolute sales as baseline performance
        fallback = df_filtered.groupby('Product')['Sales_Value'].sum().reset_index()
        fallback['Growth_Decline'] = 0.0
        return fallback
    
    # 3. Pivot to have quarters as columns for sequential delta comparison
    product_sales_pivot = product_sales.pivot(index='Product', columns='Quarter_Label', values='Sales_Value').fillna(0)
    
    # 4. Calculate growth/decline compared to the immediately preceding column index
    quarters_present = list(product_sales_pivot.columns)
    product_sales_pivot['Growth_Decline'] = product_sales_pivot[quarters_present[-1]] - product_sales_pivot[quarters_present[-2]]
    
    return product_sales_pivot.reset_index()

# Extract option profiles from baseline dataset framework
customer_list = sorted(df_long_clean['Customer'].dropna().unique().tolist())
quarter_list = ['Q1', 'Q2', 'Q3', 'Q4']
month_order_list = ['January', 'February', 'March', 'April', 'May', 'June', 
                    'July', 'August', 'September', 'October', 'November', 'December']
month_list = [m for m in month_order_list if m in df_long_clean['Month'].dropna().unique()]
day_order_list = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_list = [d for d in day_order_list if d in df_long_clean['Day_of_Week'].dropna().unique()]

# Build UI selection components
customer_select = widgets.SelectMultiple(options=customer_list, value=(), description='Customers:', rows=5)
quarter_select = widgets.SelectMultiple(options=quarter_list, value=(), description='Quarters:', rows=4)
month_select = widgets.SelectMultiple(options=month_list, value=(), description='Months:', rows=5)
day_of_week_select = widgets.SelectMultiple(options=weekday_list, value=(), description='Weekdays:', rows=5)

all_cust_check = widgets.Checkbox(value=False, description='All Customers')
all_qtr_check = widgets.Checkbox(value=False, description='All Quarters')
all_mth_check = widgets.Checkbox(value=False, description='All Months')
all_dow_check = widgets.Checkbox(value=False, description='All Weekdays')

refresh_btn = widgets.ToggleButton(value=False, description='🔄 Refresh Data', button_style='info')
download_output = widgets.Output()

ui_layout = widgets.VBox([
    widgets.HBox([customer_select, quarter_select]),
    widgets.HBox([all_cust_check, all_qtr_check]),
    widgets.HTML("<hr style='margin:10px 0;'>"),
    widgets.HBox([month_select, day_of_week_select]),
    widgets.HBox([all_mth_check, all_dow_check, refresh_btn])
])

@interact(customers=customer_select, quarters=quarter_select, months=month_select, weekdays=day_of_week_select,
          all_custs=all_cust_check, all_qtrs=all_qtr_check, all_mths=all_mth_check, all_weekdays=all_dow_check, refresh=refresh_btn)
def show_and_export_growth_dashboard(customers, quarters, months, weekdays, all_custs, all_qtrs, all_mths, all_weekdays, refresh):
    download_output.clear_output()

    if refresh_btn.value:
        refresh_btn.value = False

    # Selection State Gate Validations
    if not all_custs and not customers:
        print("Step 1: Choose Customers (or click 'All Customers')")
        return
    if not all_qtrs and not quarters:
        print("Step 2: Choose Quarters (or click 'All Quarters')")
        return
    if not all_mths and not months:
        print("Step 3: Choose Months (or click 'All Months')")
        return
    if not all_weekdays and not weekdays:
        print("Step 4: Choose Weekdays (or click 'All Weekdays')")
        return

    # 1. Execute metrics computation pipeline
    result = get_growing_and_declining_products(
        df_long_clean, customers, quarters, months, weekdays,
        show_all_customers=all_custs, show_all_quarters=all_qtrs, 
        show_all_months=all_mths, show_all_days_of_week=all_weekdays
    )

    if result.empty:
        print("No matching transaction data fits this combined filter query.")
        return

    # Sort results by net performance delta descending
    result = result.sort_values(by='Growth_Decline', ascending=False).reset_index(drop=True)

    # 2. In-Memory Excel Exporter Architecture
    output_buffer = io.BytesIO()
    with pd.ExcelWriter(output_buffer, engine='xlsxwriter') as writer:
        result.to_excel(writer, index=False, sheet_name='Product Growth Summary')
        workbook = writer.book
        worksheet = writer.sheets['Product Growth Summary']
        num_format = workbook.add_format({'num_format': '#,##0.00', 'valign': 'top'})
        
        for col_idx in range(len(result.columns)):
            worksheet.set_column(col_idx, col_idx, 18, num_format)

    excel_data = output_buffer.getvalue()
    b64_encoded = base64.b64encode(excel_data).decode()

    button_html = f'''
    <a download="product_growth_decline_report.xlsx" href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64_encoded}" target="_blank">
        <button style="background-color:#1F6E43; color:white; padding:8px 16px; border:none; border-radius:4px; font-weight:bold; cursor:pointer; margin-bottom:12px;">
            Download Performance Deltas (.xlsx)
        </button>
    </a>
    '''
    with download_output:
        display(HTML(button_html))
    display(download_output)

    # Render Clean Data Grid View Frame
    numeric_cols = [c for c in result.columns if c != 'Product']
    styled_df = result.style.format({col: '{:,.2f}' for col in numeric_cols}).set_properties(**{
        'text-align': 'right', 'vertical-align': 'top'
    }).set_properties(subset=['Product'], **{'text-align': 'left'})
    display(styled_df)
    print("\n")

    # Add categorical color tags for visualization splitting
    result['Status'] = np.where(result['Growth_Decline'] >= 0, '📈 Growth Spike', '📉 Declining Contraction')

    # 3. --- VISUALIZATION 1: PERFORMANCE BAR CHART ---
    fig_bar = px.bar(
        result,
        x='Product',
        y='Growth_Decline',
        color='Status',
        color_discrete_map={'📈 Growth Spike': '#137333', '📉 Declining Contraction': '#C5221F'},
        title='Net Sales Performance Variations per Product Line (Current vs Previous Period)',
        text='Growth_Decline',
        labels={'Growth_Decline': 'Net Shift Value (KSh)', 'Product': 'Product Catalog Name'}
    )
    fig_bar.update_traces(
        texttemplate='%{text:,.2f}',
        textposition='outside',
        hovertemplate='<b>Product Name:</b> %{x}<br><b>Net Sales Variance:</b> KSh %{y:,.2f}<extra></extra>'
    )
    fig_bar.update_layout(
        yaxis_title="Net Performance Variance (KSh)", xaxis_title="Product Lines",
        height=450, margin=dict(t=50, b=40, l=50, r=50)
    )
    fig_bar.show()

    # 4. --- VISUALIZATION 2: DISTRIBUTION DENSITY HISTOGRAM ---
    fig_hist = px.histogram(
        result,
        x='Growth_Decline',
        nbins=10,
        title='Distribution Density of Performance Variance across Catalog Lines',
        color_discrete_sequence=['#1A73E8'],
        labels={'Growth_Decline': 'Net Variance Range (KSh)'}
    )
    fig_hist.update_layout(
        yaxis_title="Product Variations Count", xaxis_title="Net Variance Shift Intervals (KSh)",
        height=400, margin=dict(t=50, b=40, l=50, r=50)
    )
    fig_hist.show()


interactive(children=(SelectMultiple(description='Customers:', options=('ABDALLAH_MUSA_OKUNGO_(2407)_-_KONDELE…

## Product Pareto analysis

In [143]:
# Product Pareto analysis(Product Revenue Concentration)
import io
import base64
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display, HTML
import plotly.graph_objects as go

def get_product_pareto_analysis(df, selected_customers, selected_quarters, selected_months, selected_days_of_week,
                                show_all_customers=False, show_all_quarters=False, show_all_months=False, show_all_days_of_week=False):
    """
    Filters data dynamically and calculates the cumulative percentage contribution of products
    to total sales value for Pareto analysis.
    """
    df_filtered = df.copy()
    
    # 1. Apply Dynamic Structural Filters
    if not show_all_customers and selected_customers:
        df_filtered = df_filtered[df_filtered['Customer'].isin(selected_customers)]
    if not show_all_quarters and selected_quarters:
        df_filtered = df_filtered[df_filtered['Quarter_Label'].isin(selected_quarters)]
    if not show_all_months and selected_months:
        df_filtered = df_filtered[df_filtered['Month'].isin(selected_months)]
    if not show_all_days_of_week and selected_days_of_week:
        df_filtered = df_filtered[df_filtered['Day_of_Week'].isin(selected_days_of_week)]
        
    if df_filtered.empty:
        return pd.DataFrame()
    
    # 2. Calculate total sales value per product
    product_sales = df_filtered.groupby('Product')['Sales_Value'].sum().reset_index()
    product_sales.columns = ['Product', 'Total_Sales_Value']
    
    # 3. Sort products by sales value descending
    product_sales = product_sales.sort_values(by='Total_Sales_Value', ascending=False).reset_index(drop=True)
    
    # 4. Calculate cumulative sales value and cumulative percentage
    product_sales['Cumulative_Sales'] = product_sales['Total_Sales_Value'].cumsum()
    total_sales_value = product_sales['Total_Sales_Value'].sum()
    
    if total_sales_value == 0:
        product_sales['Cumulative_Percentage'] = 0.0
    else:
        product_sales['Cumulative_Percentage'] = (product_sales['Cumulative_Sales'] / total_sales_value) * 100
    
    return product_sales

# Extract option profiles from baseline dataset framework
customer_list = sorted(df_long_clean['Customer'].dropna().unique().tolist())
quarter_list = ['Q1', 'Q2', 'Q3', 'Q4']
month_order_list = ['January', 'February', 'March', 'April', 'May', 'June', 
                    'July', 'August', 'September', 'October', 'November', 'December']
month_list = [m for m in month_order_list if m in df_long_clean['Month'].dropna().unique()]
day_order_list = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_list = [d for d in day_order_list if d in df_long_clean['Day_of_Week'].dropna().unique()]

# Build UI selection components
customer_select = widgets.SelectMultiple(options=customer_list, value=(), description='Customers:', rows=5)
quarter_select = widgets.SelectMultiple(options=quarter_list, value=(), description='Quarters:', rows=4)
month_select = widgets.SelectMultiple(options=month_list, value=(), description='Months:', rows=5)
day_of_week_select = widgets.SelectMultiple(options=weekday_list, value=(), description='Weekdays:', rows=5)

all_cust_check = widgets.Checkbox(value=False, description='All Customers')
all_qtr_check = widgets.Checkbox(value=False, description='All Quarters')
all_mth_check = widgets.Checkbox(value=False, description='All Months')
all_dow_check = widgets.Checkbox(value=False, description='All Weekdays')

refresh_btn = widgets.ToggleButton(value=False, description='Refresh Data', button_style='info')
download_output = widgets.Output()

ui_layout = widgets.VBox([
    widgets.HBox([customer_select, quarter_select]),
    widgets.HBox([all_cust_check, all_qtr_check]),
    widgets.HTML("<hr style='margin:10px 0;'>"),
    widgets.HBox([month_select, day_of_week_select]),
    widgets.HBox([all_mth_check, all_dow_check, refresh_btn])
])

@interact(customers=customer_select, quarters=quarter_select, months=month_select, weekdays=day_of_week_select,
          all_custs=all_cust_check, all_qtrs=all_qtr_check, all_mths=all_mth_check, all_weekdays=all_dow_check, refresh=refresh_btn)
def show_and_export_pareto_dashboard(customers, quarters, months, weekdays, all_custs, all_qtrs, all_mths, all_weekdays, refresh):
    download_output.clear_output()

    if refresh_btn.value:
        refresh_btn.value = False

    # Selection State Gate Validations
    if not all_custs and not customers:
        print("Step 1: Choose Customers (or click 'All Customers')")
        return
    if not all_qtrs and not quarters:
        print("Step 2: Choose Quarters (or click 'All Quarters')")
        return
    if not all_mths and not months:
        print("Step 3: Choose Months (or click 'All Months')")
        return
    if not all_weekdays and not weekdays:
        print("Step 4: Choose Weekdays (or click 'All Weekdays')")
        return

    # 1. Execute metrics computation pipeline
    result = get_product_pareto_analysis(
        df_long_clean, customers, quarters, months, weekdays,
        show_all_customers=all_custs, show_all_quarters=all_qtrs, 
        show_all_months=all_mths, show_all_days_of_week=all_weekdays
    )

    if result.empty:
        print("No matching transaction data fits this combined filter query.")
        return

    # 2. In-Memory Excel Exporter Architecture
    output_buffer = io.BytesIO()
    with pd.ExcelWriter(output_buffer, engine='xlsxwriter') as writer:
        result.to_excel(writer, index=False, sheet_name='Pareto Revenue Analysis')
        workbook = writer.book
        worksheet = writer.sheets['Pareto Revenue Analysis']
        num_format = workbook.add_format({'num_format': '#,##0.00', 'valign': 'top'})
        pct_format = workbook.add_format({'num_format': '0.0"%"', 'valign': 'top'})
        
        worksheet.set_column('A:A', 25)  # Product
        worksheet.set_column('B:B', 20, num_format)  # Total Sales Value
        worksheet.set_column('C:C', 20, num_format)  # Cumulative Sales
        worksheet.set_column('D:D', 22, pct_format)  # Cumulative Percentage

    excel_data = output_buffer.getvalue()
    b64_encoded = base64.b64encode(excel_data).decode()

    button_html = f'''
    <a download="product_pareto_analysis.xlsx" href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64_encoded}" target="_blank">
        <button style="background-color:#1F6E43; color:white; padding:8px 16px; border:none; border-radius:4px; font-weight:bold; cursor:pointer; margin-bottom:12px;">
            Download Pareto Report (.xlsx)
        </button>
    </a>
    '''
    with download_output:
        display(HTML(button_html))
    display(download_output)

    # Render Clean Data Grid View Frame
    styled_df = result.style.format({
        'Total_Sales_Value': 'KSh {:,.2f}',
        'Cumulative_Sales': 'KSh {:,.2f}',
        'Cumulative_Percentage': '{:.1f}%'
    }).set_properties(**{
        'text-align': 'right', 'vertical-align': 'top'
    }).set_properties(subset=['Product'], **{'text-align': 'left'})
    display(styled_df)
    print("\n")

    # 3. --- PARETO DOUBLE Y-AXIS VISUALIZATION ENGINE ---
    fig = go.Figure()

    # Add bars for individual revenue size (Primary Left Y-Axis)
    fig.add_trace(
        go.Bar(
            x=result['Product'],
            y=result['Total_Sales_Value'],
            name='Product Revenue',
            marker_color='#1A73E8',
            hovertemplate='<b>Product:</b> %{x}<br><b>Revenue:</b> KSh %{y:,.2f}<extra></extra>'
        )
    )

    # Add line for cumulative percentage curve (Secondary Right Y-Axis)
    fig.add_trace(
        go.Scatter(
            x=result['Product'],
            y=result['Cumulative_Percentage'],
            name='Cumulative %',
            yaxis='y2',
            mode='lines+markers',
            marker=dict(color='#C5221F', size=6),
            line=dict(color='#C5221F', width=2),
            hovertemplate='<b>Product:</b> %{x}<br><b>Cumulative Share:</b> %{y:.1f}%<extra></extra>'
        )
    )

    # Add a horizontal dashed line at 80% to clearly show the threshold rule cut-off
    fig.add_shape(
        type="line",
        x0=result['Product'].iloc[0],
        y0=80,
        x1=result['Product'].iloc[-1],
        y1=80,
        yref='y2',
        line=dict(color="rgba(0,0,0,0.5)", width=2, dash="dash"),
    )

    # Update complex structural axes layout
    fig.update_layout(
        title='Product Revenue Concentration Analysis (Pareto Principle)',
        height=550,
        margin=dict(t=60, b=40, l=60, r=60),
        xaxis=dict(title='Product Catalog Variations'),
        yaxis=dict(
            title=dict(text='Individual Product Revenue (KSh)', font=dict(color='#1A73E8')),
            tickfont=dict(color='#1A73E8')
        ),
        yaxis2=dict(
            title=dict(text='Cumulative Contribution Percentage', font=dict(color='#C5221F')),
            tickfont=dict(color='#C5221F'),
            overlaying='y',
            side='right',
            range=[0, 105],
            tickvals=[0, 20, 40, 60, 80, 100],
            tickformat='.0f'
        ),
        legend=dict(x=0.02, y=0.98, bgcolor='rgba(255,255,255,0.6)'),
        showlegend=True
    )
    fig.show()


interactive(children=(SelectMultiple(description='Customers:', options=('ABDALLAH_MUSA_OKUNGO_(2407)_-_KONDELE…

## CUSTOMER ANALYSIS

In [144]:
# Top customers
import io
import base64
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display, HTML
import plotly.express as px

def get_top_customers_analysis(df, selected_quarters, selected_months, selected_days_of_week):
    """
    Filters data dynamically and calculates the top customers based on total sales value
    within the selected filters.
    """
    df_filtered = df.copy()
    
    # 1. Apply Dynamic Structural Filters
    if selected_quarters:
        df_filtered = df_filtered[df_filtered['Quarter_Label'].isin(selected_quarters)]
    if selected_months:
        df_filtered = df_filtered[df_filtered['Month'].isin(selected_months)]
    if selected_days_of_week:
        df_filtered = df_filtered[df_filtered['Day_of_Week'].isin(selected_days_of_week)]
        
    if df_filtered.empty:
        return pd.DataFrame()
    
    # 2. Calculate total sales value per customer
    customer_sales = df_filtered.groupby('Customer')['Sales_Value'].sum().reset_index()
    customer_sales.columns = ['Customer', 'Total_Sales_Value']
    
    # 3. Sort customers by sales value descending
    customer_sales = customer_sales.sort_values(by='Total_Sales_Value', ascending=False).reset_index(drop=True)
    
    return customer_sales

# Extract clean option arrays from the baseline dataset framework
quarter_list = ['Q1', 'Q2', 'Q3', 'Q4']
month_order_list = ['January', 'February', 'March', 'April', 'May', 'June', 
                    'July', 'August', 'September', 'October', 'November', 'December']
month_list = [m for m in month_order_list if m in df_long_clean['Month'].dropna().unique()]
day_order_list = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_list = [d for d in day_order_list if d in df_long_clean['Day_of_Week'].dropna().unique()]

# Build the layout selector widgets
quarter_select = widgets.SelectMultiple(options=quarter_list, value=(), description='Quarters:', rows=4)
month_select = widgets.SelectMultiple(options=month_list, value=(), description='Months:', rows=5)
day_of_week_select = widgets.SelectMultiple(options=weekday_list, value=(), description='Weekdays:', rows=5)

all_qtr_check = widgets.Checkbox(value=False, description='All Quarters')
all_mth_check = widgets.Checkbox(value=False, description='All Months')
all_dow_check = widgets.Checkbox(value=False, description='All Weekdays')

refresh_btn = widgets.ToggleButton(value=False, description='Refresh Data', button_style='info')
download_output = widgets.Output()

# Arrange Filter Columns Grid Layout
ui_layout = widgets.VBox([
    widgets.HBox([quarter_select, month_select, day_of_week_select]),
    widgets.HBox([all_qtr_check, all_mth_check, all_dow_check, refresh_btn])
])

@interact(quarters=quarter_select, months=month_select, weekdays=day_of_week_select,
          all_qtrs=all_qtr_check, all_mths=all_mth_check, all_weekdays=all_dow_check, refresh=refresh_btn)
def show_and_export_customer_dashboard(quarters, months, weekdays, all_qtrs, all_mths, all_weekdays, refresh):
    download_output.clear_output()

    if refresh_btn.value:
        refresh_btn.value = False

    # Selection State Gate Validations
    chosen_qtrs = quarter_list if all_qtrs else quarters
    chosen_mths = month_list if all_mths else months
    chosen_wdays = weekday_list if all_weekdays else weekdays

    if not chosen_qtrs:
        print("Step 1: Choose Quarters (or click 'All Quarters')")
        return
    if not chosen_mths:
        print("Step 2: Choose Months (or click 'All Months')")
        return
    if not chosen_wdays:
        print("Step 3: Choose Weekdays (or click 'All Weekdays')")
        return

    # 1. Execute metrics computation pipeline
    result = get_top_customers_analysis(df_long_clean, chosen_qtrs, chosen_mths, chosen_wdays)

    if result.empty:
        print("No matching transaction data fits this combined filter query.")
        return

    # Limit view to Top 15 rows for clean chart display density, while exporting everything
    display_result = result.head(15).copy()

    # 2. In-Memory Excel Exporter Architecture
    output_buffer = io.BytesIO()
    with pd.ExcelWriter(output_buffer, engine='xlsxwriter') as writer:
        result.to_excel(writer, index=False, sheet_name='Top Customers Revenue')
        workbook = writer.book
        worksheet = writer.sheets['Top Customers Revenue']
        num_format = workbook.add_format({'num_format': '#,##0.00', 'valign': 'top'})
        
        worksheet.set_column('A:A', 50)  # Customer Column
        worksheet.set_column('B:B', 25, num_format)  # Revenue Column

    excel_data = output_buffer.getvalue()
    b64_encoded = base64.b64encode(excel_data).decode()

    button_html = f'''
    <a download="top_customers_revenue_report.xlsx" href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64_encoded}" target="_blank">
        <button style="background-color:#1F6E43; color:white; padding:8px 16px; border:none; border-radius:4px; font-weight:bold; cursor:pointer; margin-bottom:12px;">
            Download Customer Report (.xlsx)
        </button>
    </a>
    '''
    with download_output:
        display(HTML(button_html))
    display(download_output)

    # Render Clean Data Grid View Frame
    styled_df = display_result.style.format({'Total_Sales_Value': 'KSh {:,.2f}'}).set_properties(**{
        'text-align': 'right', 'vertical-align': 'top'
    }).set_properties(subset=['Customer'], **{'text-align': 'left'})
    display(styled_df)
    print("\n")

    # 3. --- GRAPH FEATURE: HORIZONTAL BAR CHART GRAPH ---
    # To display top accounts at the top of a horizontal chart, we reverse the data sorting for the y-axis
    display_result = display_result.sort_values(by='Total_Sales_Value', ascending=True)

    fig = px.bar(
        display_result,
        x='Total_Sales_Value',
        y='Customer',
        title='Top Customers Revenue Contribution Rankings',
        text='Total_Sales_Value',
        color='Total_Sales_Value',
        color_continuous_scale=px.colors.sequential.Blugrn,
        orientation='h',  # Crucial modifier flag forcing the matrix horizontal
        labels={'Total_Sales_Value': 'Total Sales Value (KSh)', 'Customer': 'Customer Identity'}
    )
    
    fig.update_traces(
        texttemplate='KSh %{text:,.2f}',
        textposition='outside',  # Positions revenue labels cleanly past the ends of the bars
        hovertemplate='<b>Customer Name:</b> %{y}<br><b>Total Value:</b> KSh %{x:,.2f}<extra></extra>'
    )
    
    fig.update_layout(
        xaxis_title="Total Corporate Revenue (KSh)", 
        yaxis_title="Customer Account Profiles",
        height=550, 
        margin=dict(t=50, b=50, l=150, r=100),  # Widened left side margin to comfortably parse text titles
        coloraxis_showscale=False
    )
    fig.show()


interactive(children=(SelectMultiple(description='Quarters:', options=('Q1', 'Q2', 'Q3', 'Q4'), rows=4, value=…

In [145]:
# Customer Frequency
import io
import base64
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display, HTML
import plotly.express as px

def get_customer_frequency_analysis(df, selected_quarters, selected_months, selected_days_of_week):
    """
    Filters data dynamically and calculates the frequency of transactions per customer
    within the selected filters.
    """
    df_filtered = df.copy()
    
    # 1. Apply Dynamic Structural Filters
    if selected_quarters:
        df_filtered = df_filtered[df_filtered['Quarter_Label'].isin(selected_quarters)]
    if selected_months:
        df_filtered = df_filtered[df_filtered['Month'].isin(selected_months)]
    if selected_days_of_week:
        df_filtered = df_filtered[df_filtered['Day_of_Week'].isin(selected_days_of_week)]
        
    if df_filtered.empty:
        return pd.DataFrame()
    
    # 2. Calculate transaction frequency per customer (row count per client)
    customer_frequency = df_filtered.groupby('Customer').size().reset_index(name='Transaction_Frequency')
    
    # 3. Sort customers by transaction frequency descending
    customer_frequency = customer_frequency.sort_values(by='Transaction_Frequency', ascending=False).reset_index(drop=True)
    
    return customer_frequency

# Extract clean option arrays from the baseline dataset framework
quarter_list = ['Q1', 'Q2', 'Q3', 'Q4']
month_order_list = ['January', 'February', 'March', 'April', 'May', 'June', 
                    'July', 'August', 'September', 'October', 'November', 'December']
month_list = [m for m in month_order_list if m in df_long_clean['Month'].dropna().unique()]
day_order_list = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_list = [d for d in day_order_list if d in df_long_clean['Day_of_Week'].dropna().unique()]

# Build the layout selector widgets
quarter_select = widgets.SelectMultiple(options=quarter_list, value=(), description='Quarters:', rows=4)
month_select = widgets.SelectMultiple(options=month_list, value=(), description='Months:', rows=5)
day_of_week_select = widgets.SelectMultiple(options=weekday_list, value=(), description='Weekdays:', rows=5)

all_qtr_check = widgets.Checkbox(value=False, description='All Quarters')
all_mth_check = widgets.Checkbox(value=False, description='All Months')
all_dow_check = widgets.Checkbox(value=False, description='All Weekdays')

refresh_btn = widgets.ToggleButton(value=False, description='Refresh Data', button_style='info')
download_output = widgets.Output()

# Arrange Filter Columns Grid Layout
ui_layout = widgets.VBox([
    widgets.HBox([quarter_select, month_select, day_of_week_select]),
    widgets.HBox([all_qtr_check, all_mth_check, all_dow_check, refresh_btn])
])

@interact(quarters=quarter_select, months=month_select, weekdays=day_of_week_select,
          all_qtrs=all_qtr_check, all_mths=all_mth_check, all_weekdays=all_dow_check, refresh=refresh_btn)
def show_and_export_frequency_dashboard(quarters, months, weekdays, all_qtrs, all_mths, all_weekdays, refresh):
    download_output.clear_output()

    if refresh_btn.value:
        refresh_btn.value = False

    # Selection State Gate Validations
    chosen_qtrs = quarter_list if all_qtrs else quarters
    chosen_mths = month_list if all_mths else months
    chosen_wdays = weekday_list if all_weekdays else weekdays

    if not chosen_qtrs:
        print("Step 1: Choose Quarters (or click 'All Quarters')")
        return
    if not chosen_mths:
        print("Step 2: Choose Months (or click 'All Months')")
        return
    if not chosen_wdays:
        print("Step 3: Choose Weekdays (or click 'All Weekdays')")
        return

    # 1. Execute metrics computation pipeline
    result = get_customer_frequency_analysis(df_long_clean, chosen_qtrs, chosen_mths, chosen_wdays)

    if result.empty:
        print("No matching transaction data fits this combined filter query.")
        return

    # Limit view to Top 15 rows for clean display density, while exporting the entire block
    display_result = result.head(15).copy()

    # 2. In-Memory Excel Exporter Architecture
    output_buffer = io.BytesIO()
    with pd.ExcelWriter(output_buffer, engine='xlsxwriter') as writer:
        result.to_excel(writer, index=False, sheet_name='Customer Visit Frequency')
        workbook = writer.book
        worksheet = writer.sheets['Customer Frequency'] if 'Customer Frequency' in writer.sheets else writer.sheets['Customer Visit Frequency']
        num_format = workbook.add_format({'num_format': '#,##0', 'valign': 'top'})
        
        worksheet.set_column('A:A', 50)  # Customer Column
        worksheet.set_column('B:B', 25, num_format)  # Frequency Column

    excel_data = output_buffer.getvalue()
    b64_encoded = base64.b64encode(excel_data).decode()

    button_html = f'''
    <a download="customer_transaction_frequency_report.xlsx" href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64_encoded}" target="_blank">
        <button style="background-color:#1F6E43; color:white; padding:8px 16px; border:none; border-radius:4px; font-weight:bold; cursor:pointer; margin-bottom:12px;">
            Download Frequency Report (.xlsx)
        </button>
    </a>
    '''
    with download_output:
        display(HTML(button_html))
    display(download_output)

    # Render Clean Data Grid View Frame
    styled_df = display_result.style.format({'Transaction_Frequency': '{:,.0f}'}).set_properties(**{
        'text-align': 'right', 'vertical-align': 'top'
    }).set_properties(subset=['Customer'], **{'text-align': 'left'})
    display(styled_df)
    print("\n")

    # 3. --- GRAPH FEATURE: HORIZONTAL FREQUENCY DISTRIBUTION CHART ---
    # Re-sort low-to-high specifically for horizontal alignment so the top item stays at the top peak
    display_result = display_result.sort_values(by='Transaction_Frequency', ascending=True)

    fig = px.bar(
        display_result,
        x='Transaction_Frequency',
        y='Customer',
        title='Top Customers by Transaction Frequency',
        text='Transaction_Frequency',
        color='Transaction_Frequency',
        color_continuous_scale=px.colors.sequential.Teal,
        orientation='h',
        labels={'Transaction_Frequency': 'Number of Transactions', 'Customer': 'Customer Identity'}
    )
    
    fig.update_traces(
        texttemplate='%{text:,.0f}',
        textposition='outside',
        hovertemplate='<b>Customer Name:</b> %{y}<br><b>Transactions Count:</b> %{x:,.0f}<extra></extra>'
    )
    
    fig.update_layout(
        xaxis_title="Total Number of Transactions", 
        yaxis_title="Customer Account Profiles",
        height=550, 
        margin=dict(t=50, b=50, l=150, r=100),  # Generous padding to prevent string truncation
        coloraxis_showscale=False
    )
    fig.show()


interactive(children=(SelectMultiple(description='Quarters:', options=('Q1', 'Q2', 'Q3', 'Q4'), rows=4, value=…

In [146]:
# Quantity per customer
import io
import base64
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display, HTML
import plotly.express as px

def get_quantity_per_customer_analysis(df, selected_quarters, selected_months, selected_days_of_week):
    """
    Filters data dynamically and calculates the total quantity of products purchased per customer
    within the selected filters.
    """
    df_filtered = df.copy()
    
    # 1. Apply Dynamic Structural Filters
    if selected_quarters:
        df_filtered = df_filtered[df_filtered['Quarter_Label'].isin(selected_quarters)]
    if selected_months:
        df_filtered = df_filtered[df_filtered['Month'].isin(selected_months)]
    if selected_days_of_week:
        df_filtered = df_filtered[df_filtered['Day_of_Week'].isin(selected_days_of_week)]
        
    if df_filtered.empty:
        return pd.DataFrame()
    
    # 2. Calculate total quantity per customer (Assumes column name is 'Sales_Value' or 'Quantity' based on long form setup)
    # If your long-form unpivoted product volumes are under 'Sales_Value', change 'Quantity' to 'Sales_Value' below
    quantity_col = 'Quantity' if 'Quantity' in df_filtered.columns else 'Sales_Value'
    
    customer_quantity = df_filtered.groupby('Customer')[quantity_col].sum().reset_index()
    customer_quantity.columns = ['Customer', 'Total_Quantity']
    
    # 3. Sort customers by total quantity descending
    customer_quantity = customer_quantity.sort_values(by='Total_Quantity', ascending=False).reset_index(drop=True)
    
    return customer_quantity

# Extract clean option arrays from the baseline dataset framework
quarter_list = ['Q1', 'Q2', 'Q3', 'Q4']
month_order_list = ['January', 'February', 'March', 'April', 'May', 'June', 
                    'July', 'August', 'September', 'October', 'November', 'December']
month_list = [m for m in month_order_list if m in df_long_clean['Month'].dropna().unique()]
day_order_list = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_list = [d for d in day_order_list if d in df_long_clean['Day_of_Week'].dropna().unique()]

# Build the layout selector widgets
quarter_select = widgets.SelectMultiple(options=quarter_list, value=(), description='Quarters:', rows=4)
month_select = widgets.SelectMultiple(options=month_list, value=(), description='Months:', rows=5)
day_of_week_select = widgets.SelectMultiple(options=weekday_list, value=(), description='Weekdays:', rows=5)

all_qtr_check = widgets.Checkbox(value=False, description='All Quarters')
all_mth_check = widgets.Checkbox(value=False, description='All Months')
all_dow_check = widgets.Checkbox(value=False, description='All Weekdays')

refresh_btn = widgets.ToggleButton(value=False, description='Refresh Data', button_style='info')
download_output = widgets.Output()

# Arrange Filter Columns Grid Layout
ui_layout = widgets.VBox([
    widgets.HBox([quarter_select, month_select, day_of_week_select]),
    widgets.HBox([all_qtr_check, all_mth_check, all_dow_check, refresh_btn])
])

@interact(quarters=quarter_select, months=month_select, weekdays=day_of_week_select,
          all_qtrs=all_qtr_check, all_mths=all_mth_check, all_weekdays=all_dow_check, refresh=refresh_btn)
def show_and_export_quantity_dashboard(quarters, months, weekdays, all_qtrs, all_mths, all_weekdays, refresh):
    download_output.clear_output()

    if refresh_btn.value:
        refresh_btn.value = False

    # Selection State Gate Validations
    chosen_qtrs = quarter_list if all_qtrs else quarters
    chosen_mths = month_list if all_mths else months
    chosen_wdays = weekday_list if all_weekdays else weekdays

    if not chosen_qtrs:
        print("Step 1: Choose Quarters (or click 'All Quarters')")
        return
    if not chosen_mths:
        print("Step 2: Choose Months (or click 'All Months')")
        return
    if not chosen_wdays:
        print("Step 3: Choose Weekdays (or click 'All Weekdays')")
        return

    # 1. Execute metrics computation pipeline
    result = get_quantity_per_customer_analysis(df_long_clean, chosen_qtrs, chosen_mths, chosen_wdays)

    if result.empty:
        print("No matching transaction data fits this combined filter query.")
        return

    # Limit view to Top 15 rows for clean display density inside the notebook, while exporting everything
    display_result = result.head(15).copy()

    # 2. In-Memory Excel Exporter Architecture
    output_buffer = io.BytesIO()
    with pd.ExcelWriter(output_buffer, engine='xlsxwriter') as writer:
        result.to_excel(writer, index=False, sheet_name='Customer Volume Quantities')
        workbook = writer.book
        worksheet = writer.sheets['Customer Volume Quantities']
        num_format = workbook.add_format({'num_format': '#,##0', 'valign': 'top'})
        
        worksheet.set_column('A:A', 50)  # Customer Column
        worksheet.set_column('B:B', 25, num_format)  # Total Quantity Column

    excel_data = output_buffer.getvalue()
    b64_encoded = base64.b64encode(excel_data).decode()

    button_html = f'''
    <a download="customer_purchased_quantity_report.xlsx" href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64_encoded}" target="_blank">
        <button style="background-color:#1F6E43; color:white; padding:8px 16px; border:none; border-radius:4px; font-weight:bold; cursor:pointer; margin-bottom:12px;">
            Download Volume Report (.xlsx)
        </button>
    </a>
    '''
    with download_output:
        display(HTML(button_html))
    display(download_output)

    # Render Clean Data Grid View Frame
    styled_df = display_result.style.format({'Total_Quantity': '{:,.0f}'}).set_properties(**{
        'text-align': 'right', 'vertical-align': 'top'
    }).set_properties(subset=['Customer'], **{'text-align': 'left'})
    display(styled_df)
    print("\n")

    # 3. --- GRAPH FEATURE: HORIZONTAL BAR CHART GRAPH ---
    # Re-sort low-to-high specifically for horizontal alignment so the top item stays at the top peak
    display_result = display_result.sort_values(by='Total_Quantity', ascending=True)

    fig = px.bar(
        display_result,
        x='Total_Quantity',
        y='Customer',
        title='Top Customers by Total Purchased Quantity Volume',
        text='Total_Quantity',
        color='Total_Quantity',
        color_continuous_scale=px.colors.sequential.Agsunset,
        orientation='h',
        labels={'Total_Quantity': 'Total Quantity Units', 'Customer': 'Customer Identity'}
    )
    
    fig.update_traces(
        texttemplate='%{text:,.0f}',
        textposition='outside',
        hovertemplate='<b>Customer Name:</b> %{y}<br><b>Total Volume Units:</b> %{x:,.0f}<extra></extra>'
    )
    
    fig.update_layout(
        xaxis_title="Total Quantity Volume Units Purchased", 
        yaxis_title="Customer Account Profiles",
        height=550, 
        margin=dict(t=50, b=50, l=150, r=100),  # Generous padding prevents string truncation
        coloraxis_showscale=False
    )
    fig.show()


interactive(children=(SelectMultiple(description='Quarters:', options=('Q1', 'Q2', 'Q3', 'Q4'), rows=4, value=…

In [147]:
# Inactive customers
import io
import base64
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML
import plotly.graph_objects as go

def get_inactive_customers_analysis(df, selected_quarters, selected_months, selected_days_of_week):
    df_filtered = df.copy()
    if selected_quarters:
        df_filtered = df_filtered[df_filtered['Quarter_Label'].isin(selected_quarters)]
    if selected_months:
        df_filtered = df_filtered[df_filtered['Month'].isin(selected_months)]
    if selected_days_of_week:
        df_filtered = df_filtered[df_filtered['Day_of_Week'].isin(selected_days_of_week)]

    all_customers = sorted(df['Customer'].dropna().unique())
    active_customers = set(df_filtered['Customer'].dropna().unique())
    inactive_list = [customer for customer in all_customers if customer not in active_customers]

    total_count = len(all_customers)
    inactive_count = len(inactive_list)
    active_count = total_count - inactive_count
    inactive_percentage = (inactive_count / total_count) * 100 if total_count else 0

    inactive_df = pd.DataFrame({
        'Number': [f'{index + 1}.' for index in range(len(inactive_list))],
        'Inactive_Customer_Name': inactive_list
    })
    if inactive_df.empty:
        inactive_df = pd.DataFrame(columns=['Number', 'Inactive_Customer_Name'])

    return inactive_df, total_count, active_count, inactive_count, inactive_percentage

quarter_list = ['Q1', 'Q2', 'Q3', 'Q4']
month_order_list = ['January', 'February', 'March', 'April', 'May', 'June',
                    'July', 'August', 'September', 'October', 'November', 'December']
day_order_list = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

quarter_select = widgets.SelectMultiple(options=quarter_list, value=(), description='Quarters:', rows=4)
month_select = widgets.SelectMultiple(options=[], value=(), description='Months:', rows=5)
day_of_week_select = widgets.SelectMultiple(
    options=[day for day in day_order_list if day in df_long_clean['Day_of_Week'].dropna().unique()],
    value=(), description='Weekdays:', rows=5
)
all_qtr_check = widgets.Checkbox(value=False, description='All Quarters')
all_mth_check = widgets.Checkbox(value=False, description='All Months')
all_dow_check = widgets.Checkbox(value=False, description='All Weekdays')
refresh_btn = widgets.ToggleButton(value=False, description='Refresh Data', button_style='info')
download_output = widgets.Output()

def update_months_options(*args):
    chosen_quarters = quarter_list if all_qtr_check.value else quarter_select.value
    if not chosen_quarters:
        month_select.options = []
        month_select.value = ()
        return

    valid_months = df_long_clean[
        df_long_clean['Quarter_Label'].isin(chosen_quarters)
    ]['Month'].dropna().unique()
    sorted_months = [month for month in month_order_list if month in valid_months]
    old_value = month_select.value
    month_select.options = sorted_months
    month_select.value = tuple(month for month in old_value if month in sorted_months)

quarter_select.observe(update_months_options, names='value')
all_qtr_check.observe(update_months_options, names='value')
update_months_options()

display(widgets.VBox([
    widgets.HBox([quarter_select, month_select, day_of_week_select]),
    widgets.HBox([all_qtr_check, all_mth_check, all_dow_check, refresh_btn])
]))

dashboard_output = widgets.Output()
display(dashboard_output)

def render_dashboard(quarters, months, weekdays, all_qtrs, all_mths, all_weekdays, refresh):
    with dashboard_output:
        dashboard_output.clear_output()
        chosen_quarters = quarter_list if all_qtrs else quarters
        chosen_months = list(month_select.options) if all_mths else months
        chosen_weekdays = list(day_of_week_select.options) if all_weekdays else weekdays

        if not chosen_quarters:
            print("Step 1: Choose Quarters (or click 'All Quarters')")
            return
        if not chosen_months:
            print("Step 2: Choose Months (or click 'All Months')")
            return
        if not chosen_weekdays:
            print("Step 3: Choose Weekdays (or click 'All Weekdays')")
            return

        result, total_db, active_count, inactive_count, inactive_pct = get_inactive_customers_analysis(
            df_long_clean, chosen_quarters, chosen_months, chosen_weekdays
        )

        display(HTML(f'''
        <div style="display:flex; gap:15px; margin-bottom:15px; font-family:sans-serif;">
            <div style="background:#F8F9FA; padding:10px; flex:1; text-align:center;"><b>Total Database</b><h2>{total_db}</h2></div>
            <div style="background:#E6F4EA; padding:10px; flex:1; text-align:center;"><b>Active Clients</b><h2>{active_count}</h2></div>
            <div style="background:#FCE8E6; padding:10px; flex:1; text-align:center;"><b>Inactive Accounts</b><h2>{inactive_count}</h2></div>
        </div>
        '''))

        if result.empty:
            print("All customers in the database were active during this filtered timeframe.")
            return

        output_buffer = io.BytesIO()
        with pd.ExcelWriter(output_buffer, engine='xlsxwriter') as writer:
            result[['Inactive_Customer_Name']].to_excel(
                writer, index=False, sheet_name='Inactive Accounts Registry'
            )
            writer.sheets['Inactive Accounts Registry'].set_column('A:A', 50)

        encoded_data = base64.b64encode(output_buffer.getvalue()).decode()
        display(HTML(f'''
        <a download="inactive_customers_churn_report.xlsx"
           href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{encoded_data}">
            <button style="background-color:#1F6E43;color:white;padding:8px 16px;border:none;border-radius:4px;cursor:pointer;">
                Download Inactive Registry (.xlsx)
            </button>
        </a>
        '''))

        display(result.style.set_properties(**{
            'text-align': 'left', 'vertical-align': 'top'
        }).set_properties(subset=['Number'], **{
            'text-align': 'right', 'width': '50px'
        }))

        fig = go.Figure(go.Indicator(
            mode='gauge+number',
            value=inactive_pct,
            domain={'x': [0, 1], 'y': [0, 1]},
            title={'text': 'Account Inactivity Churn Rate (%)', 'font': {'size': 14}},
            number={'valueformat': '.1f', 'suffix': '%'},
            gauge={
                'axis': {'range': [0, 100], 'tickwidth': 1, 'tickcolor': 'darkblue'},
                'bar': {'color': '#C5221F'},
                'bgcolor': 'white',
                'borderwidth': 2,
                'bordercolor': 'gray',
                'steps': [
                    {'range': [0, 40], 'color': '#E6F4EA'},
                    {'range': [40, 70], 'color': '#FEF7E0'},
                    {'range': [70, 100], 'color': '#FCE8E6'}
                ]
            }
        ))
        fig.update_layout(height=280, margin=dict(t=40, b=20, l=40, r=40))
        fig.show()

out = widgets.interactive_output(render_dashboard, {
    'quarters': quarter_select,
    'months': month_select,
    'weekdays': day_of_week_select,
    'all_qtrs': all_qtr_check,
    'all_mths': all_mth_check,
    'all_weekdays': all_dow_check,
    'refresh': refresh_btn
})

Output()

In [148]:
# Growing customers and Declining customers
import io
import base64
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display, HTML
import plotly.express as px

def get_growth_decline_analysis(df, selected_quarters, selected_months, selected_days_of_week,
                                show_all_quarters=False, show_all_months=False, show_all_days_of_week=False):
    """
    Filters data dynamically and calculates customer transaction velocity profiles 
    by contrasting the active filter view against its preceding historical window.
    """
    df_master = df.copy()
    
    # 1. Isolate the Active Target View Slice
    df_current = df_master.copy()
    if not show_all_quarters and selected_quarters:
        df_current = df_current[df_current['Quarter_Label'].isin(selected_quarters)]
    if not show_all_months and selected_months:
        df_current = df_current[df_current['Month'].isin(selected_months)]
    if not show_all_days_of_week and selected_days_of_week:
        df_current = df_current[df_current['Day_of_Week'].isin(selected_days_of_week)]
        
    if df_current.empty:
        return pd.DataFrame()

    # 2. Establish the Historical Comparison View Slice
    # Isolates records that fall completely outside the active current target snapshot window
    current_indices = df_current.index
    df_historical = df_master.drop(index=current_indices)
    
    # If historical data is missing (e.g. tracking the entire database), look at the first half vs second half
    if df_historical.empty:
        midpoint = len(df_master) // 2
        df_historical = df_master.iloc[:midpoint]
        df_current = df_master.iloc[midpoint:]

    # 3. Aggregate Transaction Quantities across both periods
    current_activity = df_current.groupby('Customer').size().reset_index(name='Current_Transactions')
    historic_activity = df_historical.groupby('Customer').size().reset_index(name='Previous_Transactions')
    
    # Merge periods into a single tracking matrix
    merged = pd.merge(current_activity, historic_activity, on='Customer', how='outer').fillna(0)
    
    # 4. Calculate transactional velocity deltas
    merged['Transaction_Delta'] = merged['Current_Transactions'] - merged['Previous_Transactions']
    
    def assign_velocity_status(val):
        if val > 0:
            return 'Growing Activity'
        elif val < 0:
            return 'Declining Contraction'
        return 'Stable Consistency'
        
    merged['Velocity_Status'] = merged['Transaction_Delta'].apply(assign_velocity_status)
    
    # Sort descending by the scale of net activity shift
    return merged.sort_values(by='Transaction_Delta', ascending=False).reset_index(drop=True)

# Extract option profiles from baseline dataset framework
quarter_list = ['Q1', 'Q2', 'Q3', 'Q4']
month_order_list = ['January', 'February', 'March', 'April', 'May', 'June', 
                    'July', 'August', 'September', 'October', 'November', 'December']
month_list = [m for m in month_order_list if m in df_long_clean['Month'].dropna().unique()]
day_order_list = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_list = [d for d in day_order_list if d in df_long_clean['Day_of_Week'].dropna().unique()]

# Build UI selection components
quarter_select = widgets.SelectMultiple(options=quarter_list, value=(), description='Quarters:', rows=4)
month_select = widgets.SelectMultiple(options=month_list, value=(), description='Months:', rows=5)
day_of_week_select = widgets.SelectMultiple(options=weekday_list, value=(), description='Weekdays:', rows=5)

all_qtr_check = widgets.Checkbox(value=False, description='All Quarters')
all_mth_check = widgets.Checkbox(value=False, description='All Months')
all_dow_check = widgets.Checkbox(value=False, description='All Weekdays')

refresh_btn = widgets.ToggleButton(value=False, description='Refresh Data', button_style='info')
download_output = widgets.Output()

ui_layout = widgets.VBox([
    widgets.HBox([quarter_select, month_select, day_of_week_select]),
    widgets.HBox([all_qtr_check, all_mth_check, all_dow_check, refresh_btn])
])

@interact(quarters=quarter_select, months=month_select, weekdays=day_of_week_select,
          all_qtrs=all_qtr_check, all_mths=all_mth_check, all_weekdays=all_dow_check, refresh=refresh_btn)
def show_and_export_customer_velocity_dashboard(quarters, months, weekdays, all_qtrs, all_mths, all_weekdays, refresh):
    download_output.clear_output()

    if refresh_btn.value:
        refresh_btn.value = False

    # Selection State Gate Validations
    chosen_qtrs = quarter_list if all_qtrs else quarters
    chosen_mths = month_list if all_mths else months
    chosen_wdays = weekday_list if all_weekdays else weekdays

    if not chosen_qtrs:
        print("Step 1: Choose Quarters (or click 'All Quarters')")
        return
    if not chosen_mths:
        print("Step 2: Choose Months (or click 'All Months')")
        return
    if not chosen_wdays:
        print("Step 3: Choose Weekdays (or click 'All Weekdays')")
        return

    # 1. Execute metrics velocity computation pipeline
    result = get_growth_decline_analysis(
        df_long_clean, chosen_qtrs, chosen_mths, chosen_wdays,
        show_all_quarters=all_qtrs, show_all_months=all_mths, show_all_days_of_week=all_weekdays
    )

    if result.empty:
        print("No matching transaction data fits this combined filter query.")
        return

    # Gather baseline volume dimensions for status KPI header metrics
    grow_count = len(result[result['Velocity_Status'] == 'Growing Activity'])
    decline_count = len(result[result['Velocity_Status'] == 'Declining Contraction'])
    stable_count = len(result[result['Velocity_Status'] == 'Stable Consistency'])

    # 2. Display High-Impact Summary KPI Blocks
    summary_html = f"""
    <div style="display: flex; gap: 15px; margin-bottom: 15px; font-family: sans-serif;">
        <div style="background-color: #E6F4EA; border-top: 4px solid #137333; padding: 10px; border-radius: 4px; flex: 1; text-align: center;">
            <span style="color: #137333; font-size: 11px; font-weight: bold; text-transform: uppercase;">Growing Accounts</span>
            <h2 style="margin: 5px 0 0 0; color: #137333;">{grow_count}</h2>
        </div>
        <div style="background-color: #FCE8E6; border-top: 4px solid #C5221F; padding: 10px; border-radius: 4px; flex: 1; text-align: center;">
            <span style="color: #C5221F; font-size: 11px; font-weight: bold; text-transform: uppercase;">Declining Accounts</span>
            <h2 style="margin: 5px 0 0 0; color: #C5221F;">{decline_count}</h2>
        </div>
        <div style="background-color: #F8F9FA; border-top: 4px solid #5F6368; padding: 10px; border-radius: 4px; flex: 1; text-align: center;">
            <span style="color: #5F6368; font-size: 11px; font-weight: bold; text-transform: uppercase;">Stable Accounts</span>
            <h2 style="margin: 5px 0 0 0; color: #202124;">{stable_count}</h2>
        </div>
    </div>
    """
    display(HTML(summary_html))

    # Clean display views down to top and bottom outliers to prevent massive scroll footprints
    display_df = pd.concat([result.head(10), result.tail(10)]).drop_duplicates().copy()

    # 3. In-Memory Excel Exporter Architecture
    output_buffer = io.BytesIO()
    with pd.ExcelWriter(output_buffer, engine='xlsxwriter') as writer:
        result.to_excel(writer, index=False, sheet_name='Customer Velocity Analysis')
        workbook = writer.book
        worksheet = writer.sheets['Customer Velocity Analysis']
        num_format = workbook.add_format({'num_format': '#,##0', 'valign': 'top'})
        
        worksheet.set_column('A:A', 45)  # Customer
        worksheet.set_column('B:B', 20, num_format)  # Current Transactions
        worksheet.set_column('C:C', 20, num_format)  # Previous Transactions
        worksheet.set_column('D:D', 20, num_format)  # Net Delta
        worksheet.set_column('E:E', 25)  # Status Label

    excel_data = output_buffer.getvalue()
    b64_encoded = base64.b64encode(excel_data).decode()

    button_html = f'''
    <a download="customer_growth_decline_velocity.xlsx" href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64_encoded}" target="_blank">
        <button style="background-color:#1F6E43; color:white; padding:8px 16px; border:none; border-radius:4px; font-weight:bold; cursor:pointer; margin-bottom:15px;">
            Download Velocity Registry (.xlsx)
        </button>
    </a>
    '''
    with download_output:
        display(HTML(button_html))
    display(download_output)

    # Render Clean Data Grid View Frame
    styled_df = display_df.style.format({
        'Current_Transactions': '{:,.0f}',
        'Previous_Transactions': '{:,.0f}',
        'Transaction_Delta': '{:,.0f}'
    }).set_properties(**{
        'text-align': 'right', 'vertical-align': 'top'
    }).set_properties(subset=['Customer', 'Velocity_Status'], **{'text-align': 'left'})
    display(styled_df)
    print("\n")

    # 4. --- GRAPH FEATURE: INTERACTIVE DIVERGING VELOCITY BAR CHART ---
    # Limits graphic view strictly to active fluctuating accounts for extreme visual clarity
    graph_df = display_df[display_df['Velocity_Status'] != 'Stable Consistency'].copy()
    graph_df = graph_df.sort_values(by='Transaction_Delta', ascending=True)

    fig = px.bar(
        graph_df,
        x='Transaction_Delta',
        y='Customer',
        color='Velocity_Status',
        color_discrete_map={'Growing Activity': '#137333', 'Declining Contraction': '#C5221F'},
        orientation='h',
        title='Customer Transaction Velocity Trajectories (Net Variance Count)',
        text='Transaction_Delta',
        labels={'Transaction_Delta': 'Net Frequency Shift', 'Customer': 'Customer Identity Account'}
    )
    
    fig.update_traces(
        texttemplate='%{text:+,%.0f}',  # Forces automated plus (+) or minus (-) signing prefixes
        textposition='outside',
        hovertemplate='<b>Client:</b> %{y}<br><b>Net Frequency Variance:</b> %{x:+,0f} orders<extra></extra>'
    )

    fig.update_layout(
        xaxis_title="Net Order Volume Variance (Current vs Historic)",
        yaxis_title="Customer Account Profiles",
        height=550,
        margin=dict(t=50, b=50, l=150, r=100),
        legend_title_text='Velocity Tiers'
    )
    fig.show()

interactive(children=(SelectMultiple(description='Quarters:', options=('Q1', 'Q2', 'Q3', 'Q4'), rows=4, value=…

In [149]:
# Customer concentration analysis
# CASCADING CUSTOMER CONCENTRATION ANALYSIS DASHBOARD WITH SALES VALUE
import io
import base64
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML

def get_customer_concentration_analysis(df, selected_customers, selected_quarters, selected_months, selected_days_of_week, top_n_tier=10, show_all_customers=False):
    """
    Filters data dynamically and calculates customer concentration brackets based on 
    Sales Value and Transaction Counts, grouping remainder accounts as 'Others'.
    """
    df_filtered = df.copy()
    
    # 1. Apply Dynamic Time Filters
    if selected_quarters:
        df_filtered = df_filtered[df_filtered['Quarter_Label'].isin(selected_quarters)]
    if selected_months:
        df_filtered = df_filtered[df_filtered['Month'].isin(selected_months)]
    if selected_days_of_week:
        df_filtered = df_filtered[df_filtered['Day_of_Week'].isin(selected_days_of_week)]
        
    # 2. Apply Dynamic Customer Filter
    if not show_all_customers and selected_customers:
        df_filtered = df_filtered[df_filtered['Customer'].isin(selected_customers)]
        
    if df_filtered.empty:
        return pd.DataFrame(), pd.DataFrame()
    
    # Check the structural long-form column reference name for sales value
    sales_col = 'Sales_Value' if 'Sales_Value' in df_filtered.columns else 'Quantity'
    
    # 3. Aggregate Transaction Count AND Total Revenue simultaneously
    customer_metrics = df_filtered.groupby('Customer').agg(
        Transaction_Count=(sales_col, 'count'),
        Total_Sales_Value=(sales_col, 'sum')
    ).reset_index()
    
    # CRITICAL SORT SWITCH: Rank customers by revenue footprint instead of order counts
    customer_metrics = customer_metrics.sort_values(by='Total_Sales_Value', ascending=False).reset_index(drop=True)
    
    grand_total_revenue = customer_metrics['Total_Sales_Value'].sum()
    if grand_total_revenue == 0:
        return pd.DataFrame(), pd.DataFrame()
        
    # Calculate financial exposure concentrations
    customer_metrics['Percentage_Contribution'] = (customer_metrics['Total_Sales_Value'] / grand_total_revenue) * 100
    customer_metrics['Cumulative_Percentage'] = customer_metrics['Percentage_Contribution'].cumsum()
    
    # 4. Compile the Concentration Cohort Matrix (Top N vs Others)
    top_customers = customer_metrics.head(top_n_tier).copy()
    others = customer_metrics.iloc[top_n_tier:]
    
    if not others.empty:
        others_row = pd.DataFrame([{
            'Customer': 'Others (Remaining Account Balance Group)',
            'Transaction_Count': others['Transaction_Count'].sum(),
            'Total_Sales_Value': others['Total_Sales_Value'].sum(),
            'Percentage_Contribution': others['Percentage_Contribution'].sum(),
            'Cumulative_Percentage': 100.0
        }])
        concentration_matrix = pd.concat([top_customers, others_row], ignore_index=True)
    else:
        concentration_matrix = top_customers
        
    return customer_metrics, concentration_matrix

# --- Dynamic Filter Cascading Setup ---
quarter_list = ['Q1', 'Q2', 'Q3', 'Q4']
month_order_list = ['January', 'February', 'March', 'April', 'May', 'June', 
                    'July', 'August', 'September', 'October', 'November', 'December']
day_order_list = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Initialize UI Components
quarter_select = widgets.SelectMultiple(options=quarter_list, value=(), description='Quarters:', rows=4)
month_select = widgets.SelectMultiple(options=(), value=(), description='Months:', rows=5)
day_of_week_select = widgets.SelectMultiple(
    options=[day for day in day_order_list if day in df_long_clean['Day_of_Week'].dropna().unique()],
    value=(), description='Weekdays:', rows=5
)
customer_select = widgets.SelectMultiple(options=(), value=(), description='Customers:', rows=6)

all_qtr_check = widgets.Checkbox(value=False, description='All Quarters')
all_mth_check = widgets.Checkbox(value=False, description='All Months')
all_dow_check = widgets.Checkbox(value=False, description='All Weekdays')
all_cust_check = widgets.Checkbox(value=False, description='All Customers')

top_n_dropdown = widgets.Dropdown(
    options=[('Top 5 Accounts', 5), ('Top 10 Accounts', 10), ('Top 20 Accounts', 20), 
             ('Top 30 Accounts', 30), ('Top 40 Accounts', 40), ('Top 50 Accounts', 50)],
    value=10, description='Visibility Tier:'
)
refresh_btn = widgets.ToggleButton(value=False, description='Refresh Data', button_style='info')
dashboard_output = widgets.Output()

def update_cascading_options(*args):
    chosen_quarters = quarter_list if all_qtr_check.value else quarter_select.value
    if not chosen_quarters:
        month_select.options = ()
        month_select.value = ()
        customer_select.options = ()
        customer_select.value = ()
        return

    filtered = df_long_clean[df_long_clean['Quarter_Label'].isin(chosen_quarters)]
    valid_months = filtered['Month'].dropna().unique()
    month_options = [month for month in month_order_list if month in valid_months]
    month_select.options = month_options
    month_select.value = tuple(month for month in month_select.value if month in month_options)

    chosen_months = list(month_select.options) if all_mth_check.value else month_select.value
    if chosen_months:
        filtered = filtered[filtered['Month'].isin(chosen_months)]
    chosen_days = list(day_of_week_select.options) if all_dow_check.value else day_of_week_select.value
    if chosen_days:
        filtered = filtered[filtered['Day_of_Week'].isin(chosen_days)]

    customer_options = sorted(filtered['Customer'].dropna().unique().tolist())
    customer_select.options = customer_options
    customer_select.value = tuple(customer for customer in customer_select.value if customer in customer_options)

for widget in [quarter_select, all_qtr_check, month_select, all_mth_check, day_of_week_select, all_dow_check]:
    widget.observe(update_cascading_options, names='value')
update_cascading_options()

display(widgets.VBox([
    widgets.HBox([quarter_select, month_select, day_of_week_select]),
    widgets.HBox([all_qtr_check, all_mth_check, all_dow_check]),
    widgets.HBox([customer_select, top_n_dropdown, all_cust_check, refresh_btn])
]))
display(dashboard_output)

def render_dashboard_metrics(quarters, months, weekdays, customers, all_qtrs, all_mths, all_weekdays, all_custs, top_n, refresh):
    with dashboard_output:
        dashboard_output.clear_output()
        chosen_quarters = quarter_list if all_qtrs else quarters
        chosen_months = list(month_select.options) if all_mths else months
        chosen_weekdays = list(day_of_week_select.options) if all_weekdays else weekdays
        chosen_customers = list(customer_select.options) if all_custs else customers

        if not chosen_quarters or not chosen_months or not chosen_weekdays:
            print('Choose quarters, months, and weekdays, or enable their All options.')
            return
        if not chosen_customers and not all_custs:
            print('Choose customers or enable All Customers.')
            return

        rankings, composition = get_customer_concentration_analysis(
            df_long_clean, chosen_customers, chosen_quarters, chosen_months, chosen_weekdays,
            top_n_tier=top_n, show_all_customers=all_custs
        )
        if rankings.empty:
            print('No matching transaction data fits this combined filter query.')
            return

        # Excel Writer with separate Currency Formatting for the new column
        output_buffer = io.BytesIO()
        with pd.ExcelWriter(output_buffer, engine='xlsxwriter') as writer:
            rankings.to_excel(writer, index=False, sheet_name='Full Customer Rankings')
            composition.to_excel(writer, index=False, sheet_name='Concentration Breakdown')
            workbook = writer.book
            count_format = workbook.add_format({'num_format': '#,##0', 'valign': 'top'})
            money_format = workbook.add_format({'num_format': 'KSh #,##0.00', 'valign': 'top'})
            percent_format = workbook.add_format({'num_format': '0.00"%"', 'valign': 'top'})
            
            for worksheet in writer.sheets.values():
                worksheet.set_column('A:A', 45)  # Customer
                worksheet.set_column('B:B', 18, count_format)  # Transaction Count
                worksheet.set_column('C:C', 22, money_format)  # Total Sales Value
                worksheet.set_column('D:E', 22, percent_format)  # Percentage and Cumulative Percentage

        encoded_data = base64.b64encode(output_buffer.getvalue()).decode()
        display(HTML(f'''<a download="customer_value_concentration_report.xlsx"
            href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{encoded_data}">
            <button style="background-color:#1F6E43;color:white;padding:8px 16px;border:none;border-radius:4px;cursor:pointer;margin-bottom:12px;font-weight:bold;">
            Download Concentration Matrix (.xlsx)</button></a>'''))

        # Display beautifully styled pandas frame inside Jupyter Notebook
        styled_df = composition.style.format({
            'Transaction_Count': '{:,.0f}',
            'Total_Sales_Value': 'KSh {:,.2f}',
            'Percentage_Contribution': '{:.2f}%',
            'Cumulative_Percentage': '{:.2f}%'
        }).set_properties(**{
            'text-align': 'right', 'vertical-align': 'top'
        }).set_properties(subset=['Customer'], **{
            'text-align': 'left'
        })
        
        display(styled_df)

# Bind variables dynamically to the interactive output engine loops
out = widgets.interactive_output(render_dashboard_metrics, {
    'quarters': quarter_select, 'months': month_select, 'weekdays': day_of_week_select, 'customers': customer_select,
    'all_qtrs': all_qtr_check, 'all_mths': all_mth_check, 'all_weekdays': all_dow_check, 'all_custs': all_cust_check,'top_n': top_n_dropdown, 'refresh': refresh_btn})

Output()

## ORDER / TRANSACTION ANALYSIS

In [150]:
# Order analysis using Using Invoice No
""" 
Total orders
Average quantity/order
Median quantity/order
Largest order
"""
# CASCADING ORDER ANALYSIS DASHBOARD USING INVOICE CODES
import io
import base64
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML

def get_order_analysis(df, selected_quarters, selected_months, selected_days_of_week):
    """
    Filters data dynamically and calculates order volume performance metrics 
    aggregated strictly by individual unique Invoice numbers.
    """
    df_filtered = df.copy()
    
    # 1. Apply Dynamic Structural Time Slicers
    if selected_quarters:
        df_filtered = df_filtered[df_filtered['Quarter_Label'].isin(selected_quarters)]
    if selected_months:
        df_filtered = df_filtered[df_filtered['Month'].isin(selected_months)]
    if selected_days_of_week:
        df_filtered = df_filtered[df_filtered['Day_of_Week'].isin(selected_days_of_week)]

    if df_filtered.empty:
        return pd.DataFrame(), 0, 0, 0, 0

    # Check the structural long-form volume column reference name safely
    quantity_col = 'Quantity' if 'Quantity' in df_filtered.columns else 'Sales_Value'

    # 2. Group transaction details by unique Invoice Number to capture true order footprint
    order_metrics = df_filtered.groupby('Invoice_No').agg(
        Total_Quantity=(quantity_col, 'sum')
    ).reset_index()

    # 3. Calculate descriptive statistical metrics boundaries
    total_orders = int(len(order_metrics))
    average_quantity = float(order_metrics['Total_Quantity'].mean()) if total_orders > 0 else 0.0
    median_quantity = float(order_metrics['Total_Quantity'].median()) if total_orders > 0 else 0.0
    largest_order = float(order_metrics['Total_Quantity'].max()) if total_orders > 0 else 0.0

    summary_df = pd.DataFrame({
        'Metric': ['Total Orders', 'Average Quantity Per Order', 'Median Quantity Per Order', 'Largest Order Size'],
        'Value': [total_orders, average_quantity, median_quantity, largest_order]
    })

    return summary_df, total_orders, average_quantity, median_quantity, largest_order

# --- Dynamic Filter Cascading Setup ---
quarter_list = ['Q1', 'Q2', 'Q3', 'Q4']
month_order_list = ['January', 'February', 'March', 'April', 'May', 'June', 
                    'July', 'August', 'September', 'October', 'November', 'December']
day_order_list = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Initialize UI Layout Elements
quarter_select = widgets.SelectMultiple(options=quarter_list, value=(), description='Quarters:', rows=4)
month_select = widgets.SelectMultiple(options=[], value=(), description='Months:', rows=5)
day_of_week_select = widgets.SelectMultiple(
    options=[day for day in day_order_list if day in df_long_clean['Day_of_Week'].dropna().unique()],
    value=(), description='Weekdays:', rows=5
)

# Individual Slicer Reset Checkboxes
all_qtr_check = widgets.Checkbox(value=False, description='All Quarters')
all_mth_check = widgets.Checkbox(value=False, description='All Months')
all_dow_check = widgets.Checkbox(value=False, description='All Weekdays')

refresh_btn = widgets.ToggleButton(value=False, description='Refresh Data', button_style='info')
dashboard_output = widgets.Output()

# Function to dynamically update month filters based on active quarters selected
def update_cascading_options(*args):
    chosen_quarters = quarter_list if all_qtr_check.value else quarter_select.value
    if not chosen_quarters:
        month_select.options = ()
        month_select.value = ()
        return

    filtered = df_long_clean[df_long_clean['Quarter_Label'].isin(chosen_quarters)]
    valid_months = filtered['Month'].dropna().unique()
    month_options = [month for month in month_order_list if month in valid_months]
    
    old_mth_val = month_select.value
    month_select.options = month_options
    month_select.value = tuple(month for month in old_mth_val if month in month_options)

# Attach observation listeners to track choices dynamically
for widget in [quarter_select, all_qtr_check]:
    widget.observe(update_cascading_options, names='value')
update_cascading_options()

# Render core Filter Shell Interface Panels
ui_layout = widgets.VBox([
    widgets.HBox([quarter_select, month_select, day_of_week_select]),
    widgets.HBox([all_qtr_check, all_mth_check, all_dow_check, refresh_btn])
])

display(ui_layout)
display(dashboard_output)

def render_order_dashboard(quarters, months, weekdays, all_qtrs, all_mths, all_weekdays, refresh):
    with dashboard_output:
        dashboard_output.clear_output()
        
        chosen_quarters = quarter_list if all_qtrs else quarters
        chosen_months = list(month_select.options) if all_mths else months
        chosen_weekdays = list(day_of_week_select.options) if all_weekdays else weekdays

        if not chosen_quarters or not chosen_months or not chosen_weekdays:
            print("Step 1: Choose quarters, months, and weekdays, or enable their All options.")
            return

        # Execute metrics calculation pipelines
        result, tot_orders, avg_qty, med_qty, max_qty = get_order_analysis(
            df_long_clean, chosen_quarters, chosen_months, chosen_weekdays
        )
        
        if result.empty:
            print("No matching transaction logs found for this timeframe combination.")
            return

        # 2. Display Professional Summary KPI Cards Text Layout
        summary_html = f"""
        <div style="display: flex; gap: 15px; margin-bottom: 15px; font-family: sans-serif;">
            <div style="background-color: #F8F9FA; border-top: 4px solid #1A73E8; padding: 10px; border-radius: 4px; flex: 1; text-align: center;">
                <span style="color: #5F6368; font-size: 11px; font-weight: bold; text-transform: uppercase;">Total Orders</span>
                <h2 style="margin: 5px 0 0 0; color: #202124;">{tot_orders:,.0f}</h2>
            </div>
            <div style="background-color: #E8F0FE; border-top: 4px solid #1A73E8; padding: 10px; border-radius: 4px; flex: 1; text-align: center;">
                <span style="color: #1A73E8; font-size: 11px; font-weight: bold; text-transform: uppercase;">Avg Quantity / Order</span>
                <h2 style="margin: 5px 0 0 0; color: #1A73E8;">{avg_qty:,.1f}</h2>
            </div>
            <div style="background-color: #E6F4EA; border-top: 4px solid #137333; padding: 10px; border-radius: 4px; flex: 1; text-align: center;">
                <span style="color: #137333; font-size: 11px; font-weight: bold; text-transform: uppercase;">Median Quantity</span>
                <h2 style="margin: 5px 0 0 0; color: #137333;">{med_qty:,.1f}</h2>
            </div>
            <div style="background-color: #FEF7E0; border-top: 4px solid #B06000; padding: 10px; border-radius: 4px; flex: 1; text-align: center;">
                <span style="color: #B06000; font-size: 11px; font-weight: bold; text-transform: uppercase;">Largest Order Size</span>
                <h2 style="margin: 5px 0 0 0; color: #B06000;">{max_qty:,.0f}</h2>
            </div>
        </div>
        """
        display(HTML(summary_html))

        # 3. In-Memory Excel Exporter Architecture
        output_buffer = io.BytesIO()
        with pd.ExcelWriter(output_buffer, engine='xlsxwriter') as writer:
            result.to_excel(writer, index=False, sheet_name='Order Summary Analysis')
            workbook = writer.book
            worksheet = writer.sheets['Order Summary Analysis']
            num_format = workbook.add_format({'num_format': '#,##0.00', 'valign': 'top'})
            
            worksheet.set_column('A:A', 30)  # Metric column
            worksheet.set_column('B:B', 20, num_format)  # Value column

        excel_data = output_buffer.getvalue()
        b64_encoded = base64.b64encode(excel_data).decode()

        button_html = f'''
        <a download="order_volume_analysis_report.xlsx" href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64_encoded}" target="_blank">
            <button style="background-color:#1F6E43; color:white; padding:8px 16px; border:none; border-radius:4px; font-weight:bold; cursor:pointer; margin-bottom:12px;">
                Download Order Analysis (.xlsx)
            </button>
        </a>
        '''
        display(HTML(button_html))

        # 4. Display Clean Data Grid View Frame
        styled_df = result.style.format({'Value': '{:,.2f}'}).set_properties(**{
            'text-align': 'right', 'vertical-align': 'top'
        }).set_properties(subset=['Metric'], **{'text-align': 'left'})
        
        display(styled_df)

# Bind data outputs to widgets dynamically using interactive_output manager logic loop
out = widgets.interactive_output(render_order_dashboard, {
    'quarters': quarter_select, 'months': month_select, 'weekdays': day_of_week_select,
    'all_qtrs': all_qtr_check, 'all_mths': all_mth_check, 'all_weekdays': all_dow_check, 'refresh': refresh_btn
})


Output()

In [151]:
# CASCADING CUSTOMER GROWTH ENGINE AND ORDER INVESTIGATION DASHBOARD
import io
import base64
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML

def investigate_customer_growth_drivers(df, selected_customers, selected_quarters, selected_months, selected_days_of_week, show_all_customers=False):
    """
    Performs deep investigative order analytics aggregated by Customer to determine
    if growth is volume-driven (basket size) or frequency-driven (order count).
    """
    df_filtered = df.copy()

    if selected_quarters:
        df_filtered = df_filtered[df_filtered['Quarter_Label'].isin(selected_quarters)]
    if selected_months:
        df_filtered = df_filtered[df_filtered['Month'].isin(selected_months)]
    if selected_days_of_week:
        df_filtered = df_filtered[df_filtered['Day_of_Week'].isin(selected_days_of_week)]
    if not show_all_customers and selected_customers:
        df_filtered = df_filtered[df_filtered['Customer'].isin(selected_customers)]

    if df_filtered.empty:
        return pd.DataFrame(), 0, 0, 0, 0, 0, 0

    quantity_col = 'Quantity' if 'Quantity' in df_filtered.columns else 'Sales_Value'
    invoice_grouped = df_filtered.groupby(['Customer', 'Invoice_No']).agg(
        Invoice_Quantity=(quantity_col, 'sum')
    ).reset_index()

    customer_investigation = invoice_grouped.groupby('Customer').agg(
        Total_Orders=('Invoice_No', 'count'),
        Average_Quantity_Per_Order=('Invoice_Quantity', 'mean'),
        Median_Quantity_Per_Order=('Invoice_Quantity', 'median'),
        Largest_Order_Size=('Invoice_Quantity', 'max')
    ).reset_index()

    total_active_customers = int(customer_investigation['Customer'].nunique())
    global_total_orders = int(customer_investigation['Total_Orders'].sum())
    global_avg_orders_per_cust = float(customer_investigation['Total_Orders'].mean()) if total_active_customers else 0.0
    global_avg_basket_size = float(invoice_grouped['Invoice_Quantity'].mean()) if global_total_orders else 0.0
    global_median_basket_size = float(invoice_grouped['Invoice_Quantity'].median()) if global_total_orders else 0.0
    global_largest_single_order = float(invoice_grouped['Invoice_Quantity'].max()) if global_total_orders else 0.0

    total_qty_per_cust = df_filtered.groupby('Customer')[quantity_col].sum().reset_index(name='Grand_Total_Quantity')
    customer_investigation = pd.merge(customer_investigation, total_qty_per_cust, on='Customer', how='left')
    customer_investigation = customer_investigation.sort_values(
        by='Total_Orders', ascending=False
    ).reset_index(drop=True)

    column_order = [
        'Customer', 'Total_Orders', 'Grand_Total_Quantity',
        'Average_Quantity_Per_Order', 'Median_Quantity_Per_Order', 'Largest_Order_Size'
    ]
    return (
        customer_investigation[column_order],
        total_active_customers,
        global_total_orders,
        global_avg_orders_per_cust,
        global_avg_basket_size,
        global_median_basket_size,
        global_largest_single_order
    )

quarter_list = ['Q1', 'Q2', 'Q3', 'Q4']
month_order_list = ['January', 'February', 'March', 'April', 'May', 'June',
                    'July', 'August', 'September', 'October', 'November', 'December']
day_order_list = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

quarter_select = widgets.SelectMultiple(options=quarter_list, value=(), description='Quarters:', rows=4)
month_select = widgets.SelectMultiple(options=(), value=(), description='Months:', rows=5)
day_of_week_select = widgets.SelectMultiple(
    options=[day for day in day_order_list if day in df_long_clean['Day_of_Week'].dropna().unique()],
    value=(), description='Weekdays:', rows=5
)
customer_select = widgets.SelectMultiple(options=(), value=(), description='Customers:', rows=6)
all_qtr_check = widgets.Checkbox(value=False, description='All Quarters')
all_mth_check = widgets.Checkbox(value=False, description='All Months')
all_dow_check = widgets.Checkbox(value=False, description='All Weekdays')
all_cust_check = widgets.Checkbox(value=False, description='All Customers')
refresh_btn = widgets.ToggleButton(value=False, description='Refresh Calculations', button_style='info')
dashboard_output = widgets.Output()

def update_cascading_options(*args):
    chosen_quarters = quarter_list if all_qtr_check.value else quarter_select.value
    if not chosen_quarters:
        month_select.options = ()
        month_select.value = ()
        customer_select.options = ()
        customer_select.value = ()
        return

    filtered = df_long_clean[df_long_clean['Quarter_Label'].isin(chosen_quarters)]
    valid_months = filtered['Month'].dropna().unique()
    month_options = [month for month in month_order_list if month in valid_months]
    month_select.options = month_options
    month_select.value = tuple(month for month in month_select.value if month in month_options)

    chosen_months = list(month_select.options) if all_mth_check.value else month_select.value
    if chosen_months:
        filtered = filtered[filtered['Month'].isin(chosen_months)]
    chosen_days = list(day_of_week_select.options) if all_dow_check.value else day_of_week_select.value
    if chosen_days:
        filtered = filtered[filtered['Day_of_Week'].isin(chosen_days)]

    customer_options = sorted(filtered['Customer'].dropna().unique().tolist())
    customer_select.options = customer_options
    customer_select.value = tuple(customer for customer in customer_select.value if customer in customer_options)

for widget in [quarter_select, all_qtr_check, month_select, all_mth_check, day_of_week_select, all_dow_check]:
    widget.observe(update_cascading_options, names='value')
update_cascading_options()

display(widgets.VBox([
    widgets.HBox([quarter_select, month_select, day_of_week_select]),
    widgets.HBox([all_qtr_check, all_mth_check, all_dow_check]),
    widgets.HBox([customer_select, all_cust_check, refresh_btn])
]))
display(dashboard_output)

def render_growth_investigation_dashboard(quarters, months, weekdays, customers, all_qtrs, all_mths, all_weekdays, all_custs, refresh):
    with dashboard_output:
        dashboard_output.clear_output()

        chosen_quarters = quarter_list if all_qtrs else quarters
        chosen_months = list(month_select.options) if all_mths else months
        chosen_weekdays = list(day_of_week_select.options) if all_weekdays else weekdays
        chosen_customers = list(customer_select.options) if all_custs else customers

        if not chosen_quarters or not chosen_months or not chosen_weekdays:
            print('Step 1: Choose quarters, months, and weekdays, or enable their All options.')
            return
        if not chosen_customers and not all_custs:
            print('Step 2: Choose customers or enable All Customers.')
            return

        result, active_cust_cnt, global_orders, avg_orders_per_cust, avg_basket, median_basket, largest_order = investigate_customer_growth_drivers(
            df_long_clean, chosen_customers, chosen_quarters, chosen_months, chosen_weekdays,
            show_all_customers=all_custs
        )
        if result.empty:
            print('No matching transaction data fits this combined filter query.')
            return

        summary_html = f"""
        <div style="display:flex; gap:15px; margin-bottom:20px; font-family:sans-serif;">
            <div style="background:#F8F9FA; border-top:4px solid #1A73E8; padding:10px; border-radius:4px; flex:1; text-align:center;">
                <span style="color:#5F6368; font-size:10px; font-weight:bold;">TOTAL ACTIVE CLIENTS</span>
                <h2 style="margin:4px 0; color:#202124;">{active_cust_cnt:,.0f}</h2>
            </div>
            <div style="background:#F8F9FA; border-top:4px solid #1A73E8; padding:10px; border-radius:4px; flex:1; text-align:center;">
                <span style="color:#5F6368; font-size:10px; font-weight:bold;">COMBINED ORDERS</span>
                <h2 style="margin:4px 0; color:#202124;">{global_orders:,.0f}</h2>
            </div>
            <div style="background:#FEF7E0; border-top:4px solid #B06000; padding:10px; border-radius:4px; flex:1; text-align:center;">
                <span style="color:#B06000; font-size:10px; font-weight:bold;">AVG ORDERS / CUSTOMER</span>
                <h2 style="margin:4px 0; color:#B06000;">{avg_orders_per_cust:,.1f}</h2>
            </div>
            <div style="background:#E6F4EA; border-top:4px solid #137333; padding:10px; border-radius:4px; flex:1; text-align:center;">
                <span style="color:#137333; font-size:10px; font-weight:bold;">AVG BASKET QUANTITY</span>
                <h2 style="margin:4px 0; color:#137333;">{avg_basket:,.1f}</h2>
            </div>
            <div style="background:#E6F4EA; border-top:4px solid #137333; padding:10px; border-radius:4px; flex:1; text-align:center;">
                <span style="color:#137333; font-size:10px; font-weight:bold;">MEDIAN BASKET SIZE</span>
                <h2 style="margin:4px 0; color:#137333;">{median_basket:,.1f}</h2>
            </div>
            <div style="background:#FEF7E0; border-top:4px solid #B06000; padding:10px; border-radius:4px; flex:1; text-align:center;">
                <span style="color:#B06000; font-size:10px; font-weight:bold;">LARGEST ORDER</span>
                <h2 style="margin:4px 0; color:#B06000;">{largest_order:,.1f}</h2>
            </div>
        </div>
        """
        display(HTML(summary_html))

        output_buffer = io.BytesIO()
        with pd.ExcelWriter(output_buffer, engine='xlsxwriter') as writer:
            result.to_excel(writer, index=False, sheet_name='Growth Driver Metrics')
            workbook = writer.book
            worksheet = writer.sheets['Growth Driver Metrics']
            int_format = workbook.add_format({'num_format': '#,##0', 'valign': 'top'})
            dec_format = workbook.add_format({'num_format': '#,##0.00', 'valign': 'top'})
            worksheet.set_column('A:A', 45)
            worksheet.set_column('B:C', 18, int_format)
            worksheet.set_column('D:E', 22, dec_format)
            worksheet.set_column('F:F', 22, dec_format)

        encoded_data = base64.b64encode(output_buffer.getvalue()).decode()
        display(HTML(f'''<a download="customer_growth_driver_metrics.xlsx"
            href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{encoded_data}">
            <button style="background:#1F6E43;color:white;padding:8px 16px;border:none;border-radius:4px;cursor:pointer;">
            Download Growth Investigation Metrics (.xlsx)</button></a>'''))

        styled_df = result.style.format({
            'Total_Orders': '{:,.0f}',
            'Grand_Total_Quantity': '{:,.0f}',
            'Average_Quantity_Per_Order': '{:,.2f}',
            'Median_Quantity_Per_Order': '{:,.2f}',
            'Largest_Order_Size': '{:,.1f}'
        }).set_properties(**{
            'text-align': 'right', 'vertical-align': 'top'
        }).set_properties(subset=['Customer'], **{'text-align': 'left'})
        display(styled_df)

out = widgets.interactive_output(render_growth_investigation_dashboard, {
    'quarters': quarter_select,
    'months': month_select,
    'weekdays': day_of_week_select,
    'customers': customer_select,
    'all_qtrs': all_qtr_check,
    'all_mths': all_mth_check,
    'all_weekdays': all_dow_check,
    'all_custs': all_cust_check,
    'refresh': refresh_btn
})

Output()

# SALES GROWTH

In [152]:
# Sales growth analysis
# Month-over-month growth
# CASCADING MONTH-OVER-MONTH SALES GROWTH ANALYSIS DASHBOARD WITH TOOLTIP VISUALIZATIONS
import io
import base64
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML
import plotly.express as px
import plotly.graph_objects as go

def calculate_monthly_growth_interactive(df, selected_customers, selected_quarters, selected_months, selected_days_of_week,
                                           show_all_customers=False, show_all_quarters=False, show_all_months=False, show_all_days_of_week=False):
    df_filtered = df.copy()

    if not show_all_quarters and selected_quarters:
        df_filtered = df_filtered[df_filtered['Quarter_Label'].isin(selected_quarters)]
    if not show_all_months and selected_months:
        df_filtered = df_filtered[df_filtered['Month'].isin(selected_months)]
    if not show_all_days_of_week and selected_days_of_week:
        df_filtered = df_filtered[df_filtered['Day_of_Week'].isin(selected_days_of_week)]
    if not show_all_customers and selected_customers:
        df_filtered = df_filtered[df_filtered['Customer'].isin(selected_customers)]

    if df_filtered.empty:
        return pd.DataFrame()

    sales_col = 'Sales_Value' if 'Sales_Value' in df_filtered.columns else 'Quantity'
    monthly_metrics = df_filtered.groupby('Month').agg(
        Total_Sales_Value=(sales_col, 'sum'),
        Transaction_Count=('Invoice_No', 'nunique')
    ).reset_index()

    month_order_list = ['January', 'February', 'March', 'April', 'May', 'June',
                        'July', 'August', 'September', 'October', 'November', 'December']
    monthly_metrics['Month'] = pd.Categorical(
        monthly_metrics['Month'], categories=month_order_list, ordered=True
    )
    monthly_metrics = monthly_metrics.sort_values('Month').reset_index(drop=True)
    monthly_metrics['Sales_Growth_Rate_Pct'] = monthly_metrics['Total_Sales_Value'].pct_change() * 100
    monthly_metrics['Transaction_Growth_Rate_Pct'] = monthly_metrics['Transaction_Count'].pct_change() * 100
    monthly_metrics['Month'] = monthly_metrics['Month'].astype(str)
    return monthly_metrics

quarter_list = ['Q1', 'Q2', 'Q3', 'Q4']
month_order_list = ['January', 'February', 'March', 'April', 'May', 'June',
                    'July', 'August', 'September', 'October', 'November', 'December']
day_order_list = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

quarter_select = widgets.SelectMultiple(options=quarter_list, value=(), description='Quarters:', rows=4)
month_select = widgets.SelectMultiple(options=(), value=(), description='Months:', rows=5)
day_of_week_select = widgets.SelectMultiple(
    options=[day for day in day_order_list if day in df_long_clean['Day_of_Week'].dropna().unique()],
    value=(), description='Weekdays:', rows=5
)
customer_select = widgets.SelectMultiple(options=(), value=(), description='Customers:', rows=6)
all_qtr_check = widgets.Checkbox(value=False, description='All Quarters')
all_mth_check = widgets.Checkbox(value=False, description='All Months')
all_dow_check = widgets.Checkbox(value=False, description='All Weekdays')
all_cust_check = widgets.Checkbox(value=False, description='All Customers')
refresh_btn = widgets.ToggleButton(value=False, description='Refresh Calculations', button_style='info')
dashboard_output = widgets.Output()

def update_cascading_options(*args):
    chosen_quarters = quarter_list if all_qtr_check.value else quarter_select.value
    if not chosen_quarters:
        month_select.options = ()
        month_select.value = ()
        customer_select.options = ()
        customer_select.value = ()
        return

    filtered = df_long_clean[df_long_clean['Quarter_Label'].isin(chosen_quarters)]
    valid_months = filtered['Month'].dropna().unique()
    month_options = [month for month in month_order_list if month in valid_months]
    month_select.options = month_options
    month_select.value = tuple(month for month in month_select.value if month in month_options)

    chosen_months = list(month_select.options) if all_mth_check.value else month_select.value
    if chosen_months:
        filtered = filtered[filtered['Month'].isin(chosen_months)]
    chosen_days = list(day_of_week_select.options) if all_dow_check.value else day_of_week_select.value
    if chosen_days:
        filtered = filtered[filtered['Day_of_Week'].isin(chosen_days)]

    customer_options = sorted(filtered['Customer'].dropna().unique().tolist())
    customer_select.options = customer_options
    customer_select.value = tuple(customer for customer in customer_select.value if customer in customer_options)

for widget in [quarter_select, all_qtr_check, month_select, all_mth_check, day_of_week_select, all_dow_check]:
    widget.observe(update_cascading_options, names='value')
update_cascading_options()

display(widgets.VBox([
    widgets.HBox([quarter_select, month_select, day_of_week_select]),
    widgets.HBox([all_qtr_check, all_mth_check, all_dow_check]),
    widgets.HBox([customer_select, all_cust_check, refresh_btn])
]))
display(dashboard_output)

def render_growth_analysis_dashboard(quarters, months, weekdays, customers, all_qtrs, all_mths, all_weekdays, all_custs, refresh):
    with dashboard_output:
        dashboard_output.clear_output()

        chosen_quarters = quarter_list if all_qtrs else quarters
        chosen_months = list(month_select.options) if all_mths else months
        chosen_weekdays = list(day_of_week_select.options) if all_weekdays else weekdays
        chosen_customers = list(customer_select.options) if all_custs else customers

        if not chosen_quarters or not chosen_months or not chosen_weekdays:
            print('Step 1: Choose quarters, months, and weekdays, or enable their All options.')
            return
        if not chosen_customers and not all_custs:
            print('Step 2: Choose customers or enable All Customers.')
            return

        result = calculate_monthly_growth_interactive(
            df_long_clean, chosen_customers, chosen_quarters, chosen_months, chosen_weekdays,
            show_all_customers=all_custs, show_all_quarters=all_qtrs,
            show_all_months=all_mths, show_all_days_of_week=all_weekdays
        )
        if result.empty:
            print('No matching transaction data fits this combined filter query.')
            return

        output_buffer = io.BytesIO()
        with pd.ExcelWriter(output_buffer, engine='xlsxwriter') as writer:
            result.to_excel(writer, index=False, sheet_name='Monthly Growth Trends')
            workbook = writer.book
            worksheet = writer.sheets['Monthly Growth Trends']
            count_format = workbook.add_format({'num_format': '#,##0', 'valign': 'top'})
            money_format = workbook.add_format({'num_format': 'KSh #,##0.00', 'valign': 'top'})
            percent_format = workbook.add_format({'num_format': '0.00"%"', 'valign': 'top'})
            worksheet.set_column('A:A', 18)
            worksheet.set_column('B:B', 22, money_format)
            worksheet.set_column('C:C', 18, count_format)
            worksheet.set_column('D:E', 24, percent_format)

        encoded_data = base64.b64encode(output_buffer.getvalue()).decode()
        display(HTML(f'''<a download="monthly_sales_growth_analysis.xlsx"
            href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{encoded_data}">
            <button style="background-color:#1F6E43;color:white;padding:8px 16px;border:none;border-radius:4px;cursor:pointer;margin-bottom:15px;font-weight:bold;">
            Download Growth Analysis (.xlsx)</button></a>'''))

        styled_df = result.style.format({
            'Total_Sales_Value': 'KSh {:,.2f}',
            'Transaction_Count': '{:,.0f}',
            'Sales_Growth_Rate_Pct': '{:+.2f}%',
            'Transaction_Growth_Rate_Pct': '{:+.2f}%'
        }, na_rep='Baseline').set_properties(**{
            'text-align': 'right', 'vertical-align': 'top'
        }).set_properties(subset=['Month'], **{'text-align': 'left'})
        display(styled_df)
        print("\n")

        graph_df = result.copy()
        graph_df['Sales_Growth_Rate_Pct'] = graph_df['Sales_Growth_Rate_Pct'].fillna(0.0)
        graph_df['Transaction_Growth_Rate_Pct'] = graph_df['Transaction_Growth_Rate_Pct'].fillna(0.0)

        # GRAPH 1: DUAL-AXIS LINE CHART WITH ADVANCED TOOLTIPS
        fig_line = go.Figure()
        fig_line.add_trace(go.Scatter(
            x=graph_df['Month'],
            y=graph_df['Sales_Growth_Rate_Pct'],
            name='Sales Value Growth',
            mode='lines+markers',
            marker=dict(color='#1A73E8', size=8),
            line=dict(width=3),
            customdata=graph_df['Total_Sales_Value'],
            hovertemplate='<b>Month:</b> %{x}<br><b>Sales Growth Rate:</b> %{y:+.2f}%<br><b>Total Revenue:</b> KSh %{customdata:,.2f}<extra></extra>'
        ))

        # Transaction Volume Growth Trace (Secondary Right Y-Axis)
        fig_line.add_trace(go.Scatter(
            x=graph_df['Month'],
            y=graph_df['Transaction_Growth_Rate_Pct'],
            name='Order Volume Growth',
            mode='lines+markers',
            yaxis='y2',
            marker=dict(color='#C5221F', size=8),
            line=dict(width=3, dash='dash'),
            customdata=graph_df['Transaction_Count'],
            hovertemplate='<b>Month:</b> %{x}<br><b>Order Growth Rate:</b> %{y:+.2f}%<br><b>Total Orders Placed:</b> %{customdata:,.0f}<extra></extra>'
        ))

        fig_line.update_layout(
            title=dict(text='Month-over-Month Percentage Growth Progression (Revenue vs Order Counts)', font=dict(size=14)),
            height=400,
            margin=dict(t=60, b=50, l=60, r=60),
            xaxis=dict(title='Timeline Calendar Horizon'),
            yaxis=dict(
                title=dict(text='Revenue Growth Rate (%)', font=dict(color='#1A73E8')),
                tickfont=dict(color='#1A73E8'),
                tickformat='.1f'
            ),
            yaxis2=dict(
                title=dict(text='Order Volume Growth Rate (%)', font=dict(color='#C5221F')),
                tickfont=dict(color='#C5221F'),
                overlaying='y',
                side='right',
                tickformat='.1f'
            ),
            legend=dict(x=0.02, y=0.98, bgcolor='rgba(255,255,255,0.6)'),
            hovermode='x unified'
        )
        fig_line.show()

        # GRAPH 2: TOTAL SALES VOLUME HISTOGRAM BAR CHART
        fig_bar = px.bar(
            graph_df,
            x='Month',
            y='Total_Sales_Value',
            title='Total Sales Absolute Value Magnitude Breakdown per Month',
            color='Total_Sales_Value',
            color_continuous_scale=px.colors.sequential.Viridis,
            labels={'Total_Sales_Value': 'Absolute Sales (KSh)', 'Month': 'Timeline Month'}
        )
        fig_bar.update_traces(
            texttemplate='KSh %{y:,.2f}',
            textposition='outside',
            customdata=np.stack((graph_df['Transaction_Count'], graph_df['Sales_Growth_Rate_Pct']), axis=-1),
            hovertemplate='<b>Month Name:</b> %{x}<br><b>Total Value:</b> KSh %{y:,.2f}<br><b>Unique Invoices:</b> %{customdata[0]:,.0f}<br><b>MoM Shift Rate:</b> %{customdata[1]:+.2f}%<extra></extra>'
        )
        fig_bar.update_layout(
            yaxis_title='Total Dispatched Sales Revenue (KSh)',
            xaxis_title='Timeline Calendar Path',
            height=420,
            margin=dict(t=60, b=50, l=60, r=40),
            coloraxis_showscale=False
        )
        fig_bar.show()

out = widgets.interactive_output(render_growth_analysis_dashboard, {
    'quarters': quarter_select,
    'months': month_select,
    'weekdays': day_of_week_select,
    'customers': customer_select,
    'all_qtrs': all_qtr_check,
    'all_mths': all_mth_check,
    'all_weekdays': all_dow_check,
    'all_custs': all_cust_check,
    'refresh': refresh_btn
})

Output()

In [153]:
# Year-over-year growth analysis
# CASCADING YEAR-OVER-YEAR SALES GROWTH ANALYSIS DASHBOARD
import io
import base64
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML
import plotly.express as px
import plotly.graph_objects as go

def calculate_year_over_year_growth(df, selected_customers, selected_years, selected_quarters, selected_months, selected_days_of_week,
                                    show_all_customers=False, show_all_years=False, show_all_quarters=False,
                                    show_all_months=False, show_all_days_of_week=False):
    df_filtered = df.copy()

    if not show_all_years and selected_years:
        df_filtered = df_filtered[df_filtered['Year'].isin(selected_years)]
    if not show_all_quarters and selected_quarters:
        df_filtered = df_filtered[df_filtered['Quarter_Label'].isin(selected_quarters)]
    if not show_all_months and selected_months:
        df_filtered = df_filtered[df_filtered['Month'].isin(selected_months)]
    if not show_all_days_of_week and selected_days_of_week:
        df_filtered = df_filtered[df_filtered['Day_of_Week'].isin(selected_days_of_week)]
    if not show_all_customers and selected_customers:
        df_filtered = df_filtered[df_filtered['Customer'].isin(selected_customers)]

    if df_filtered.empty:
        return pd.DataFrame()

    sales_col = 'Sales_Value' if 'Sales_Value' in df_filtered.columns else 'Quantity'
    yearly_monthly = df_filtered.groupby(['Year', 'Month']).agg(
        Total_Sales_Value=(sales_col, 'sum'),
        Transaction_Count=('Invoice_No', 'nunique')
    ).reset_index()

    month_order = ['January', 'February', 'March', 'April', 'May', 'June',
                   'July', 'August', 'September', 'October', 'November', 'December']
    yearly_monthly['Month'] = pd.Categorical(
        yearly_monthly['Month'], categories=month_order, ordered=True
    )
    yearly_monthly = yearly_monthly.sort_values(['Month', 'Year']).reset_index(drop=True)

    yearly_monthly['Sales_Growth_Rate_Pct'] = (
        yearly_monthly.groupby('Month', observed=False)['Total_Sales_Value'].pct_change() * 100
    )
    yearly_monthly['Transaction_Growth_Rate_Pct'] = (
        yearly_monthly.groupby('Month', observed=False)['Transaction_Count'].pct_change() * 100
    )
    yearly_monthly['Month'] = yearly_monthly['Month'].astype(str)
    yearly_monthly['Year'] = yearly_monthly['Year'].astype(int)
    return yearly_monthly

# Filter options
quarter_list = ['Q1', 'Q2', 'Q3', 'Q4']
month_order_list = ['January', 'February', 'March', 'April', 'May', 'June',
                    'July', 'August', 'September', 'October', 'November', 'December']
day_order_list = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
year_list = sorted(pd.to_numeric(df_long_clean['Year'], errors='coerce').dropna().astype(int).unique().tolist())

quarter_select = widgets.SelectMultiple(options=quarter_list, value=(), description='Quarters:', rows=4)
year_select = widgets.SelectMultiple(options=year_list, value=(), description='Years:', rows=5)
month_select = widgets.SelectMultiple(options=month_order_list, value=(), description='Months:', rows=6)
day_of_week_select = widgets.SelectMultiple(
    options=[day for day in day_order_list if day in df_long_clean['Day_of_Week'].dropna().unique()],
    value=(), description='Weekdays:', rows=5
)
customer_select = widgets.SelectMultiple(options=(), value=(), description='Customers:', rows=6)

all_year_check = widgets.Checkbox(value=False, description='All Years')
all_qtr_check = widgets.Checkbox(value=False, description='All Quarters')
all_mth_check = widgets.Checkbox(value=False, description='All Months')
all_dow_check = widgets.Checkbox(value=False, description='All Weekdays')
all_cust_check = widgets.Checkbox(value=False, description='All Customers')
refresh_btn = widgets.ToggleButton(value=False, description='Refresh Calculations', button_style='info')
dashboard_output = widgets.Output()

def update_customer_options(*args):
    chosen_years = year_list if all_year_check.value else year_select.value
    if not chosen_years:
        customer_select.options = ()
        customer_select.value = ()
        return

    filtered = df_long_clean[df_long_clean['Year'].isin(chosen_years)]
    if not all_qtr_check.value and quarter_select.value:
        filtered = filtered[filtered['Quarter_Label'].isin(quarter_select.value)]
    if not all_mth_check.value and month_select.value:
        filtered = filtered[filtered['Month'].isin(month_select.value)]
    if not all_dow_check.value and day_of_week_select.value:
        filtered = filtered[filtered['Day_of_Week'].isin(day_of_week_select.value)]

    customer_options = sorted(filtered['Customer'].dropna().unique().tolist())
    customer_select.options = customer_options
    customer_select.value = tuple(customer for customer in customer_select.value if customer in customer_options)

for widget in [year_select, all_year_check, quarter_select, all_qtr_check, month_select, all_mth_check,
               day_of_week_select, all_dow_check]:
    widget.observe(update_customer_options, names='value')
update_customer_options()

display(widgets.VBox([
    widgets.HBox([year_select, quarter_select, month_select]),
    widgets.HBox([day_of_week_select, customer_select]),
    widgets.HBox([all_year_check, all_qtr_check, all_mth_check, all_dow_check, all_cust_check, refresh_btn])
]))
display(dashboard_output)

def render_year_over_year_dashboard(years, quarters, months, weekdays, customers, all_years, all_qtrs, all_mths, all_weekdays, all_customers, refresh):
    with dashboard_output:
        dashboard_output.clear_output()

        chosen_years = year_list if all_years else years
        chosen_quarters = quarter_list if all_qtrs else quarters
        chosen_months = month_order_list if all_mths else months
        chosen_weekdays = list(day_of_week_select.options) if all_weekdays else weekdays
        chosen_customers = list(customer_select.options) if all_customers else customers

        if not chosen_years:
            print('Step 1: Choose at least one year or enable All Years.')
            return
        if not chosen_months:
            print('Step 2: Choose months or enable All Months.')
            return
        if not chosen_weekdays:
            print('Step 3: Choose weekdays or enable All Weekdays.')
            return
        if not chosen_customers and not all_customers:
            print('Step 4: Choose customers or enable All Customers.')
            return

        result = calculate_year_over_year_growth(
            df_long_clean, chosen_customers, chosen_years, chosen_quarters, chosen_months, chosen_weekdays,
            show_all_customers=all_customers, show_all_years=all_years,
            show_all_quarters=all_qtrs, show_all_months=all_mths,
            show_all_days_of_week=all_weekdays
        )
        if result.empty:
            print('No matching transaction data fits this combined filter query.')
            return

        output_buffer = io.BytesIO()
        with pd.ExcelWriter(output_buffer, engine='xlsxwriter') as writer:
            result.to_excel(writer, index=False, sheet_name='Year-over-Year Growth')
            workbook = writer.book
            worksheet = writer.sheets['Year-over-Year Growth']
            int_format = workbook.add_format({'num_format': '#,##0', 'valign': 'top'})
            money_format = workbook.add_format({'num_format': 'KSh #,##0.00', 'valign': 'top'})
            percent_format = workbook.add_format({'num_format': '0.00"%"', 'valign': 'top'})
            worksheet.set_column('A:A', 12, int_format)
            worksheet.set_column('B:B', 18)
            worksheet.set_column('C:C', 22, money_format)
            worksheet.set_column('D:D', 18, int_format)
            worksheet.set_column('E:F', 24, percent_format)

        encoded_data = base64.b64encode(output_buffer.getvalue()).decode()
        display(HTML(f'''<a download="year_over_year_growth_analysis.xlsx"
            href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{encoded_data}">
            <button style="background:#1F6E43;color:white;padding:8px 16px;border:none;border-radius:4px;cursor:pointer;font-weight:bold;">
            Download Year-over-Year Analysis (.xlsx)</button></a>'''))

        styled_df = result.style.format({
            'Year': '{:,.0f}',
            'Total_Sales_Value': 'KSh {:,.2f}',
            'Transaction_Count': '{:,.0f}',
            'Sales_Growth_Rate_Pct': '{:+.2f}%',
            'Transaction_Growth_Rate_Pct': '{:+.2f}%'
        }, na_rep='Baseline').set_properties(**{
            'text-align': 'right', 'vertical-align': 'top'
        }).set_properties(subset=['Year', 'Month'], **{'text-align': 'left'})
        display(styled_df)

        graph_df = result.copy()
        graph_df['Sales_Growth_Rate_Pct'] = graph_df['Sales_Growth_Rate_Pct'].fillna(0.0)
        graph_df['Transaction_Growth_Rate_Pct'] = graph_df['Transaction_Growth_Rate_Pct'].fillna(0.0)
        graph_df['Month_Idx'] = graph_df['Month'].map({month: index for index, month in enumerate(month_order_list)})
        graph_df = graph_df.sort_values(by=['Month_Idx', 'Year']).reset_index(drop=True)
        graph_df['Timeline_Context'] = graph_df['Month'] + ' ' + graph_df['Year'].astype(str)

        # GRAPH 1: INTERACTIVE DUAL-AXIS YoY LINES TRAJECTORY
        delta_df = graph_df[graph_df['Sales_Growth_Rate_Pct'] != 0.0].copy()
        if not delta_df.empty:
            fig_line = go.Figure()
            fig_line.add_trace(go.Scatter(
                x=delta_df['Timeline_Context'],
                y=delta_df['Sales_Growth_Rate_Pct'],
                name='YoY Revenue Growth',
                mode='lines+markers',
                marker=dict(color='#1A73E8', size=8),
                line=dict(width=3),
                customdata=delta_df['Total_Sales_Value'],
                hovertemplate='<b>Period:</b> %{x}<br><b>YoY Revenue Growth:</b> %{y:+.2f}%<br><b>Total Sales:</b> KSh %{customdata:,.2f}<extra></extra>'
            ))
            fig_line.add_trace(go.Scatter(
                x=delta_df['Timeline_Context'],
                y=delta_df['Transaction_Growth_Rate_Pct'],
                name='YoY Order Volume Growth',
                mode='lines+markers',
                yaxis='y2',
                marker=dict(color='#C5221F', size=8),
                line=dict(width=3, dash='dash'),
                customdata=delta_df['Transaction_Count'],
                hovertemplate='<b>Period:</b> %{x}<br><b>YoY Order Growth:</b> %{y:+.2f}%<br><b>Total Orders:</b> %{customdata:,.0f}<extra></extra>'
            ))
            fig_line.update_layout(
                title=dict(text='Year-over-Year Percentage Growth Progression Curve', font=dict(size=13)),
                height=380,
                margin=dict(t=50, b=40, l=60, r=60),
                xaxis=dict(title='Timeline Calendar Horizon'),
                yaxis=dict(
                    title=dict(text='Revenue Growth Rate (%)', font=dict(color='#1A73E8')),
                    tickfont=dict(color='#1A73E8')
                ),
                yaxis2=dict(
                    title=dict(text='Order Growth Rate (%)', font=dict(color='#C5221F')),
                    tickfont=dict(color='#C5221F'),
                    overlaying='y',
                    side='right'
                ),
                hovermode='x unified'
            )
            fig_line.show()
        else:
            print('YoY growth lines require at least two years with the same month available.')

        # GRAPH 2: GROUPED REVENUE COMPARISON MAGNITUDE BAR CHART
        fig_bar = px.bar(
            graph_df,
            x='Month',
            y='Total_Sales_Value',
            color='Year',
            barmode='group',
            color_continuous_scale=px.colors.sequential.Plotly3,
            title='Absolute Sales Revenue Magnitude Matrix Comparison (Year-on-Year)',
            labels={'Total_Sales_Value': 'Absolute Sales Value (KSh)', 'Month': 'Calendar Month', 'Year': 'Year'},
            category_orders={'Month': month_order_list}
        )
        fig_bar.update_traces(
            texttemplate='KSh %{y:,.0f}',
            textposition='outside',
            customdata=np.stack((graph_df['Transaction_Count'], graph_df['Sales_Growth_Rate_Pct']), axis=-1),
            hovertemplate='<b>Period:</b> %{x}<br><b>Sales Value:</b> KSh %{y:,.2f}<br><b>Orders Count:</b> %{customdata[0]:,.0f}<br><b>YoY Shift Rate:</b> %{customdata[1]:+.2f}%<extra></extra>'
        )
        fig_bar.update_layout(
            yaxis_title='Dispatched Sales Revenue (KSh)',
            xaxis_title='Calendar Month Grouping',
            height=400,
            margin=dict(t=60, b=50, l=60, r=40),
            coloraxis_showscale=False
        )
        fig_bar.show()

out = widgets.interactive_output(render_year_over_year_dashboard, {
    'years': year_select,
    'quarters': quarter_select,
    'months': month_select,
    'weekdays': day_of_week_select,
    'customers': customer_select,
    'all_years': all_year_check,
    'all_qtrs': all_qtr_check,
    'all_mths': all_mth_check,
    'all_weekdays': all_dow_check,
    'all_customers': all_cust_check,
    'refresh': refresh_btn
})

Output()

## CUSTOMER × PRODUCT

In [154]:
# Customer-product analysis
# Which customers buy which products?
# CASCADING CUSTOMER-PRODUCT ANALYTICS MATRIX DASHBOARD
import io
import base64
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML
import plotly.express as px

def analyze_customer_product_relationship(df, selected_customers, selected_quarters, selected_months, selected_days_of_week,
                                          show_all_customers=False, show_all_quarters=False, show_all_months=False, show_all_days_of_week=False):
    """
    Filters data dynamically across 4 axes and maps customer-to-product purchasing histories
    capturing dispatched volumes and absolute monetary weights simultaneously.
    """
    df_filtered = df.copy()
    
    # 1. Apply Dynamic Structural Slicers
    if not show_all_quarters and selected_quarters:
        df_filtered = df_filtered[df_filtered['Quarter_Label'].isin(selected_quarters)]
    if not show_all_months and selected_months:
        df_filtered = df_filtered[df_filtered['Month'].isin(selected_months)]
    if not show_all_days_of_week and selected_days_of_week:
        df_filtered = df_filtered[df_filtered['Day_of_Week'].isin(selected_days_of_week)]
    if not show_all_customers and selected_customers:
        df_filtered = df_filtered[df_filtered['Customer'].isin(selected_customers)]
        
    if df_filtered.empty:
        return pd.DataFrame()
        
    sales_col = 'Sales_Value' if 'Sales_Value' in df_filtered.columns else 'Quantity'
    
    # 2. Group by Customer and Product pairing matrices
    relationship_df = df_filtered.groupby(['Customer', 'Product']).agg(
        Total_Quantity_Sold=(sales_col, 'count'),
        Total_Sales_Value=(sales_col, 'sum')
    ).reset_index()
    
    # Sort descending by value footprint to keep top moving lines upfront
    return relationship_df.sort_values(by=['Customer', 'Total_Sales_Value'], ascending=[True, False]).reset_index(drop=True)

# --- Dynamic Filter Cascading Setup ---
quarter_list = ['Q1', 'Q2', 'Q3', 'Q4']
month_order_list = ['January', 'February', 'March', 'April', 'May', 'June', 
                    'July', 'August', 'September', 'October', 'November', 'December']
day_order_list = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Initialize UI Layout Elements
quarter_select = widgets.SelectMultiple(options=quarter_list, value=(), description='Quarters:', rows=4)
month_select = widgets.SelectMultiple(options=(), value=(), description='Months:', rows=5)
day_of_week_select = widgets.SelectMultiple(
    options=[day for day in day_order_list if day in df_long_clean['Day_of_Week'].dropna().unique()],
    value=(), description='Weekdays:', rows=5
)
customer_select = widgets.SelectMultiple(options=(), value=(), description='Customers:', rows=6)

all_qtr_check = widgets.Checkbox(value=False, description='All Quarters')
all_mth_check = widgets.Checkbox(value=False, description='All Months')
all_dow_check = widgets.Checkbox(value=False, description='All Weekdays')
all_cust_check = widgets.Checkbox(value=False, description='All Customers')

refresh_btn = widgets.ToggleButton(value=False, description='Refresh Matrix', button_style='info')
dashboard_output = widgets.Output()

def update_cascading_options(*args):
    chosen_quarters = quarter_list if all_qtr_check.value else quarter_select.value
    if not chosen_quarters:
        month_select.options = ()
        month_select.value = ()
        customer_select.options = ()
        customer_select.value = ()
        return

    filtered = df_long_clean[df_long_clean['Quarter_Label'].isin(chosen_quarters)]
    valid_months = filtered['Month'].dropna().unique()
    month_options = [month for month in month_order_list if month in valid_months]
    month_select.options = month_options
    month_select.value = tuple(month for month in month_select.value if month in month_options)

    chosen_months = list(month_select.options) if all_mth_check.value else month_select.value
    if chosen_months:
        filtered = filtered[filtered['Month'].isin(chosen_months)]
    chosen_days = list(day_of_week_select.options) if all_dow_check.value else day_of_week_select.value
    if chosen_days:
        filtered = filtered[filtered['Day_of_Week'].isin(chosen_days)]

    customer_options = sorted(filtered['Customer'].dropna().unique().tolist())
    customer_select.options = customer_options
    customer_select.value = tuple(customer for customer in customer_select.value if customer in customer_options)

for widget in [quarter_select, all_qtr_check, month_select, all_mth_check, day_of_week_select, all_dow_check]:
    widget.observe(update_cascading_options, names='value')
update_cascading_options()

display(widgets.VBox([
    widgets.HBox([quarter_select, month_select, day_of_week_select]),
    widgets.HBox([all_qtr_check, all_mth_check, all_dow_check]),
    widgets.HBox([customer_select, all_cust_check, refresh_btn])
]))
display(dashboard_output)

def render_customer_product_analysis_dashboard(quarters, months, weekdays, customers, all_qtrs, all_mths, all_weekdays, all_custs, refresh):
    with dashboard_output:
        dashboard_output.clear_output()

        chosen_quarters = quarter_list if all_qtrs else quarters
        chosen_months = list(month_select.options) if all_mths else months
        chosen_weekdays = list(day_of_week_select.options) if all_weekdays else weekdays
        chosen_customers = list(customer_select.options) if all_custs else customers

        if not chosen_quarters or not chosen_months or not chosen_weekdays:
            print('Step 1: Choose quarters, months, and weekdays, or enable their All options.')
            return
        if not chosen_customers and not all_custs:
            print('Step 2: Choose customers or enable All Customers.')
            return

        result = analyze_customer_product_relationship(
            df_long_clean, chosen_customers, chosen_quarters, chosen_months, chosen_weekdays,
            show_all_customers=all_custs, show_all_quarters=all_qtrs, show_all_months=all_mths, show_all_days_of_week=all_weekdays
        )
        if result.empty:
            print('No matching transaction data fits this combined filter query.')
            return

        # Limit panel viewing to top outliers while preserving full database downloads
        display_result = result.head(25).copy()

        output_buffer = io.BytesIO()
        with pd.ExcelWriter(output_buffer, engine='xlsxwriter') as writer:
            result.to_excel(writer, index=False, sheet_name='Customer-Product Analysis')
            workbook = writer.book
            worksheet = writer.sheets['Customer-Product Analysis']
            int_format = workbook.add_format({'num_format': '#,##0', 'valign': 'top'})
            money_format = workbook.add_format({'num_format': 'KSh #,##0.00', 'valign': 'top'})
            worksheet.set_column('A:A', 35)
            worksheet.set_column('B:B', 20)
            worksheet.set_column('C:C', 22, int_format)
            worksheet.set_column('D:D', 22, money_format)

        encoded_data = base64.b64encode(output_buffer.getvalue()).decode()
        display(HTML(f'''<a download="customer_product_relationship_matrix.xlsx"
            href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{encoded_data}">
            <button style="background:#1F6E43;color:white;padding:8px 16px;border:none;border-radius:4px;cursor:pointer;font-weight:bold;margin-bottom:12px;">
            Download Customer-Product Analysis (.xlsx)</button></a>'''))

        # Render clean stylized pandas grid frame
        styled_df = display_result.style.format({
            'Total_Quantity_Sold': '{:,.0f}',
            'Total_Sales_Value': 'KSh {:,.2f}'
        }).set_properties(**{
            'text-align': 'right', 'vertical-align': 'top'
        }).set_properties(subset=['Customer', 'Product'], **{'text-align': 'left'})
        
        display(styled_df)
        print("\n")

        # --- GRAPH FEATURE: INTERACTIVE HORIZONTAL REVENUE DISTRIBUTION CHART ---
        display_result = display_result.sort_values(by='Total_Sales_Value', ascending=True)
        
        fig = px.bar(
            display_result,
            x='Total_Sales_Value',
            y='Customer',
            color='Product',
            barmode='group',
            orientation='h',
            title='Product Revenue Breakdown Matrix per Selected Customer Account',
            color_discrete_sequence=px.colors.qualitative.Prism,
            labels={'Total_Sales_Value': 'Total Value (KSh)', 'Customer': 'Customer Identity'}
        )
        
        fig.update_traces(
            customdata=display_result['Total_Quantity_Sold'],
            hovertemplate='<b>Client Name:</b> %{y}<br><b>Product Code:</b> %{legendgroup}<br><b>Revenue Value:</b> KSh %{x:,.2f}<br><b>Units Packaged:</b> %{customdata:,.0f}<extra></extra>'
        )
        
        fig.update_layout(
            xaxis_title="Aggregated Sales Value Volume (KSh)",
            yaxis_title="Customer Account Profiles",
            height=550,
            margin=dict(t=60, b=50, l=150, r=40),
            legend_title_text='Catalog Variations'
        )
        fig.show()

# Bind parameters to data outputs dynamically using interactive_output manager logic loop
out = widgets.interactive_output(render_customer_product_analysis_dashboard, {
    'quarters': quarter_select, 'months': month_select, 'weekdays': day_of_week_select, 'customers': customer_select,
    'all_qtrs': all_qtr_check, 'all_mths': all_mth_check, 'all_weekdays': all_dow_check, 'all_custs': all_cust_check,
    'refresh': refresh_btn
})


Output()